<a href="https://colab.research.google.com/github/PhilipJones-6294/DataLoader/blob/claude%2Frefactor-nvidia-dgx-011CUuAPTPZPuYYKCaoqDCyB/MuZero_Under_the_Hood_orignal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Copy of MuZero_From_Scratch.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1wE2jcT0GR5SBFmwoxedty1AQLpV9TIaN
"""

# ==============================================================================
# @title #SECTION 1: SETUP AND INITIAL CONFIGURATION (MUZERO PROJECT)
# ==============================================================================

# ------------------------------------------------------------------------------
# A. COLAB ENVIRONMENT SETUP (if applicable)
# ------------------------------------------------------------------------------
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    from google.colab import drive
    if not os.path.isdir('/content/drive'): # Check if drive is already mounted
        drive.mount('/content/drive')
    else:
        print("Google Drive already mounted.")


    # Check GPU availability
    try:
        gpu_info = subprocess.check_output('nvidia-smi --query-gpu=gpu_name,memory.total --format=csv,noheader', shell=True).decode('ascii').strip()
        if gpu_info:
            print(f"GPU detected: {gpu_info}")
        else:
            print("No GPU detected by nvidia-smi, or nvidia-smi not found.")
    except Exception as e:
        print(f"Could not run nvidia-smi: {e}. GPU status unknown.")
else:
    print("Not running in Google Colab environment.")

# ------------------------------------------------------------------------------
# B. INSTALL NECESSARY PACKAGES (with TA-Lib from afterford_load_data.py)
# ------------------------------------------------------------------------------
print("\nInstalling/Checking Python packages...")

def install_packages(packages):
    for package_name, import_name in packages.items():
        try:
            __import__(import_name)
            print(f"{package_name} already installed.")
        except ImportError:
            print(f"Installing {package_name}...")
            if IN_COLAB:
                pip_command = [sys.executable, "-m", "pip", "install", "-q", package_name]
                process = subprocess.run(pip_command, capture_output=True, text=True)
                if process.returncode == 0:
                    print(f"{package_name} installed successfully.")
                else:
                    print(f"Error installing {package_name}. Return code: {process.returncode}")
                    print(f"Stdout: {process.stdout}")
                    print(f"Stderr: {process.stderr}")
            else:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        except Exception as e:
            print(f"Error checking/installing {package_name}: {e}")

required_packages = {
    "tensorflow": "tensorflow",
    "numpy": "numpy",
    "pandas": "pandas",
    "conda-package-handling": "conda_package_handling" # Ensure cph is listed for install if needed
}
install_packages(required_packages)

if IN_COLAB:
    print("\nChecking TA-Lib installation for Colab...")
    try:
        import talib
        print("TA-Lib already available.")
    except ImportError:
        print("TA-Lib not found, attempting installation using method from 'afterford_load_data.py'...")
        try:
            # --- TA-Lib Installation from afterford_load_data.py ---
            TA_LIB_PYTHON_DEST_DIR = "/usr/local/lib/python3.11/dist-packages/talib" # As in your script
            TA_LIB_C_LIB_FILE = "/usr/lib/x86_64-linux-gnu/libta-lib.so.0"

            print("  Checking for TA-Lib C library...")
            if not os.path.exists(TA_LIB_C_LIB_FILE):
                print("  TA-Lib C library not found. Downloading and installing...")
                c_lib_url = 'https://anaconda.org/conda-forge/libta-lib/0.4.0/download/linux-64/libta-lib-0.4.0-h166bdaf_1.tar.bz2'
                curl_proc = subprocess.Popen(['curl', '-L', c_lib_url], stdout=subprocess.PIPE)
                tar_proc = subprocess.Popen(
                    ['tar', 'xj', '-C', '/usr/lib/x86_64-linux-gnu/', 'lib', '--strip-components=1'],
                    stdin=curl_proc.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE
                )
                curl_proc.stdout.close()
                tar_stdout, tar_stderr = tar_proc.communicate()
                if tar_proc.returncode == 0:
                    print("  TA-Lib C library installed successfully.")
                else:
                    print(f"  Error installing TA-Lib C library. RC: {tar_proc.returncode}, Stderr: {tar_stderr.decode()}")
            else:
                print(f"  TA-Lib C library already exists at {TA_LIB_C_LIB_FILE}.")

            print("\n  Checking for TA-Lib Python wrapper...")
            if not os.path.isdir(TA_LIB_PYTHON_DEST_DIR): # Check destination first
                print(f"  TA-Lib Python wrapper not found at {TA_LIB_PYTHON_DEST_DIR}. Downloading and installing...")

                CONDA_PKG_URL = "https://anaconda.org/conda-forge/ta-lib/0.5.1/download/linux-64/ta-lib-0.5.1-py311h9ecbd09_0.conda"
                CONDA_PKG_NAME = "ta-lib-0.5.1-py311h9ecbd09_0.conda" # Filename after download
                EXTRACTED_DIR_NAME = "ta-lib-0.5.1-py311h9ecbd09_0" # Default cph extraction dir name

                print(f"    Downloading {CONDA_PKG_NAME} from {CONDA_PKG_URL}...")
                subprocess.run(["wget", "-q", "-nc", CONDA_PKG_URL, "-O", CONDA_PKG_NAME], check=True) # -nc for no-clobber, -O to rename

                print(f"    Removing old extracted directory ./{EXTRACTED_DIR_NAME} if it exists...")
                if os.path.exists(f"./{EXTRACTED_DIR_NAME}"): # Check before removing
                    subprocess.run(f"rm -rf ./{EXTRACTED_DIR_NAME}", shell=True, check=True)

                print(f"    Extracting {CONDA_PKG_NAME}...")
                # cph might not have a quiet flag, redirect output if too verbose
                # Using shell=True for cph as it might be an alias or simple script
                extract_proc = subprocess.run(f"cph x {CONDA_PKG_NAME}", shell=True, check=True, capture_output=True, text=True)
                if extract_proc.returncode != 0:
                    print(f"    Wrapper extraction Stderr: {extract_proc.stderr}")
                    raise subprocess.CalledProcessError(extract_proc.returncode, extract_proc.args, output=extract_proc.stdout, stderr=extract_proc.stderr)


                # Determine current Python version for path construction
                # Your load_data script hardcodes python3.11. Let's try to be dynamic but have a fallback.
                current_py_ver_long = f"python{sys.version_info.major}.{sys.version_info.minor}"
                ga_runner_py_ver_long = "python3.11" # From your successful GA runner

                source_talib_path_template = "./{extracted_dir}/lib/{py_ver}/site-packages/talib"
                dest_talib_path_template = "/usr/local/lib/{py_ver}/dist-packages/talib"

                source_talib_path = source_talib_path_template.format(extracted_dir=EXTRACTED_DIR_NAME, py_ver=current_py_ver_long)
                dest_talib_path = dest_talib_path_template.format(py_ver=current_py_ver_long)

                if not os.path.isdir(source_talib_path):
                    print(f"    Path {source_talib_path} not found. Trying fallback to python3.11 structure.")
                    source_talib_path = source_talib_path_template.format(extracted_dir=EXTRACTED_DIR_NAME, py_ver=ga_runner_py_ver_long)
                    dest_talib_path = dest_talib_path_template.format(py_ver=ga_runner_py_ver_long)


                if os.path.isdir(source_talib_path):
                    print(f"    Moving {source_talib_path} to {dest_talib_path}...")
                    os.makedirs(os.path.dirname(dest_talib_path), exist_ok=True) # Ensure parent of destination exists
                    subprocess.run(["mv", source_talib_path, dest_talib_path], check=True) # mv command as in your script
                    print("    TA-Lib Python wrapper installed.")
                else:
                    print(f"    ERROR: Source path {source_talib_path} not found after extraction. Check EXTRACTED_DIR_NAME and package contents.")
                    # Attempt to list contents for debugging
                    if os.path.isdir(f"./{EXTRACTED_DIR_NAME}/lib/"):
                        print(f"    Contents of ./{EXTRACTED_DIR_NAME}/lib/: {os.listdir(f'./{EXTRACTED_DIR_NAME}/lib/')}")
                    raise FileNotFoundError(f"Could not find source talib path: {source_talib_path}")

            else: # TA_LIB_PYTHON_DEST_DIR already exists
                print(f"  TA-Lib Python wrapper already exists at {TA_LIB_PYTHON_DEST_DIR}. Skipping download and move.")

            # --- Cleanup downloaded and extracted files ---
            files_to_clean = [CONDA_PKG_NAME]
            dirs_to_clean = [f"./{EXTRACTED_DIR_NAME}"]

            print("  Cleaning up downloaded and extracted TA-Lib files...")
            for f_clean in files_to_clean:
                if os.path.exists(f_clean):
                    try:
                        os.remove(f_clean)
                        print(f"    Removed file: {f_clean}")
                    except Exception as e_clean:
                        print(f"    Warning: Could not remove file {f_clean}: {e_clean}")
            for d_clean in dirs_to_clean:
                 if os.path.exists(d_clean):
                    try:
                        import shutil
                        shutil.rmtree(d_clean)
                        print(f"    Removed directory: {d_clean}")
                    except Exception as e_clean:
                        print(f"    Warning: Could not remove directory {d_clean}: {e_clean}")
            # --- End Cleanup ---

            import talib # Final verification
            print("TA-Lib successfully installed and imported for Colab.")

        except subprocess.CalledProcessError as cpe:
            print(f"Error installing TA-Lib (CalledProcessError): {cpe}")
            if hasattr(cpe, 'cmd'): print(f"  Command: {cpe.cmd}")
            if hasattr(cpe, 'returncode'): print(f"  Return code: {cpe.returncode}")
            if hasattr(cpe, 'stdout') and cpe.stdout: print(f"  Stdout: {cpe.stdout}")
            if hasattr(cpe, 'stderr') and cpe.stderr: print(f"  Stderr: {cpe.stderr}")
        except FileNotFoundError as fnf_err:
            print(f"Error installing TA-Lib (FileNotFoundError): {fnf_err}")
        except Exception as e:
            print(f"An unexpected error occurred during TA-Lib installation: {e}")


# ------------------------------------------------------------------------------
# C. LOAD EXISTING CONFIG AND DEFINE MUZERO-SPECIFIC CONFIGURATIONS
# ------------------------------------------------------------------------------
print("\nLoading configurations...")

PROJECT_ROOT_FOR_PATH = '/content/drive/MyDrive/Afterford' if IN_COLAB else '.'
if PROJECT_ROOT_FOR_PATH not in sys.path:
    sys.path.insert(0, PROJECT_ROOT_FOR_PATH)
    print(f"Added {PROJECT_ROOT_FOR_PATH} to sys.path for importing 'src.config'")

try:
    from src import config as existing_cfg
    print("Successfully imported existing configuration from 'src.config'.")
except ImportError as e:
    print(f"FATAL: Could not import 'src.config' from {PROJECT_ROOT_FOR_PATH}. Error: {e}")
    raise

# --- Define MuZero Base Directory ---
MUZERO_BASE_DIR = os.path.join(PROJECT_ROOT_FOR_PATH, 'MU')
if IN_COLAB and not os.path.isdir(MUZERO_BASE_DIR):
    try:
        os.makedirs(MUZERO_BASE_DIR, exist_ok=True)
        print(f"Created MuZero base directory: {MUZERO_BASE_DIR}")
    except Exception as e:
        print(f"Could not create MuZero base directory: {MUZERO_BASE_DIR}. Error: {e}")
        raise

class MuZeroConfig:
    def __init__(self, existing_config_module, muzero_project_base_dir):
        self.MUZERO_PROJECT_BASE_DIR = muzero_project_base_dir #

        # --- MuZero Artifact Paths (Outputs of MuZero, saved within MUZERO_BASE_DIR) ---
        # Define MUZERO_MODELS_DIR FIRST
        self.MUZERO_MODELS_DIR = os.path.join(self.MUZERO_PROJECT_BASE_DIR, 'models') #
        self.MUZERO_LOGS_DIR = os.path.join(self.MUZERO_PROJECT_BASE_DIR, 'logs') #
        self.MUZERO_REPLAY_BUFFER_DIR = os.path.join(self.MUZERO_PROJECT_BASE_DIR, 'replay_buffer') #
        self.MUZERO_CHECKPOINT_FORMAT = os.path.join(self.MUZERO_MODELS_DIR, 'muzero_checkpoint_step_{step}') #

        # --- Environment & Action Space ---
        self.NUM_ACTIONS = 3 #
        self.PORTFOLIO_STATE_SIZE = 3 #

        # --- Paths to Existing Artifacts (Inputs for MuZero's INITIAL setup) ---
        # Now self.MUZERO_MODELS_DIR can be safely used.
        # These files are located within a 'best_model' subdirectory inside MUZERO_MODELS_DIR.
        BEST_MODEL_SUBDIR = 'best_model' # Define the subdirectory name

        self.FINAL_BEST_NN_PATH = os.path.join(self.MUZERO_MODELS_DIR, BEST_MODEL_SUBDIR, 'best_model.keras')
        self.GA_BEST_MASK_PATH = os.path.join(self.MUZERO_MODELS_DIR, BEST_MODEL_SUBDIR, 'best_model_mask.txt')
        self.FEATURE_LIST_PATH = os.path.join(self.MUZERO_MODELS_DIR, BEST_MODEL_SUBDIR, 'training_features.json')

        original_scaler_filename = os.path.basename(existing_config_module.FINAL_SCALER_SAVE_PATH)
        self.SCALER_PATH = os.path.join(self.MUZERO_MODELS_DIR, BEST_MODEL_SUBDIR, original_scaler_filename)

        self.MARKET_DATA_PATH = existing_config_module.FINAL_DATA_FILE_PATH #
        # Ensure ETF_DATA_PATH is correctly derived
        etf_data_filename = 'data_etf.csv' # As seen in afterford_load_data (3).py
        self.ETF_DATA_PATH = os.path.join(existing_config_module.CORE_DIRECTORIES['output'], etf_data_filename) #


        # --- Model Architecture ---
        self.MUZERO_HIDDEN_STATE_SIZE = 256 #
        self.DYNAMICS_INTERNAL_SIZE = 128 #
        self.DYNAMICS_INTERNAL_SIZE_REWARD = 64#
        self.PREDICTION_INTERNAL_SIZE_POLICY = 128 #
        self.PREDICTION_INTERNAL_SIZE_VALUE = 64 #
        self.TRANSFER_LAYER_NAME = 'BatchNorm2' # CRITICAL: Verify this layer name later #

        # --- MCTS ---
        self.NUM_SIMULATIONS = 100 #  CHANGE TO 100
        self.MCTS_ROOT_DIRICHLET_ALPHA = 0.3 #
        self.MCTS_ROOT_EXPLORATION_FRACTION = 0.25 #
        self.MCTS_PB_C_BASE = 19652 #
        self.MCTS_PB_C_INIT = 1.25 #

        # --- Training ---
        self.BATCH_SIZE = 128 #
        self.REPLAY_BUFFER_SIZE = 10000 #
        self.NUM_UNROLL_STEPS = 3 #
        self.TD_STEPS = 10 #
        self.LEARNING_RATE = 0.000005 # MuZero often uses smaller LRs #
        self.WEIGHT_DECAY = 0.0001 #
        self.REWARD_LOSS_WEIGHT = 1.0 #
        self.VALUE_LOSS_WEIGHT = 0.25 #
        self.POLICY_LOSS_WEIGHT = 1.0 #
        self.MAX_TRAINING_STEPS = 1000000 # Typically large #
        self.CHECKPOINT_INTERVAL = 100 # How often to save model weights #

        self.INITIAL_TEMPERATURE = 1.0 #
        self.ANNEAL_TEMPERATURE_AFTER_N_MOVES = 30000 # Example, based on agent steps #
        self.FINAL_TEMPERATURE = 0.1 #

        self.FREEZE_TRANSFER_WEIGHTS_INITIALLY = True #
        self.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP = 9000 # Example training steps #

        # Adjusted VALUE_TARGET_SCALING_DIVISOR as per previous discussion
        self.VALUE_TARGET_SCALING_DIVISOR = 30.0 # Changed from 10.0
        self.REWARD_TARGET_SCALING_DIVISOR = 2.0 #

        # Min/Max values for reward/value, these were in your original config
        # but ensure they are present if used for any direct clipping/scaling logic elsewhere
        # or for semantic understanding of target ranges. The tanh output of networks and
        # tanh in target scaling already bounds things to [-1, 1] for loss inputs.
        self.MIN_MAX_REWARD_LOWER = -1.0 #
        self.MIN_MAX_REWARD_UPPER = 1.0 #
        self.MIN_MAX_VALUE_LOWER = -1.0 #
        self.MIN_MAX_VALUE_UPPER = 1.0 #
        self.DISCOUNT_FACTOR = 0.997 #

        self.SEED = existing_config_module.RANDOM_SEED # Use existing seed for reproducibility #

        # Store existing config module if direct access is needed later
        self.existing_config_module = existing_config_module #

    def create_muzero_directories(self): #
        """Creates all necessary directories for MuZero artifacts."""
        dirs_to_create = [
            self.MUZERO_PROJECT_BASE_DIR,
            self.MUZERO_MODELS_DIR,
            self.MUZERO_LOGS_DIR,
            self.MUZERO_REPLAY_BUFFER_DIR
        ] #
        for d in dirs_to_create:
            try:
                os.makedirs(d, exist_ok=True) #
                print(f"Ensured MuZero directory exists: {d}") #
            except Exception as e:
                print(f"Could not create MuZero directory: {d}. Error: {e}") #
                # Depending on severity, you might want to raise an error here

# Instantiate MuZeroConfig
config_mz = MuZeroConfig(existing_cfg, MUZERO_BASE_DIR)
config_mz.create_muzero_directories() # Create the directories

print("MuZero-specific configuration (config_mz) created and directories ensured.")
print(f"  MuZero Project Base Directory: {config_mz.MUZERO_PROJECT_BASE_DIR}")
print(f"  Path to existing forecasting NN: {config_mz.FINAL_BEST_NN_PATH}")
print(f"  Path for MuZero model checkpoints: {config_mz.MUZERO_MODELS_DIR}")

# ------------------------------------------------------------------------------
# D. SETUP LOGGING FOR MUZERO PROJECT
# ------------------------------------------------------------------------------
import logging
from datetime import datetime

# Modified logging setup to save logs within the MUZERO_LOGS_DIR
def setup_muzero_logging(log_dir, log_level=logging.INFO, log_to_console=True, log_file_prefix="muzero_afterford"):
    """Configure logging for the MuZero project."""
    log_file = None
    if log_dir:
        try:
            os.makedirs(log_dir, exist_ok=True)
            log_file = os.path.join(log_dir, f'{log_file_prefix}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
        except OSError as e:
            print(f"Warning: Could not create log directory {log_dir}. File logging disabled. Error: {e}")
            log_dir = None # Disable file logging if dir creation fails
    else:
        print("Warning: Log directory not provided. File logging disabled.")

    handlers = []
    if log_dir and log_file:
        handlers.append(logging.FileHandler(log_file))
    if log_to_console:
        handlers.append(logging.StreamHandler(sys.stdout))

    if not handlers:
        print("Warning: No logging handlers configured (console or file). Logging will not occur.")
        return None # Or return a basic logger if preferred

    logging.basicConfig(level=log_level,
                        format='%(asctime)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(module)s - %(message)s',
                        datefmt='%Y-%m-%d %H:%M:%S',
                        handlers=handlers,
                        force=True) # Force reconfig if run multiple times in Colab
    logger = logging.getLogger("MuZeroProject") # Get a specific logger name
    if log_dir and log_file:
        logger.info(f"MuZero project logging initialized. Level: {logging.getLevelName(log_level)}. Log file: {log_file}") #
    elif log_to_console:
        logger.info(f"MuZero project logging initialized for console output only. Level: {logging.getLevelName(log_level)}.")
    else:
        print("Logging not fully initialized as no handlers were set up.")
    return logger

# Initialize MuZero logger
muzero_logger = setup_muzero_logging(config_mz.MUZERO_LOGS_DIR, log_level=logging.INFO)

if muzero_logger:
    muzero_logger.info("MuZero Project Setup and Configuration Section Initialized.") #
else:
    print("MuZero logger could not be initialized. Check console for warnings.")

# ------------------------------------------------------------------------------
# E. UTILITY FUNCTIONS (Placeholder - Add as needed)
# ------------------------------------------------------------------------------
import json
import numpy as np

def load_ga_feature_mask(mask_path, logger_instance=None):
    if logger_instance is None: logger_instance = logging.getLogger("MuZeroProject")
    if not os.path.exists(mask_path):
        logger_instance.error(f"GA feature mask file NOT FOUND at: {mask_path}")
        return None
    try:
        mask_array = np.loadtxt(mask_path, dtype=int)
        # Ensure it's 1D, sometimes loadtxt might make it 0D if it's a single number, or 2D
        if mask_array.ndim == 0: mask_array = np.array([mask_array]) # Make it 1D
        elif mask_array.ndim > 1: mask_array = mask_array.flatten() # Flatten if it's multi-dimensional

        logger_instance.info(f"Successfully loaded feature mask from {mask_path}. Original shape: {mask_array.shape}, Sum (selected features): {np.sum(mask_array)}") #
        return mask_array.astype(bool)
    except Exception as e:
        logger_instance.error(f"Error loading or processing feature mask from {mask_path}: {e}")
        return None

# --- Global variable for feature mask (to be loaded once) ---
# This will be populated later, before building network h
# For now, just showing how it could be loaded. The actual loading should happen
# right before you define the input shape for network h.
# feature_mask_global = load_ga_feature_mask(config_mz.GA_BEST_MASK_PATH, muzero_logger)
# if feature_mask_global is not None:
#     NUM_MASKED_MARKET_FEATURES = np.sum(feature_mask_global)
#     muzero_logger.info(f"Number of features after applying GA mask: {NUM_MASKED_MARKET_FEATURES}")
# else:
#     muzero_logger.critical("Failed to load GA feature mask. This is likely essential for the representation network.")
    # Consider halting or using all features with a warning if the mask is critical.
    # For this setup, it's assumed the mask is essential.

print("\n--- Section 1: Setup and Initial Configuration Complete ---")

# ==============================================================================
# @title #PRELUDE TO STEP 3: DETERMINE INPUT SHAPES
# ==============================================================================
# Ensure the logger from Section 1 is available
if 'muzero_logger' not in globals() or muzero_logger is None:
    # Fallback if the logger wasn't correctly passed or initialized
    import logging
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(module)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    muzero_logger = logging.getLogger("MuZeroProject_Step3")
    muzero_logger.warning("Re-initialized basic logger for Step 3 as 'muzero_logger' was not found globally.")

# Ensure config_mz from Section 1 is available
if 'config_mz' not in globals() or config_mz is None:
    muzero_logger.critical("MuZero configuration 'config_mz' not found. Please ensure Section 1 ran successfully.")
    # In a real script, you'd raise an error here. For Colab, this cell might just fail.
    raise NameError("MuZero configuration 'config_mz' is not defined.")


muzero_logger.info("--- Prelude to Step 3: Determining Input Shapes ---") #

# Load the GA feature mask
feature_mask_global = load_ga_feature_mask(config_mz.GA_BEST_MASK_PATH, muzero_logger)

if feature_mask_global is not None:
    NUM_MASKED_MARKET_FEATURES = np.sum(feature_mask_global)
    muzero_logger.info(f"Number of market features after applying GA mask: {NUM_MASKED_MARKET_FEATURES}") #
else:
    muzero_logger.critical("Failed to load GA feature mask. This is essential for the representation network's input shape.")
    muzero_logger.info("Please ensure the mask file exists at: {}".format(config_mz.GA_BEST_MASK_PATH))
    muzero_logger.info("If you wish to proceed without a mask (using all features from the original feature list), this needs to be explicitly handled.")
    # For now, we will assume the mask is critical and raise an error if not found.
    raise FileNotFoundError(f"GA Feature Mask not found at {config_mz.GA_BEST_MASK_PATH} and is considered essential.")

# Other necessary dimensions from config_mz (defined in Section 1)
NUM_PORTFOLIO_STATES = config_mz.PORTFOLIO_STATE_SIZE
NUM_ACTIONS = config_mz.NUM_ACTIONS
MUZERO_HIDDEN_STATE_SIZE = config_mz.MUZERO_HIDDEN_STATE_SIZE

muzero_logger.info(f"  NUM_MASKED_MARKET_FEATURES: {NUM_MASKED_MARKET_FEATURES}") #
muzero_logger.info(f"  NUM_PORTFOLIO_STATES (for one-hot encoding): {NUM_PORTFOLIO_STATES}") #
muzero_logger.info(f"  NUM_ACTIONS (for policy output and dynamics input): {NUM_ACTIONS}") #
muzero_logger.info(f"  MUZERO_HIDDEN_STATE_SIZE (for s, s'): {MUZERO_HIDDEN_STATE_SIZE}") #

# ==============================================================================
# STEP 3: IMPLEMENT THE REPRESENTATION NETWORK (h) WITH TRANSFER LEARNING
# ==============================================================================
import tensorflow as tf
from tensorflow import keras

muzero_logger.info("\n--- Step 3: Implementing the Representation Network (h) ---") #

# 1. Load your "Final Best NN" (output from afterford_ga_runner.py)
muzero_logger.info(f"Loading 'Final Best NN' from: {config_mz.FINAL_BEST_NN_PATH}") #
if not os.path.exists(config_mz.FINAL_BEST_NN_PATH):
    muzero_logger.critical(f"Pre-trained model file NOT FOUND at: {config_mz.FINAL_BEST_NN_PATH}")
    raise FileNotFoundError(f"Pre-trained model file not found: {config_mz.FINAL_BEST_NN_PATH}")

try:
    loaded_final_best_nn = keras.models.load_model(config_mz.FINAL_BEST_NN_PATH)
    muzero_logger.info("Successfully loaded 'Final Best NN'. Printing summary:") #
    loaded_final_best_nn.summary(print_fn=muzero_logger.info)
except Exception as e:
    muzero_logger.critical(f"Error loading the 'Final Best NN' from {config_mz.FINAL_BEST_NN_PATH}: {e}")
    raise

# 2. Extract the Base Representation Model (processing market features)
#    CRITICAL: Update config_mz.TRANSFER_LAYER_NAME in Section 1 after inspecting the summary above.
transfer_layer_name = config_mz.TRANSFER_LAYER_NAME
muzero_logger.info(f"Attempting to use layer '{transfer_layer_name}' for transfer learning from the loaded NN.") #

try:
    base_representation_model = keras.Model(
        inputs=loaded_final_best_nn.input, # Input of the loaded forecasting model
        outputs=loaded_final_best_nn.get_layer(transfer_layer_name).output,
        name='Base_Market_Representation'
    )
    muzero_logger.info(f"Base market representation model created using output of layer: '{transfer_layer_name}'.") #
    # Verify the expected input shape of the base_representation_model
    # It should match the number of features your original model was trained on (before masking for MuZero).
    # The GA mask is applied to the *data* fed into this model, not by changing the model structure here.
    muzero_logger.info(f"Input shape expected by base_representation_model: {base_representation_model.input_shape}") #

except ValueError as e:
    muzero_logger.critical(f"Error creating base representation model. Ensure layer '{transfer_layer_name}' exists in loaded_final_best_nn: {e}")
    muzero_logger.critical("Please check the summary of loaded_final_best_nn printed above and CORRECT 'config_mz.TRANSFER_LAYER_NAME' in Section 1, then re-run.")
    raise
except Exception as e:
    muzero_logger.critical(f"An unexpected error occurred while creating the base_representation_model: {e}")
    raise


# 3. Define Input Layers for the full Representation Network h
#    The market_features_input shape here MUST match NUM_MASKED_MARKET_FEATURES
#    However, the base_representation_model itself expects the *original* number of features
#    because the GA mask is applied to the *data* not the model structure here.
#    This means the data fed to 'market_features_input' will be pre-masked.
#    If base_representation_model expects unmasked features, data preparation for MuZero needs to handle this.

#    Correction: The input to base_representation_model should be the *original* features
#    that loaded_final_best_nn expects. The masking happens *before* data is fed to THIS input layer.
#    So, the input layer for the *data that will be fed* to base_representation_model needs to match
#    the original feature count of loaded_final_best_nn.

#    Let's get the original feature count from the loaded model's input shape.
original_feature_count = loaded_final_best_nn.input_shape[-1]
muzero_logger.info(f"Original feature count from loaded_final_best_nn.input_shape: {original_feature_count}") #
# We'll assume the GA mask (feature_mask_global) has this length.
# If NUM_MASKED_MARKET_FEATURES is different, it means some features are always zeroed out by the mask.

# The input data pipeline for MuZero will provide data that is ALREADY MASKED or selected.
# So, the input layer for *this part of h* should expect NUM_MASKED_MARKET_FEATURES.
# THIS IS A SUBTLE POINT. If base_representation_model is used directly, its input layer
# expects the original number of features. If we are building a *new path* for market features
# that involves selecting specific inputs, then the shape would be NUM_MASKED_MARKET_FEATURES.

# Given we are directly using base_representation_model, the data fed to its input must be
# the original feature set, and the *effect* of the mask is that some of those features might be zero.
# However, for clarity and to match standard MuZero where 'h' takes the *observed state*,
# it's better if the input layer explicitly matches what we feed it.

# Let's assume the data preparation stage for MuZero will handle providing the
# correctly shaped (masked) input. So market_features_input for h should have shape NUM_MASKED_MARKET_FEATURES.
# The `base_representation_model` will then need to be rebuilt or adapted if its internal structure
# relies on the original feature positions. This is a key challenge with transfer learning and feature selection.

# Simplest approach for transfer learning with a feature mask:
# The `base_representation_model` is ALREADY TRAINED on all original features.
# When you feed data to MuZero's `h` network:
# 1. Take the full original feature vector for the current timestep.
# 2. Apply your `feature_mask_global` to it (e.g., by multiplication: `masked_input_data = full_input_data * feature_mask_global`).
#    This effectively feeds zeros for the deselected features.
# 3. This `masked_input_data` (which still has the original feature dimension, but with zeros) is fed to `base_representation_model`.

# Therefore, the input layer for our `h` network that *wraps* `base_representation_model` should match
# the input of `base_representation_model`.
market_features_input_h = keras.layers.Input(shape=(original_feature_count,), name='market_features_input_h')

portfolio_state_input_h = keras.layers.Input(shape=(NUM_PORTFOLIO_STATES,), name='portfolio_state_input_h')

# 4. Process Market Features using the (potentially frozen) base model
#    The data fed to market_features_input_h will need to be the original feature vector,
#    potentially with masked features zeroed out *before* being passed.
encoded_market_features = base_representation_model(market_features_input_h)

# 5. Combine with portfolio state and finalize the hidden state s
concatenated_for_h = keras.layers.Concatenate(name='h_concatenate')([encoded_market_features, portfolio_state_input_h])
hidden_state_s_output = keras.layers.Dense(
    MUZERO_HIDDEN_STATE_SIZE,
    activation='relu',
    name='h_final_dense_to_s'
)(concatenated_for_h)

representation_network_h = keras.Model(
    inputs=[market_features_input_h, portfolio_state_input_h],
    outputs=hidden_state_s_output,
    name='RepresentationNetwork_h'
)
muzero_logger.info("Representation Network (h) defined. Summary:") #
representation_network_h.summary(print_fn=muzero_logger.info)

# 6. Weight Freezing (Initial Strategy)
if config_mz.FREEZE_TRANSFER_WEIGHTS_INITIALLY:
    base_representation_model.trainable = False
    muzero_logger.info(f"Initial training: base_representation_model (transfer learned part of h) is FROZEN (trainable = {base_representation_model.trainable}).") #
else:
    base_representation_model.trainable = True
    muzero_logger.info(f"Initial training: base_representation_model (transfer learned part of h) is NOT frozen (trainable = {base_representation_model.trainable}).")

muzero_logger.info("--- Step 3: Representation Network (h) Implementation Complete ---") #

# Ensure the logger and config_mz from Section 1, and other relevant variables
# like NUM_ACTIONS and MUZERO_HIDDEN_STATE_SIZE from the "Prelude to Step 3" are available.

if 'muzero_logger' not in globals() or muzero_logger is None:
    import logging
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(module)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    muzero_logger = logging.getLogger("MuZeroProject_Step4_5")
    muzero_logger.warning("Re-initialized basic logger for Step 4-5 as 'muzero_logger' was not found globally.")

if 'config_mz' not in globals() or config_mz is None:
    muzero_logger.critical("MuZero configuration 'config_mz' not found. Please ensure Section 1 ran successfully.")
    raise NameError("MuZero configuration 'config_mz' is not defined.")

if 'NUM_ACTIONS' not in globals() or 'MUZERO_HIDDEN_STATE_SIZE' not in globals():
    muzero_logger.critical("NUM_ACTIONS or MUZERO_HIDDEN_STATE_SIZE not found. Ensure 'Prelude to Step 3' ran successfully.")
    raise NameError("Required shape variables (NUM_ACTIONS, MUZERO_HIDDEN_STATE_SIZE) are not defined.")

import tensorflow as tf
from tensorflow import keras

# ==============================================================================
# @title #STEP 4: IMPLEMENT THE DYNAMICS NETWORK (g)
# ==============================================================================
muzero_logger.info("\n--- Step 4: Implementing the Dynamics Network (g) ---") #

# Inputs for the Dynamics Network g
previous_hidden_state_input_g = keras.layers.Input(shape=(MUZERO_HIDDEN_STATE_SIZE,), name='g_prev_hidden_state_input')
# Action should be one-hot encoded, so shape is (NUM_ACTIONS,)
action_input_g = keras.layers.Input(shape=(NUM_ACTIONS,), name='g_action_input')

# Process combined input
concatenated_dynamics_input = keras.layers.Concatenate(name='g_concatenate')([previous_hidden_state_input_g, action_input_g])

# Branch 1: Predict Next Hidden State (s')
dynamics_state_hidden1 = keras.layers.Dense(
    config_mz.DYNAMICS_INTERNAL_SIZE,
    activation='relu',
    name='g_dense1_state_prediction'
)(concatenated_dynamics_input)
# You could add more layers for complexity if needed, e.g.:
# dynamics_state_hidden2 = keras.layers.Dense(config_mz.DYNAMICS_INTERNAL_SIZE // 2, activation='relu', name='g_dense2_state_prediction')(dynamics_state_hidden1)
next_hidden_state_s_prime_output = keras.layers.Dense(
    MUZERO_HIDDEN_STATE_SIZE, # Output must match the hidden state size
    activation='relu', # Keep consistent with the hidden state's nature from network h
    name='g_output_next_state'
)(dynamics_state_hidden1) # or dynamics_state_hidden2 if you add another layer

# Branch 2: Predict Reward (r)
dynamics_reward_hidden1 = keras.layers.Dense(
    config_mz.DYNAMICS_INTERNAL_SIZE_REWARD,
    activation='relu',
    name='g_dense1_reward_prediction'
)(concatenated_dynamics_input)
predicted_reward_r_output = keras.layers.Dense(
    1, # Single scalar value for reward
    activation='linear', # Rewards can be positive or negative, no squashing needed here
                         # unless you plan to normalize rewards to a specific range like [-1, 1] during training.
                         # If so, you might use 'tanh' here and scale target rewards accordingly.
    name='g_output_reward'
)(dynamics_reward_hidden1)

dynamics_network_g = keras.Model(
    inputs=[previous_hidden_state_input_g, action_input_g],
    outputs=[next_hidden_state_s_prime_output, predicted_reward_r_output],
    name='DynamicsNetwork_g'
)
muzero_logger.info("Dynamics Network (g) defined. Summary:") #
dynamics_network_g.summary(print_fn=muzero_logger.info)

muzero_logger.info("--- Step 4: Dynamics Network (g) Implementation Complete ---") #

# ==============================================================================
# @title #STEP 5: IMPLEMENT THE PREDICTION NETWORK (f)
# ==============================================================================
muzero_logger.info("\n--- Step 5: Implementing the Prediction Network (f) ---") #

# Input for the Prediction Network f
current_hidden_state_input_f = keras.layers.Input(shape=(MUZERO_HIDDEN_STATE_SIZE,), name='f_current_hidden_state_input')

# Branch 1: Predict Policy (p) - Action probabilities
prediction_policy_hidden1 = keras.layers.Dense(
    config_mz.PREDICTION_INTERNAL_SIZE_POLICY,
    activation='relu',
    name='f_dense1_policy_prediction'
)(current_hidden_state_input_f)
# Output raw logits for the policy. Softmax is applied either during MCTS action selection (to sample)
# or within the loss function (e.g., tf.nn.softmax_cross_entropy_with_logits).
policy_logits_p_output = keras.layers.Dense(
    NUM_ACTIONS, # Output size matches the number of defined actions
    name='f_output_policy_logits'
)(prediction_policy_hidden1)

# Branch 2: Predict Value (v) - Expected cumulative future reward
prediction_value_hidden1 = keras.layers.Dense(
    config_mz.PREDICTION_INTERNAL_SIZE_VALUE,
    activation='relu',
    name='f_dense1_value_prediction'
)(current_hidden_state_input_f)
predicted_value_v_output = keras.layers.Dense(
    1, # Single scalar value for the state's value
    activation='tanh', # Common to use tanh if values/rewards are scaled to [-1, 1].
                       # If not, 'linear' might be better. MuZero often scales value targets.
    name='f_output_value'
)(prediction_value_hidden1)

prediction_network_f = keras.Model(
    inputs=current_hidden_state_input_f,
    outputs=[policy_logits_p_output, predicted_value_v_output],
    name='PredictionNetwork_f'
)
muzero_logger.info("Prediction Network (f) defined. Summary:") #
prediction_network_f.summary(print_fn=muzero_logger.info)

muzero_logger.info("--- Step 5: Prediction Network (f) Implementation Complete ---") #

# ==============================================================================
# @title #STEP 6: THE "MUZERO BEST MODEL" (CONCEPTUAL COMBINATION)
# ==============================================================================
muzero_logger.info("\n--- Step 6: The 'MuZero Best Model' (Conceptual Combination) ---") #
muzero_logger.info("The MuZero 'model' consists of three interconnected neural networks:") #
muzero_logger.info(f"1. Representation Network (h): Takes (market_state, portfolio_state) -> hidden_state_s. Name: {representation_network_h.name}") #
muzero_logger.info(f"   - Input shapes: {representation_network_h.input_shape}") #
muzero_logger.info(f"   - Output shape: {representation_network_h.output_shape}") #
muzero_logger.info(f"   - Base market representation part is initially frozen: {not base_representation_model.trainable if 'base_representation_model' in globals() else 'N/A'}") #

muzero_logger.info(f"2. Dynamics Network (g): Takes (prev_hidden_state_s, action) -> (next_hidden_state_s', reward_r). Name: {dynamics_network_g.name}") #
muzero_logger.info(f"   - Input shapes: {dynamics_network_g.input_shape}") #
muzero_logger.info(f"   - Output shapes: {dynamics_network_g.output_shape}") #

muzero_logger.info(f"3. Prediction Network (f): Takes current_hidden_state_s -> (policy_logits_p, value_v). Name: {prediction_network_f.name}") #
muzero_logger.info(f"   - Input shape: {prediction_network_f.input_shape}") #
muzero_logger.info(f"   - Output shapes: {prediction_network_f.output_shape}") #

muzero_logger.info("These networks are trained jointly as part of the MuZero algorithm.") #
muzero_logger.info("When saving a 'MuZero checkpoint', you would save the weights of all three networks.") #
muzero_logger.info("--- Step 6: Conceptual Understanding Complete ---") #

# ==============================================================================
# @title #STEP 7: TRAINING STRATEGY (BRIEF OVERVIEW) - Conceptual
# ==============================================================================
muzero_logger.info("\n--- Step 7: Training Strategy (Brief Overview) ---") #
muzero_logger.info("1. Initial Training Phase:") #
muzero_logger.info(f"   - Keep 'base_representation_model.trainable = {config_mz.FREEZE_TRANSFER_WEIGHTS_INITIALLY}'.") #
muzero_logger.info(f"   - Train h (new parts), g, and f using MCTS-generated data and MuZero loss functions.") #
muzero_logger.info(f"   - This allows g, f, and new parts of h to adapt to the fixed, pre-trained market features.") #

muzero_logger.info("2. Fine-Tuning Phase:") #
muzero_logger.info(f"   - After approx. {config_mz.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP} training steps (or based on performance):") #
muzero_logger.info("     Set 'base_representation_model.trainable = True'.") #
muzero_logger.info(f"   - Continue training. Now, the entire system (h, g, f) learns end-to-end.") #
muzero_logger.info(f"   - Pre-trained weights in 'base_representation_model' will be updated/fine-tuned by MuZero's learning process.") #
muzero_logger.info("--- Step 7: Training Strategy Overview Complete ---") #

Running in Google Colab environment.
Google Drive already mounted.
GPU detected: Tesla T4, 15360 MiB

Installing/Checking Python packages...
tensorflow already installed.
numpy already installed.
pandas already installed.
Installing conda-package-handling...
conda-package-handling installed successfully.

Checking TA-Lib installation for Colab...
TA-Lib not found, attempting installation using method from 'afterford_load_data.py'...
  Checking for TA-Lib C library...
  TA-Lib C library not found. Downloading and installing...
  TA-Lib C library installed successfully.

  Checking for TA-Lib Python wrapper...
  TA-Lib Python wrapper not found at /usr/local/lib/python3.11/dist-packages/talib. Downloading and installing...
    Removing old extracted directory ./ta-lib-0.5.1-py311h9ecbd09_0 if it exists...
    Extracting ta-lib-0.5.1-py311h9ecbd09_0.conda...
    Moving ./ta-lib-0.5.1-py311h9ecbd09_0/lib/python3.11/site-packages/talib to /usr/local/lib/python3.11/dist-packages/talib...
    

2025-06-09 10:17:52 - INFO - [summary_utils.py:389] - summary_utils - Model: "Simple_Model_with_Autoencoder"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer (InputLayer)        │ (None, 1272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_L1 (Dense)                │ (None, 83)             │       105,659 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ LeakyReLU_1 (LeakyReLU)         │ (None, 83)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_L2 (Dense)                │ (None, 800)            │        67,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GaussianNoise (GaussianNoise)   │ (None, 8

2025-06-09 10:17:52 - INFO - [summary_utils.py:389] - summary_utils - Model: "RepresentationNetwork_h"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ market_features_in… │ (None, 1272)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Base_Market_Repres… │ (None, 800)       │    199,271 │ market_features_… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ portfolio_state_in… │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼───────

2025-06-09 10:17:52 - INFO - [summary_utils.py:389] - summary_utils - Model: "DynamicsNetwork_g"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ g_prev_hidden_stat… │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ g_action_input      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ g_concatenate       │ (None, 259)       │          0 │ g_prev_hidden_st… │
│ (Concatenate)       │                   │            │ g_action_input[0… │
├─────────────────────┼───────────────────┼────────────┼

2025-06-09 10:17:52 - INFO - [summary_utils.py:389] - summary_utils - Model: "PredictionNetwork_f"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ f_current_hidden_s… │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f_dense1_policy_pr… │ (None, 128)       │     32,896 │ f_current_hidden… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f_dense1_value_pre… │ (None, 64)        │     16,448 │ f_current_hidden… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼───────────

In [ ]:
# This entire block should be run as a SINGLE CELL
# @title MAIN RUN Section 12 NN 12-1024 Policy with Dropout Modified for Specific Architecture (No Optuna) - Softmax Debug

import tensorflow as tf
from tensorflow import keras
import numpy as np
import time
import os
import math
import collections
import json
import logging
import sys
import random
import pandas as pd
import pickle
import copy

# --- Basic Logger & Config Check ---
if 'muzero_logger' not in globals() or muzero_logger is None:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(module)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    muzero_logger = logging.getLogger("MuZeroProject_SpecificArch_SoftmaxDebug")
    muzero_logger.warning("Re-initialized basic logger for Specific Architecture run with Softmax Debug.")

essential_globals_for_script = ['config_mz', 'muzero_logger', 'load_ga_feature_mask']
for var_name in essential_globals_for_script:
    if var_name not in globals() or globals()[var_name] is None:
        muzero_logger.critical(f"Essential global component '{var_name}' from Section 1 is missing. Halting.")
        raise NameError(f"Missing Section 1 component '{var_name}'.")

# Ensure these are set in your config_mz object before running:
# Example of how to set them in your config loading section:
# config_mz.FREEZE_H_NET_MAIN_RUN = True
# config_mz.POLICY_HEAD_ARCHITECTURE = 'custom_v2'
# config_mz.POLICY_HEAD_V2_SETTINGS = {
# 'dense1_units': 12, 'noise1_stddev': 0.2,
# 'dense2_units': 1024, 'noise2_stddev': 0.2, 'dropout_rate': 0.2
# }
# # Ensure other necessary configs are present
if not hasattr(config_mz, 'MUZERO_CHECKPOINT_FORMAT'):
    config_mz.MUZERO_CHECKPOINT_FORMAT = os.path.join(config_mz.MUZERO_MODELS_DIR, "ckpt_S{step}")
if not hasattr(config_mz, 'VALUE_TARGET_SCALING_DIVISOR'):
    config_mz.VALUE_TARGET_SCALING_DIVISOR = 1.0
if not hasattr(config_mz, 'REWARD_TARGET_SCALING_DIVISOR'):
    config_mz.REWARD_TARGET_SCALING_DIVISOR = 1.0
if not hasattr(config_mz, 'ANNEAL_TEMPERATURE_AFTER_N_MOVES'):
    # Default annealing halfway through max training steps if not set
    config_mz.ANNEAL_TEMPERATURE_AFTER_N_MOVES = getattr(config_mz, 'MAX_TRAINING_STEPS', 300000) // 2


# ==============================================================================
# PRELUDE: Load Feature Lists and Masks
# ==============================================================================
muzero_logger.info("--- MONOLITH PRELUDE: Loading Training Features List and GA Mask ---")
TRAINED_FEATURE_NAMES_GLOBAL = None
ORIGINAL_MODEL_FEATURE_COUNT = 0
if not hasattr(config_mz, 'FEATURE_LIST_PATH') or not os.path.exists(config_mz.FEATURE_LIST_PATH):
    raise FileNotFoundError(f"Training features file path missing/invalid in config: {getattr(config_mz, 'FEATURE_LIST_PATH', 'N/A')}")
try:
    with open(config_mz.FEATURE_LIST_PATH, 'r') as f: TRAINED_FEATURE_NAMES_GLOBAL = json.load(f)
    muzero_logger.info(f"Loaded {len(TRAINED_FEATURE_NAMES_GLOBAL)} feature names from {config_mz.FEATURE_LIST_PATH}.")
    ORIGINAL_MODEL_FEATURE_COUNT = len(TRAINED_FEATURE_NAMES_GLOBAL)
except Exception as e: muzero_logger.critical(f"Error loading training features: {e}"); raise
feature_mask_global = load_ga_feature_mask(config_mz.GA_BEST_MASK_PATH, muzero_logger)
if feature_mask_global is None: raise FileNotFoundError(f"GA Feature Mask not found: {config_mz.GA_BEST_MASK_PATH}")
if len(feature_mask_global) != ORIGINAL_MODEL_FEATURE_COUNT: raise ValueError("GA mask length mismatch with training features.")
NUM_EFFECTIVE_MASKED_FEATURES = np.sum(feature_mask_global)
muzero_logger.info(f"GA Mask loaded. Length: {len(feature_mask_global)}. Effective features: {NUM_EFFECTIVE_MASKED_FEATURES}")
NUM_PORTFOLIO_STATES = config_mz.PORTFOLIO_STATE_SIZE
NUM_ACTIONS = config_mz.NUM_ACTIONS
MUZERO_HIDDEN_STATE_SIZE = config_mz.MUZERO_HIDDEN_STATE_SIZE
muzero_logger.info("--- MONOLITH PRELUDE Complete ---")

essential_globals_after_prelude = ['TRAINED_FEATURE_NAMES_GLOBAL', 'feature_mask_global', 'ORIGINAL_MODEL_FEATURE_COUNT']
for var_name in essential_globals_after_prelude:
    if var_name not in globals() or globals()[var_name] is None:
        raise NameError(f"Essential Prelude component '{var_name}' missing.")
muzero_logger.info("Prelude global variable checks passed.")

# ==============================================================================
# STEP 8: FinancialMuZeroEnv
# ==============================================================================
muzero_logger.info("\n--- Defining Step 8: FinancialMuZeroEnv ---")
class FinancialMuZeroEnv:
    def __init__(self, config_mz_instance, logger_instance,
                 trained_feature_names_list, global_ga_mask_arr,
                 start_index=0, episode_length=None):
        self.config = config_mz_instance; self.logger = logger_instance
        self.trained_feature_names = trained_feature_names_list
        self.global_ga_mask = global_ga_mask_arr
        self.original_model_feature_count = ORIGINAL_MODEL_FEATURE_COUNT
        self.start_index = start_index
        self.initial_episode_length = episode_length
        self.episode_length = episode_length
        self.POSITION_SPXL, self.POSITION_SPXS, self.POSITION_CASH = 0, 1, 2
        self.portfolio_state_names = {self.POSITION_SPXL: "SPXL", self.POSITION_SPXS: "SPXS", self.POSITION_CASH: "CASH"}
        self.num_portfolio_states = len(self.portfolio_state_names)
        self.ACTION_TARGET_SPXL, self.ACTION_TARGET_SPXS, self.ACTION_TARGET_CASH = 0, 1, 2
        self.num_actions = self.config.NUM_ACTIONS
        self.transaction_cost_bps = getattr(self.config, 'TRANSACTION_COST_BPS', 10)
        self.leverage = getattr(self.config, 'LEVERAGE', 3.0)
        self._load_and_prepare_data()
        self.current_step_in_episode = 0; self.current_portfolio_position = self.POSITION_CASH
        self.current_data_idx = self.start_index; self.done = False
        self.episode_start_data_idx = self.start_index
        self.logger.info(f"FinancialMuZeroEnv initialized. Usable steps: {self.total_data_steps}. Initial default ep length: {self.initial_episode_length if self.initial_episode_length is not None else 'End of Data'}.")

    def _rename_etf_columns(self, df):
        new_cols = {c: c.replace("SPXS_('Open', 'SPXS')", 'SPXS_Open').replace("SPXS_('Close', 'SPXS')", 'SPXS_Close')
                         .replace("SPXL_('Open', 'SPXL')", 'SPXL_Open').replace("SPXL_('Close', 'SPXL')", 'SPXL_Close')
                         .replace("SPXS_('High', 'SPXS')", 'SPXS_High').replace("SPXS_('Low', 'SPXS')", 'SPXS_Low')
                         .replace("SPXS_('Volume', 'SPXS')", 'SPXS_Volume')
                         .replace("SPXL_('High', 'SPXL')", 'SPXL_High').replace("SPXL_('Low', 'SPXL')", 'SPXL_Low')
                         .replace("SPXL_('Volume', 'SPXL')", 'SPXL_Volume')
                      for c in df.columns}
        df.rename(columns=new_cols, inplace=True); return df

    def _load_and_prepare_data(self):
            date_col = self.config.existing_config_module.DATE_COLUMN
            data_csv = pd.read_csv(self.config.MARKET_DATA_PATH, index_col=date_col, parse_dates=True)
            etf_raw = pd.read_csv(self.config.ETF_DATA_PATH, index_col=date_col, parse_dates=True)
            etf_proc = self._rename_etf_columns(etf_raw.copy())
            req_etf = ['SPXL_Close', 'SPXS_Close', 'SPXL_Open', 'SPXS_Open']
            if not all(c in etf_proc.columns for c in req_etf): raise ValueError(f"ETF data missing required columns. Found: {etf_proc.columns.tolist()}")

            aligned_features = pd.DataFrame(index=data_csv.index, columns=self.trained_feature_names)
            cols_to_fill = [c for c in self.trained_feature_names if c in data_csv.columns]
            if cols_to_fill: aligned_features[cols_to_fill] = data_csv[cols_to_fill]
            missing = set(self.trained_feature_names) - set(cols_to_fill)
            if missing:
                self.logger.warning(f"{len(missing)} features expected by model were MISSING in data.csv and filled with 0s: {list(missing)[:5]}...")
                aligned_features[list(missing)] = 0.0

            common_idx = aligned_features.index.intersection(etf_proc.index)
            market_features_all_aligned = aligned_features.loc[common_idx].sort_index()
            etf_prices_all_aligned = etf_proc.loc[common_idx].sort_index()

            if market_features_all_aligned.empty: raise ValueError("Data empty after alignment.")

            num_oos_days = 252
            if len(market_features_all_aligned) > num_oos_days:
                self.logger.info(f"Holding out the last {num_oos_days} days for out-of-sample testing.")
                self.market_features_df_raw_aligned = market_features_all_aligned.iloc[:-num_oos_days]
                self.etf_prices_df = etf_prices_all_aligned.iloc[:-num_oos_days]
                self.logger.info(f"Training data will use {len(self.market_features_df_raw_aligned)} days.")
            else:
                self.logger.warning(f"Total available data ({len(market_features_all_aligned)} days) is not enough to hold out {num_oos_days} days. Using all available data for training.")
                self.market_features_df_raw_aligned = market_features_all_aligned
                self.etf_prices_df = etf_prices_all_aligned
            if self.market_features_df_raw_aligned.empty: raise ValueError("Training data became empty after attempting to hold out OOS data.")

            self.market_features_df = self.market_features_df_raw_aligned.copy()
            if len(self.global_ga_mask) != self.market_features_df.shape[1]:
                raise ValueError(f"GA Mask length ({len(self.global_ga_mask)}) != Aligned Market Features cols ({self.market_features_df.shape[1]})")
            self.market_features_df.loc[:, ~self.global_ga_mask] = 0.0
            self.total_data_steps = len(self.market_features_df)
            self.logger.info(f"Data prep done for training. Market features (masked): {self.market_features_df.shape}. Effective non-zero: {np.sum(self.global_ga_mask)}. Total usable steps for this env instance: {self.total_data_steps}")

    def _get_observation(self):
        market_obs = np.zeros(self.original_model_feature_count, dtype=np.float32)
        if self.current_data_idx < self.total_data_steps:
            market_obs = self.market_features_df.iloc[self.current_data_idx].values.astype(np.float32)
        portfolio_obs = np.zeros(self.num_portfolio_states, dtype=np.float32); portfolio_obs[self.current_portfolio_position] = 1.0
        return [market_obs, portfolio_obs]

    def reset(self, start_index=None, episode_length=None):
        self.current_step_in_episode = 0; self.done = False; self.current_portfolio_position = self.POSITION_CASH
        self.episode_start_data_idx = start_index if start_index is not None else self.start_index
        self.current_data_idx = self.episode_start_data_idx

        if episode_length is not None:
            current_ep_len_to_set = episode_length
        elif self.initial_episode_length is not None:
            current_ep_len_to_set = self.initial_episode_length
        else:
            current_ep_len_to_set = self.total_data_steps - self.current_data_idx

        self.episode_length = max(0, min(current_ep_len_to_set, self.total_data_steps - self.current_data_idx))

        if self.current_data_idx >= self.total_data_steps:
            self.logger.warning(f"Reset start index {self.current_data_idx} is out of bounds. Setting done.")
            self.done = True; self.episode_length = 0
            return self._get_observation()
        self.logger.debug(f"Env reset. Start: {self.current_data_idx}. Pos: {self.portfolio_state_names[self.current_portfolio_position]}. Effective EpLen for this ep: {self.episode_length}.")
        return self._get_observation()

    def step(self, action):
        if self.done: return self._get_observation(), 0.0, True, {'status': 'Episode ended'}
        if self.current_data_idx + 1 >= self.total_data_steps:
            self.logger.warning(f"Env step: End of data for ETF prices at index {self.current_data_idx + 1} (current_data_idx: {self.current_data_idx}).")
            self.done = True; return self._get_observation(), 0.0, True, {'status': 'End of data for reward'}

        next_day_prices = self.etf_prices_df.iloc[self.current_data_idx + 1]
        spxl_open, spxl_close = next_day_prices.get('SPXL_Open', np.nan), next_day_prices.get('SPXL_Close', np.nan)
        spxs_open, spxs_close = next_day_prices.get('SPXS_Open', np.nan), next_day_prices.get('SPXS_Close', np.nan)

        raw_reward, prev_pos, transacted = 0.0, self.current_portfolio_position, False
        target_pos = action if action in range(self.num_actions) else self.POSITION_CASH

        if prev_pos != target_pos:
            transacted = True
            if prev_pos != self.POSITION_CASH: raw_reward -= (self.transaction_cost_bps / 10000.0)
            if target_pos != self.POSITION_CASH: raw_reward -= (self.transaction_cost_bps / 10000.0)

        pnl = 0.0
        if target_pos == self.POSITION_SPXL:
            if not pd.isna(spxl_open) and spxl_open != 0 and not pd.isna(spxl_close): pnl = self.leverage * (spxl_close - spxl_open) / spxl_open
            elif pd.isna(spxl_open) or pd.isna(spxl_close): self.logger.warning(f"SPXL price NaN at data_idx {self.current_data_idx+1}")
        elif target_pos == self.POSITION_SPXS:
            if not pd.isna(spxs_open) and spxs_open != 0 and not pd.isna(spxs_close): pnl = self.leverage * (spxs_close - spxs_open) / spxs_open
            elif pd.isna(spxs_open) or pd.isna(spxs_close): self.logger.warning(f"SPXS price NaN at data_idx {self.current_data_idx+1}")

        if pd.isna(pnl): pnl = 0.0

        raw_reward += pnl; self.current_portfolio_position = target_pos
        self.current_step_in_episode += 1; self.current_data_idx += 1

        if (self.episode_length > 0 and self.current_step_in_episode >= self.episode_length) or \
           self.current_data_idx >= self.total_data_steps:
            self.done = True

        next_obs = self._get_observation()
        info = {'ts': self.market_features_df.index[self.current_data_idx-1] if self.current_data_idx > 0 and self.current_data_idx <= len(self.market_features_df) else "N/A",
                'act':action, 'tgt_pos': self.portfolio_state_names.get(target_pos), 'raw_reward': raw_reward}
        return next_obs, raw_reward, self.done, info
    def get_observation_space_shape(self): return [(self.original_model_feature_count,), (self.num_portfolio_states,)]
    def get_action_space_size(self): return self.num_actions
muzero_logger.info("--- Step 8: FinancialMuZeroEnv Class Defined ---")

# ==============================================================================
# STEP 9: MCTS
# ==============================================================================
muzero_logger.info("\n--- Step 9: Defining Monte Carlo Tree Search (MCTS) ---")
class MCTSNode:
    def __init__(self, parent=None, action_that_led_here=None, prior_p=0.0, hidden_state=None):
        self.parent, self.action_that_led_here, self.prior_p, self.hidden_state = parent, action_that_led_here, prior_p, hidden_state
        self.children, self.visit_count, self.value_sum, self.reward_from_parent_action = {}, 0, 0.0, 0.0
        self.policy_p_values_from_f_net, self.value_v_from_f_net = None, None
    def get_mean_value(self): return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0
    def is_expanded(self): return self.policy_p_values_from_f_net is not None
    def select_child_with_ucb(self, config):
        best_score, best_action, best_child_node = -float('inf'), -1, None
        if not self.is_expanded(): muzero_logger.error("UCB on non-expanded node!"); return np.random.choice(config.NUM_ACTIONS), None
        for action_idx in range(config.NUM_ACTIONS):
            q_val, visits = 0.0, 0
            if action_idx in self.children: child = self.children[action_idx]; q_val = child.reward_from_parent_action + config.DISCOUNT_FACTOR * child.get_mean_value(); visits = child.visit_count
            prior = self.policy_p_values_from_f_net[action_idx]
            parent_visit_count_for_ucb = self.visit_count if self.visit_count > 0 else 1
            ucb_exploration_component = prior * (config.MCTS_PB_C_BASE + math.log((parent_visit_count_for_ucb + config.MCTS_PB_C_BASE + 1) / config.MCTS_PB_C_BASE)) * (math.sqrt(parent_visit_count_for_ucb) / (1 + visits))
            ucb = q_val + ucb_exploration_component
            if ucb > best_score: best_score, best_action, best_child_node = ucb, action_idx, self.children.get(action_idx)
        if best_action == -1 and config.NUM_ACTIONS > 0: best_action = np.random.choice(config.NUM_ACTIONS); best_child_node = self.children.get(best_action)
        return best_action, best_child_node

def run_mcts(initial_observation, h_net_func, g_net_func, f_net_func, config_obj, num_simulations_mcts, num_actions_mcts):
    obs_np = [np.array(obs, dtype=np.float32)[np.newaxis,...] for obs in initial_observation]
    root_s = h_net_func.predict_on_batch(obs_np)
    root = MCTSNode(hidden_state=root_s)

    policy_logits_at_root, value_tanh_at_root, _ = f_net_func.predict_on_batch(root.hidden_state)
    probs_r = tf.nn.softmax(policy_logits_at_root[0]).numpy()
    root.value_v_from_f_net = value_tanh_at_root[0,0]
    root.policy_p_values_from_f_net = probs_r

    if config_obj.MCTS_ROOT_DIRICHLET_ALPHA > 0: root.policy_p_values_from_f_net = (1-config_obj.MCTS_ROOT_EXPLORATION_FRACTION)*root.policy_p_values_from_f_net + config_obj.MCTS_ROOT_EXPLORATION_FRACTION*np.random.dirichlet([config_obj.MCTS_ROOT_DIRICHLET_ALPHA]*num_actions_mcts)
    for _ in range(num_simulations_mcts):
        node, path, action_sel = root, [root], -1
        while node.is_expanded():
            action_sel, child_cand = node.select_child_with_ucb(config_obj)
            if child_cand is not None: node = child_cand; path.append(node)
            else: break
        parent_exp, action_exp = node, action_sel
        val_backup = 0.0
        if parent_exp.children.get(action_exp) is None :
            action_oh = np.zeros(num_actions_mcts,dtype=np.float32); action_oh[action_exp] = 1.0
            next_s, reward_g_pred_tanh, _ = g_net_func.predict_on_batch(
                [parent_exp.hidden_state, action_oh[np.newaxis,...]]
            )
            reward_g_pred_scalar = reward_g_pred_tanh[0,0]
            policy_logits_new, value_tanh_new, _ = f_net_func.predict_on_batch(next_s)
            new_node = MCTSNode(parent=parent_exp,action_that_led_here=action_exp,prior_p=parent_exp.policy_p_values_from_f_net[action_exp],hidden_state=next_s)
            new_node.reward_from_parent_action=reward_g_pred_scalar
            new_node.policy_p_values_from_f_net=tf.nn.softmax(policy_logits_new[0]).numpy();
            new_node.value_v_from_f_net=value_tanh_new[0,0]
            parent_exp.children[action_exp]=new_node; path.append(new_node)
            val_backup = new_node.value_v_from_f_net
        else: val_backup = node.value_v_from_f_net if node.value_v_from_f_net is not None else 0.0

        for node_bp in reversed(path):
            node_bp.visit_count+=1
            if node_bp.parent is not None: val_backup = node_bp.reward_from_parent_action + config_obj.DISCOUNT_FACTOR * val_backup
            node_bp.value_sum += val_backup

    visits = np.array([root.children[a].visit_count if a in root.children else 0 for a in range(num_actions_mcts)])
    action_probs = (visits/np.sum(visits)) if np.sum(visits)>0 else (np.ones(num_actions_mcts)/num_actions_mcts)
    return action_probs, root.get_mean_value(), root
muzero_logger.info("--- Step 9: MCTS Implementation Defined ---")

# ==============================================================================
# STEP 10: REPLAY BUFFER
# ==============================================================================
muzero_logger.info("\n--- Step 10: Defining Replay Buffer ---")
class ReplayBuffer:
    def __init__(self, config, logger):
        self.config, self.logger, self.buffer_size = config, logger, config.REPLAY_BUFFER_SIZE
        self.buffer = collections.deque(maxlen=self.buffer_size); self.num_games_added, self.num_samples_drawn = 0,0
        self.save_path = os.path.join(self.config.MUZERO_REPLAY_BUFFER_DIR, "replay_buffer.pkl")
        self.logger.debug(f"ReplayBuffer init. Size: {self.buffer_size}. Path: {self.save_path}")
    def add_game(self, game_trajectory):
        if len(self.buffer) >= self.buffer_size and self.buffer_size > 0 : self.buffer.popleft()
        self.buffer.append(game_trajectory); self.num_games_added += 1
        if self.num_games_added % 10 == 0: self.logger.debug(f"Game {self.num_games_added} added. Buffer: {len(self.buffer)}/{self.buffer_size}")
    def sample_batch(self, batch_size):
        if not self.buffer: self.logger.warning("Buffer empty, cannot sample."); return None
        actual_batch_size = min(batch_size, len(self.buffer))
        unroll_K = self.config.NUM_UNROLL_STEPS
        indices = np.random.choice(len(self.buffer), size=actual_batch_size, replace=len(self.buffer) < batch_size)
        samples = []
        for i in indices:
            game = self.buffer[i]; game_len_actions = len(game['actions_list'])
            if game_len_actions < 1: continue
            max_start_idx_for_obs = len(game['observations_list']) -1
            if max_start_idx_for_obs < 0 : continue
            start_idx = np.random.randint(0, max_start_idx_for_obs + 1)
            obs_init = game['observations_list'][start_idx]
            actions, rewards_target, policies_target, values_target = [], [], [], []
            policies_target.append(game['mcts_policy_targets_list'][start_idx])
            values_target.append(game['value_targets_list'][start_idx])
            for k_idx in range(unroll_K):
                action_step_idx = start_idx + k_idx
                next_state_target_idx = start_idx + k_idx + 1
                if action_step_idx < game_len_actions:
                    actions.append(game['actions_list'][action_step_idx])
                    rewards_target.append(game['rewards_list'][action_step_idx])
                    if next_state_target_idx < len(game['observations_list']):
                        policies_target.append(game['mcts_policy_targets_list'][next_state_target_idx])
                        values_target.append(game['value_targets_list'][next_state_target_idx])
                    else: policies_target.append(np.ones(self.config.NUM_ACTIONS)/self.config.NUM_ACTIONS); values_target.append(0.0)
                else:
                    actions.append(np.random.randint(0,self.config.NUM_ACTIONS)); rewards_target.append(0.0)
                    policies_target.append(np.ones(self.config.NUM_ACTIONS)/self.config.NUM_ACTIONS); values_target.append(0.0)
            samples.append({'initial_observation':obs_init, 'action_history':actions,
                            'target_rewards':rewards_target,
                            'target_policies':policies_target,
                            'target_values':values_target})
        return samples if samples else None
    def __len__(self): return len(self.buffer)
    def save(self):
        try:
            os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
            with open(self.save_path,'wb') as f: pickle.dump(list(self.buffer),f)
            self.logger.info(f"Replay buffer saved to {self.save_path}. Size: {len(self.buffer)} games.")
        except Exception as e: self.logger.error(f"Error saving replay buffer: {e}", exc_info=True)
    def load(self):
        if os.path.exists(self.save_path):
            try:
                with open(self.save_path,'rb') as f:
                    loaded_list = pickle.load(f)
                    self.buffer=collections.deque(loaded_list, maxlen=self.buffer_size)
                    self.num_games_added=len(self.buffer)
                self.logger.info(f"Replay buffer loaded from {self.save_path}. Size: {len(self.buffer)} games.")
            except Exception as e: self.logger.error(f"Error loading replay buffer: {e}. Starting empty.", exc_info=True); self.buffer=collections.deque(maxlen=self.buffer_size)
        else: self.logger.info(f"No replay buffer file found at {self.save_path}. Starting with an empty buffer.")
muzero_logger.info("--- Step 10: ReplayBuffer Class Defined ---")

# ==============================================================================
# STEP 11: LOSS FUNCTIONS
# ==============================================================================
muzero_logger.info("\n--- Step 11: Defining Loss Functions ---")
mse_loss_fn = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE)
cross_entropy_loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE)
def calculate_muzero_losses(pred_vals, target_vals, pred_rewards, target_rewards, pred_policies, target_policies, nets_for_l2, cfg):
    v_loss, r_loss, p_loss = tf.constant(0.0), tf.constant(0.0), tf.constant(0.0)
    for k in range(cfg.NUM_UNROLL_STEPS + 1): v_loss += mse_loss_fn(target_vals[:,k,:], pred_vals[k])
    for k in range(cfg.NUM_UNROLL_STEPS): r_loss += mse_loss_fn(target_rewards[:,k,:], pred_rewards[k])
    for k in range(cfg.NUM_UNROLL_STEPS + 1): p_loss += cross_entropy_loss_fn(target_policies[:,k,:], pred_policies[k])
    l2_loss = tf.constant(0.0)
    if cfg.WEIGHT_DECAY > 0:
        for net_item in nets_for_l2:
            if hasattr(net_item, 'trainable_variables') and net_item.trainable:
                for var in net_item.trainable_variables:
                    if 'bias' not in var.name and 'normalization' not in var.name and 'gamma' not in var.name and 'beta' not in var.name : l2_loss += tf.nn.l2_loss(var)
    l2_loss *= cfg.WEIGHT_DECAY
    avg_v = v_loss / tf.cast(cfg.NUM_UNROLL_STEPS+1,tf.float32); avg_r = r_loss / tf.cast(tf.maximum(1,cfg.NUM_UNROLL_STEPS),tf.float32); avg_p = p_loss / tf.cast(cfg.NUM_UNROLL_STEPS+1,tf.float32)
    w_v, w_r, w_p = avg_v*cfg.VALUE_LOSS_WEIGHT, avg_r*cfg.REWARD_LOSS_WEIGHT, avg_p*cfg.POLICY_LOSS_WEIGHT
    total = w_v + w_r + w_p + l2_loss
    return total, {"total_loss":total, "value_loss":avg_v, "reward_loss":avg_r, "policy_loss":avg_p, "l2_reg_loss":l2_loss}
muzero_logger.info("--- Step 11: Loss Functions Defined ---")

# ==============================================================================
# STEP 12: NETWORK DEFINITION & MAIN TRAINING LOOP
# ==============================================================================
muzero_logger.info("\n--- Step 12: Defining Networks and Main Training Loop ---")

TF_PRINT_INTERVAL = tf.constant(10, dtype=tf.int64)

def get_muzero_agent_networks(config, num_original_features, num_portfolio_states, num_actions, logger):
    logger.info(f"Instantiating MuZero networks. Base NN: {config.FINAL_BEST_NN_PATH}, Transfer Layer: {config.TRANSFER_LAYER_NAME}")
    if not os.path.exists(config.FINAL_BEST_NN_PATH): raise FileNotFoundError(f"Base NN {config.FINAL_BEST_NN_PATH} not found.")

    loaded_nn = keras.models.load_model(config.FINAL_BEST_NN_PATH, compile=False)
    try:
        base_model_output = loaded_nn.get_layer(config.TRANSFER_LAYER_NAME).output
        base_model = keras.Model(inputs=loaded_nn.input, outputs=base_model_output, name="BaseRepresentation")
    except ValueError as e:
        logger.critical(f"Layer '{config.TRANSFER_LAYER_NAME}' not in loaded NN."); loaded_nn.summary(print_fn=logger.error); raise e

    if getattr(config, 'FREEZE_H_NET_MAIN_RUN', False):
        base_model.trainable = False
        logger.info(f"Main run: base_model (GA NN) explicitly set to trainable=False (due to FREEZE_H_NET_MAIN_RUN).")
    else:
        base_model.trainable = not config.FREEZE_TRANSFER_WEIGHTS_INITIALLY
        logger.info(f"Base representation model (GA NN) trainable: {base_model.trainable} (initial freeze: {config.FREEZE_TRANSFER_WEIGHTS_INITIALLY})")

    m_in = keras.layers.Input(shape=(num_original_features,), name='h_market_in')
    p_in = keras.layers.Input(shape=(num_portfolio_states,), name='h_portfolio_in')
    enc_m = base_model(m_in)
    concat_h_layer = keras.layers.Concatenate()([enc_m, p_in])
    s_out_h_layer = keras.layers.Dense(config.MUZERO_HIDDEN_STATE_SIZE, activation='relu', name='h_final_dense')(concat_h_layer)
    h_net = keras.Model(inputs=[m_in, p_in], outputs=s_out_h_layer, name="Representation_h")

    if getattr(config, 'FREEZE_H_NET_MAIN_RUN', False):
        h_net.trainable = False
        logger.info("Main run: Full h_net set to trainable=False (due to FREEZE_H_NET_MAIN_RUN).")

    # Log the .trainable status of base_model *after* it's potentially modified by h_net.trainable = False
    # If h_net.trainable is set to False, all its layers, including base_model if it's a sub-layer, become non-trainable through h_net.
    # However, base_model.trainable reflects its standalone state or how it was set before being part of h_net.
    # The effective trainability comes from the outermost model containing it.
    is_base_model_in_h_trainable = False
    try:
        if h_net.get_layer('BaseRepresentation'): # Check if base_model is indeed a layer of h_net
            is_base_model_in_h_trainable = h_net.get_layer('BaseRepresentation').trainable
    except ValueError: # Layer not found directly (e.g. nested differently)
        is_base_model_in_h_trainable = base_model.trainable # Fallback to base_model's own status

    logger.info(f"Final h_net.trainable state: {h_net.trainable}. Base_model's own .trainable: {base_model.trainable}. Effective base_model trainable (via h_net): {is_base_model_in_h_trainable}")


    s_g_in = keras.layers.Input(shape=(config.MUZERO_HIDDEN_STATE_SIZE,), name='g_s_in')
    a_g_in = keras.layers.Input(shape=(num_actions,), name='g_a_in')
    concat_g_layer = keras.layers.Concatenate(name='g_concat_s_a')([s_g_in, a_g_in])
    x_dyn_s = keras.layers.Dense(config.DYNAMICS_INTERNAL_SIZE, name='g_dynamics_hidden_dense_linear')(concat_g_layer)
    x_dyn_s = keras.layers.BatchNormalization(name='g_dynamics_bn')(x_dyn_s)
    g_h1_processed_s = keras.layers.Activation('relu', name='g_dynamics_relu')(x_dyn_s)
    s_g_out_layer = keras.layers.Dense(config.MUZERO_HIDDEN_STATE_SIZE, activation='relu', name='g_s_out')(g_h1_processed_s)
    x_dyn_r = keras.layers.Dense(config.DYNAMICS_INTERNAL_SIZE_REWARD, name='g_reward_hidden_dense_linear')(concat_g_layer)
    x_dyn_r = keras.layers.BatchNormalization(name='g_reward_bn')(x_dyn_r)
    g_h1_processed_r = keras.layers.Activation('relu', name='g_reward_relu')(x_dyn_r)
    r_g_logits_layer = keras.layers.Dense(1, name='g_r_logits')(g_h1_processed_r)
    r_g_out_layer = keras.layers.Activation('tanh', name='g_r_out')(r_g_logits_layer)
    g_net = keras.Model(inputs=[s_g_in, a_g_in], outputs=[s_g_out_layer, r_g_out_layer, r_g_logits_layer], name="Dynamics_g_BN")

    s_f_in = keras.layers.Input(shape=(config.MUZERO_HIDDEN_STATE_SIZE,), name='f_s_in')

    policy_arch_name = getattr(config, 'POLICY_HEAD_ARCHITECTURE', 'original')

    p_f_out_layer = None # Initialize to ensure it's defined

    if policy_arch_name == 'custom_v2':
        settings = getattr(config, 'POLICY_HEAD_V2_SETTINGS', None)
        if not settings:
            logger.error("POLICY_HEAD_ARCHITECTURE is 'custom_v2' but POLICY_HEAD_V2_SETTINGS is missing. Reverting to original.")
            policy_arch_name = 'original' # Fallback
        else:
            logger.info(f"Using CUSTOM V2 policy head with settings: {settings}")
            x_p = keras.layers.Dense(settings['dense1_units'], name='f_policy_v2_dense1')(s_f_in)
            x_p = keras.layers.BatchNormalization(name='f_policy_v2_bn1')(x_p)
            x_p = keras.layers.Activation('relu', name='f_policy_v2_relu1')(x_p)
            x_p = keras.layers.GaussianNoise(settings['noise1_stddev'], name='f_policy_v2_noise1')(x_p)
            x_p = keras.layers.Dense(settings['dense2_units'], name='f_policy_v2_dense2')(x_p)
            x_p = keras.layers.BatchNormalization(name='f_policy_v2_bn2')(x_p)
            x_p = keras.layers.Activation('relu', name='f_policy_v2_relu2')(x_p)
            x_p = keras.layers.GaussianNoise(settings['noise2_stddev'], name='f_policy_v2_noise2')(x_p)
            x_p = keras.layers.Dropout(settings['dropout_rate'], name='f_policy_v2_dropout')(x_p)
            p_f_out_layer = keras.layers.Dense(num_actions, activation='linear', name='f_p_logits')(x_p)

    if policy_arch_name == 'original': # Handles default or explicit 'original' or fallback
        logger.info("Using ORIGINAL policy head design.")
        # Define x_p for the original policy head as well if it wasn't defined by custom_v2
        x_p_orig = keras.layers.Dense(config.PREDICTION_INTERNAL_SIZE_POLICY, name='f_policy_orig_dense_linear')(s_f_in)
        x_p_orig = keras.layers.BatchNormalization(name='f_policy_orig_bn')(x_p_orig)
        x_p_orig = keras.layers.Activation('relu', name='f_policy_orig_relu')(x_p_orig)
        p_f_out_layer = keras.layers.Dense(num_actions, name='f_p_logits')(x_p_orig)


    x_pred_v = keras.layers.Dense(config.PREDICTION_INTERNAL_SIZE_VALUE, name='f_value_hidden_dense_linear')(s_f_in)
    x_pred_v = keras.layers.BatchNormalization(name='f_value_bn')(x_pred_v)
    f_v_h1_processed = keras.layers.Activation('relu', name='f_value_relu')(x_pred_v)
    v_f_logits_layer = keras.layers.Dense(1, name='f_v_logits')(f_v_h1_processed)
    v_f_out_layer = keras.layers.Activation('tanh', name='f_v_out')(v_f_logits_layer)

    f_net = keras.Model(inputs=s_f_in, outputs=[p_f_out_layer, v_f_out_layer, v_f_logits_layer], name="Prediction_f_PolicyUpdated")
    logger.info("MuZero networks created.")
    return h_net, g_net, f_net, base_model


def train_muzero_agent(config, logger):
    original_log_level = logger.level
    script_version = getattr(config, 'SCRIPT_VERSION', 'SpecificArch_Run_V2_SoftmaxDebug')
    logger.info(f"===== STARTING MUZERO AGENT TRAINING (Version: {script_version}) =====")
    np.random.seed(config.SEED); tf.random.set_seed(config.SEED); random.seed(config.SEED)

    logger.info("Initializing components...")
    environment = FinancialMuZeroEnv(
        config_mz_instance=config, logger_instance=logger,
        trained_feature_names_list=TRAINED_FEATURE_NAMES_GLOBAL,
        global_ga_mask_arr=feature_mask_global,
        start_index=0,
        episode_length=getattr(config, 'INITIAL_EPISODE_LENGTH', 252)
    )
    h_net_train, g_net_train, f_net_train, base_repr_train = get_muzero_agent_networks(
        config, ORIGINAL_MODEL_FEATURE_COUNT, config.PORTFOLIO_STATE_SIZE, config.NUM_ACTIONS, logger
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=config.LEARNING_RATE,
        clipnorm=getattr(config, 'GRADIENT_CLIP_NORM', 1.0)
    )
    replay_buffer = ReplayBuffer(config, logger)

    start_step = 1
    latest_step_file_path = os.path.join(config.MUZERO_MODELS_DIR, getattr(config, 'LATEST_CHECKPOINT_FILENAME', "latest_checkpoint_step.txt"))
    load_checkpoint_explicitly_requested_step = getattr(config, 'LOAD_CHECKPOINT_FROM_STEP', 0)
    always_try_resume_flag = getattr(config, 'ALWAYS_ATTEMPT_RESUME', True)
    determined_load_step = 0
    if load_checkpoint_explicitly_requested_step > 0:
        determined_load_step = load_checkpoint_explicitly_requested_step
    elif always_try_resume_flag and load_checkpoint_explicitly_requested_step != 0:
        if os.path.exists(latest_step_file_path):
            try:
                with open(latest_step_file_path, 'r') as f_s_l: latest_saved_step = int(f_s_l.read().strip())
                determined_load_step = latest_saved_step
                logger.info(f"Found latest checkpoint step file: {latest_step_file_path}, indicates step: {latest_saved_step}.")
            except Exception as e_l: logger.warning(f"Could not read latest checkpoint step from {latest_step_file_path}: {e_l}.")
        else: logger.info(f"No latest checkpoint file found at {latest_step_file_path} (when trying to resume).")
        if load_checkpoint_explicitly_requested_step == -1 and determined_load_step == 0 :
            logger.info("LOAD_CHECKPOINT_FROM_STEP was -1 (latest) but no valid latest step found to load.")

    if determined_load_step > 0:
        logger.info(f"Attempting to load checkpoint from determined step {determined_load_step}...")
        try:
            checkpoint_base_path = getattr(config, 'MUZERO_CHECKPOINT_FORMAT', os.path.join(config.MUZERO_MODELS_DIR, "ckpt_S{step}"))
            h_path = checkpoint_base_path.format(step=determined_load_step) + "_h.weights.h5"
            g_path = checkpoint_base_path.format(step=determined_load_step) + "_g.weights.h5"
            f_path = checkpoint_base_path.format(step=determined_load_step) + "_f.weights.h5"

            if os.path.exists(h_path) and os.path.exists(g_path) and os.path.exists(f_path):
                h_net_train.load_weights(h_path); g_net_train.load_weights(g_path); f_net_train.load_weights(f_path)
                logger.info(f"Successfully loaded model weights from step {determined_load_step}.")
                replay_buffer.load()
                start_step = determined_load_step + 1
                logger.info(f"Resuming training from step {start_step}.")

                if not getattr(config, 'FREEZE_H_NET_MAIN_RUN', False) and \
                   config.FREEZE_TRANSFER_WEIGHTS_INITIALLY and \
                   start_step > config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP and \
                   not base_repr_train.trainable:
                    base_repr_train.trainable = True
                    logger.info(f"Resuming: Unfroze base_repr_train. h_net.trainable is {h_net_train.trainable}")
                elif not getattr(config, 'FREEZE_H_NET_MAIN_RUN', False) and \
                     config.FREEZE_TRANSFER_WEIGHTS_INITIALLY and \
                     base_repr_train.trainable and \
                     start_step <= config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP:
                    base_repr_train.trainable = False
                    logger.info(f"Resuming: Kept base_repr_train frozen. h_net.trainable is {h_net_train.trainable}")

            else:
                logger.warning(f"Checkpoint weight files for step {determined_load_step} not fully found. Paths checked:\nH: {h_path}\nG: {g_path}\nF: {f_path}\nStarting training from scratch.")
                start_step = 1
        except Exception as e_load:
            logger.error(f"Error loading checkpoint from step {determined_load_step}: {e_load}. Starting fresh.", exc_info=True)
            start_step = 1
    else:
        logger.info("Starting training from scratch (no valid checkpoint specified or found to load).")


    @tf.function
    def train_step_tf(batch_tf, current_train_step_tf):
        with tf.GradientTape() as tape:
            vars_to_train = []
            if h_net_train.trainable:
                for layer_h in h_net_train.layers:
                    if layer_h.name == 'BaseRepresentation':
                        if base_repr_train.trainable:
                             vars_to_train.extend(base_repr_train.trainable_variables)
                    elif layer_h.trainable:
                        vars_to_train.extend(layer_h.trainable_variables)

            vars_to_train.extend(g_net_train.trainable_variables)
            vars_to_train.extend(f_net_train.trainable_variables)

            if not vars_to_train:
                tf.print("CRITICAL WARNING: No trainable variables found in train_step_tf!", output_stream=sys.stderr)
                dummy_loss_val = tf.constant(0.0, dtype=tf.float32)
                dummy_metrics = {"total_loss":dummy_loss_val, "value_loss":dummy_loss_val,
                                 "reward_loss":dummy_loss_val, "policy_loss":dummy_loss_val, "l2_reg_loss":dummy_loss_val}
                return dummy_loss_val, dummy_metrics

            s_k = h_net_train([batch_tf['initial_market_features'], batch_tf['initial_portfolio_states']], training=h_net_train.trainable)

            preds_p, preds_v_tanh, preds_r_tanh = [], [], []
            preds_v_logits_debug, preds_r_logits_debug = [], [] # For debug prints only

            policy_s0_logits, value_s0_tanh, value_s0_logits_for_debug = f_net_train(s_k, training=True)
            preds_p.append(policy_s0_logits)
            preds_v_tanh.append(value_s0_tanh)
            preds_v_logits_debug.append(value_s0_logits_for_debug) # Store for debug

            # ---- MODIFIED TF_DEBUG BLOCK for s_0 ----
            if tf.equal(tf.math.floormod(current_train_step_tf, TF_PRINT_INTERVAL), tf.cast(0, dtype=TF_PRINT_INTERVAL.dtype)):
                tf.print("--- S", current_train_step_tf, "TF_DEBUG (Initial s_0, Batch[0]) ---", output_stream=sys.stdout)
                tf.print(" Target Value (scaled):", batch_tf['target_values'][0,0,0], output_stream=sys.stdout)
                tf.print(" Pred Value (tanh, s_0):", preds_v_tanh[0][0,0], output_stream=sys.stdout)
                tf.print(" Pred Policy Logits (s_0):", preds_p[0][0,:], summarize=-1, output_stream=sys.stdout)
                pred_policy_probs_s0 = tf.nn.softmax(preds_p[0][0,:])
                tf.print(" Pred Policy Probs (s_0):", pred_policy_probs_s0, summarize=-1, output_stream=sys.stdout)
            # ---- END OF MODIFIED TF_DEBUG BLOCK for s_0 ----


            for k_step in range(config.NUM_UNROLL_STEPS):
                actions_g = tf.one_hot(batch_tf['action_history'][:, k_step], depth=config.NUM_ACTIONS)
                s_k_plus_1, rew_k_tanh, rew_k_logits_for_debug = g_net_train([s_k, actions_g], training=True)
                preds_r_tanh.append(rew_k_tanh)
                preds_r_logits_debug.append(rew_k_logits_for_debug) # Store for debug
                s_k = s_k_plus_1
                policy_sk_plus_1_logits, value_sk_plus_1_tanh, value_sk_plus_1_logits_for_debug = f_net_train(s_k, training=True)
                preds_p.append(policy_sk_plus_1_logits)
                preds_v_tanh.append(value_sk_plus_1_tanh)
                preds_v_logits_debug.append(value_sk_plus_1_logits_for_debug) # Store for debug

                # ---- RESTORED/ADDED TF_DEBUG BLOCK for s_{k+1} (first unroll step only) ----
                if tf.logical_and(
                    tf.equal(tf.math.floormod(current_train_step_tf, TF_PRINT_INTERVAL), tf.cast(0, dtype=TF_PRINT_INTERVAL.dtype)),
                    tf.equal(k_step, 0) # Only for the first unroll step (k_step=0)
                ):
                    tf.print("--- S", current_train_step_tf, "TF_DEBUG (Unroll k_step=", k_step, ", Batch[0]) ---", output_stream=sys.stdout)
                    tf.print("  Target Reward (scaled, r_k):", batch_tf['target_rewards'][0,k_step,0], output_stream=sys.stdout)
                    tf.print("  Pred Reward (tanh, r_k):", preds_r_tanh[k_step][0,0], output_stream=sys.stdout) # Use k_step as index for preds_r_tanh
                    tf.print("  Target Value (scaled, v_{k+1}):", batch_tf['target_values'][0,k_step+1,0], output_stream=sys.stdout)
                    tf.print("  Pred Value (tanh, s_{k+1}):", preds_v_tanh[k_step+1][0,0], output_stream=sys.stdout) # Use k_step+1 as index
                    tf.print("  Pred Policy Logits (s_{k+1}):", preds_p[k_step+1][0,:], summarize=-1, output_stream=sys.stdout) # Use k_step+1
                    pred_policy_probs_sk_plus_1 = tf.nn.softmax(preds_p[k_step+1][0,:])
                    tf.print("  Pred Policy Probs (s_{k+1}):", pred_policy_probs_sk_plus_1, summarize=-1, output_stream=sys.stdout)
                # ---- END OF RESTORED TF_DEBUG BLOCK for s_{k+1} ----

            nets_for_l2_calculation = []
            if h_net_train.trainable: nets_for_l2_calculation.append(h_net_train)
            nets_for_l2_calculation.append(g_net_train)
            nets_for_l2_calculation.append(f_net_train)

            loss_total, loss_comps = calculate_muzero_losses(
                preds_v_tanh, batch_tf['target_values'],
                preds_r_tanh, batch_tf['target_rewards'],
                preds_p, batch_tf['target_policies'],
                nets_for_l2=nets_for_l2_calculation,
                cfg=config
            )

            grads = tape.gradient(loss_total, vars_to_train)
            optimizer.apply_gradients(zip(grads, vars_to_train))
            return loss_total, loss_comps

    for step in range(start_step, config.MAX_TRAINING_STEPS + 1):
        loop_start_time = time.time()

        if not getattr(config, 'FREEZE_H_NET_MAIN_RUN', False) and \
           config.FREEZE_TRANSFER_WEIGHTS_INITIALLY and \
           step >= config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP and \
           not base_repr_train.trainable:
            base_repr_train.trainable = True
            logger.info(f"Step {step}: Unfroze base_repr_train. h_net.trainable is {h_net_train.trainable}")

        intended_episode_len = getattr(config, 'DEFAULT_EPISODE_LENGTH_PLAY', 252)
        safe_intended_ep_len = max(1, intended_episode_len)

        max_possible_start_index = environment.total_data_steps - safe_intended_ep_len
        if max_possible_start_index < 0: max_possible_start_index = 0

        current_start_idx_play = np.random.randint(0, max_possible_start_index + 1) if max_possible_start_index >= 0 else 0

        obs_loop = environment.reset(start_index=current_start_idx_play, episode_length=intended_episode_len)

        current_episode_run_actual_len = environment.episode_length

        game_data = {'observations_list':[obs_loop],'actions_list':[],'rewards_list':[],
                     'mcts_policy_targets_list':[],'value_targets_list':[],
                     'raw_mcts_values_list': [], 'raw_rewards_list': [],
                     'done_list':[],'total_episode_reward_raw':0.0}
        done_loop, game_steps_count = False, 0

        current_temp = config.INITIAL_TEMPERATURE if step < getattr(config, 'ANNEAL_TEMPERATURE_AFTER_N_MOVES', config.MAX_TRAINING_STEPS // 2) else config.FINAL_TEMPERATURE

        while not done_loop and game_steps_count < current_episode_run_actual_len :
            mcts_policy_target, mcts_val_raw, _ = run_mcts(
                obs_loop, h_net_train,g_net_train,f_net_train,config,config.NUM_SIMULATIONS,config.NUM_ACTIONS)

            mcts_val_scaled_target = np.tanh(mcts_val_raw / config.VALUE_TARGET_SCALING_DIVISOR)

            action_probs_for_sampling = mcts_policy_target
            if np.sum(action_probs_for_sampling) > 1e-6 and current_temp > 1e-3:
                action_probs_temp = np.power(action_probs_for_sampling + 1e-9, 1.0 / current_temp)
                action_probs_temp_sum = np.sum(action_probs_temp)
                if action_probs_temp_sum > 1e-6 : action_probs_temp /= action_probs_temp_sum
                else: action_probs_temp = np.ones_like(action_probs_temp) / len(action_probs_temp)
                if np.isnan(action_probs_temp).any(): action = np.argmax(action_probs_for_sampling)
                else: action = np.random.choice(config.NUM_ACTIONS, p=action_probs_temp)
            else: action = np.argmax(action_probs_for_sampling)
            logger.debug(f"  S{step} G{len(replay_buffer)+1}-P{game_steps_count+1}: Pol={np.round(mcts_policy_target,2)}, RawVal={mcts_val_raw:.3f}, ScaledValTgt={mcts_val_scaled_target:.3f}, Act={action} (T={current_temp:.2f})")

            next_obs, raw_reward_from_env, done_loop, info = environment.step(action)

            scaled_reward_target_for_buffer = np.tanh(raw_reward_from_env / config.REWARD_TARGET_SCALING_DIVISOR)
            logger.debug(f"    Env Step: RawRew={raw_reward_from_env:.4f}, ScaledRewTgt={scaled_reward_target_for_buffer:.4f} Done={done_loop}, TgtPos={info.get('tgt_pos')}")

            game_data['actions_list'].append(action);
            game_data['rewards_list'].append(scaled_reward_target_for_buffer)
            game_data['raw_rewards_list'].append(raw_reward_from_env);
            game_data['mcts_policy_targets_list'].append(mcts_policy_target);
            game_data['value_targets_list'].append(mcts_val_scaled_target);
            game_data['raw_mcts_values_list'].append(mcts_val_raw);
            game_data['done_list'].append(done_loop);
            game_data['total_episode_reward_raw'] += raw_reward_from_env
            obs_loop = next_obs
            if not done_loop: game_data['observations_list'].append(obs_loop)
            game_steps_count += 1

        if not done_loop and len(game_data['observations_list']) > len(game_data['mcts_policy_targets_list']):
            last_obs = game_data['observations_list'][-1]
            last_s = h_net_train.predict_on_batch([np.array(o,dtype=np.float32)[np.newaxis,...] for o in last_obs])
            last_policy_logits, last_value_output_tanh, _ = f_net_train.predict_on_batch(last_s)
            game_data['mcts_policy_targets_list'].append(tf.nn.softmax(last_policy_logits[0]).numpy());
            game_data['value_targets_list'].append(last_value_output_tanh[0,0])
            game_data['raw_mcts_values_list'].append(last_value_output_tanh[0,0])

        replay_buffer.add_game(game_data)
        logger.debug(f"S{step} GameStore: Game {len(replay_buffer)} added. EpRawRew={game_data['total_episode_reward_raw']:.2f}, Steps={game_steps_count}")

        if step % 50 == 0:
            logger.info(f"--- S{step} Post-Game Monitoring (Targets from Last Episode) ---")
            if game_data.get('raw_mcts_values_list') and game_data.get('value_targets_list'):
                raw_vals_to_log = game_data['raw_mcts_values_list']
                scaled_val_targets_to_log = game_data['value_targets_list']
                num_value_samples_to_display = min(5, len(raw_vals_to_log))
                if num_value_samples_to_display > 0:
                    logger.info(f"  Last {num_value_samples_to_display} Raw MCTS Values from episode: {['{:.3f}'.format(x) for x in raw_vals_to_log[-num_value_samples_to_display:]]}")
                    logger.info(f"  Last {num_value_samples_to_display} Scaled Value Targets (for loss): {['{:.3f}'.format(x) for x in scaled_val_targets_to_log[-num_value_samples_to_display:]]}")
            if game_data.get('raw_rewards_list') and game_data.get('rewards_list'):
                raw_rewards_to_log = game_data['raw_rewards_list']
                scaled_reward_targets_to_log = game_data['rewards_list']
                num_reward_samples_to_display = min(5, len(raw_rewards_to_log))
                if num_reward_samples_to_display > 0:
                    logger.info(f"  Last {num_reward_samples_to_display} Raw Env Rewards from episode: {['{:.3f}'.format(x) for x in raw_rewards_to_log[-num_reward_samples_to_display:]]}")
                    logger.info(f"  Last {num_reward_samples_to_display} Scaled Reward Targets (for loss): {['{:.3f}'.format(x) for x in scaled_reward_targets_to_log[-num_reward_samples_to_display:]]}")
            logger.info(f"--- End Post-Game Monitoring S{step} ---")


        min_buffer_for_training = getattr(config, 'MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING', config.BATCH_SIZE * 2)
        if len(replay_buffer) >= min_buffer_for_training:
            raw_b = replay_buffer.sample_batch(config.BATCH_SIZE)
            if raw_b:
                tf_b = {'initial_market_features': tf.constant(np.array([s['initial_observation'][0] for s in raw_b],dtype=np.float32)),
                        'initial_portfolio_states': tf.constant(np.array([s['initial_observation'][1] for s in raw_b],dtype=np.float32)),
                        'action_history': tf.constant(np.array([s['action_history'] for s in raw_b],dtype=np.int32)),
                        'target_rewards': tf.constant(np.array([s['target_rewards'] for s in raw_b],dtype=np.float32)[...,np.newaxis]),
                        'target_policies': tf.constant(np.array([s['target_policies'] for s in raw_b],dtype=np.float32)),
                        'target_values': tf.constant(np.array([s['target_values'] for s in raw_b],dtype=np.float32)[...,np.newaxis])}
                loss_val, metrics = train_step_tf(tf_b, tf.cast(step, dtype=tf.int64))
                if step % 10 == 0:
                    log_msg = f"S{step} Train: Loss={loss_val.numpy():.4f}, " + \
                              f"P Loss={metrics['policy_loss'].numpy():.3f}, V Loss={metrics['value_loss'].numpy():.3f}, R Loss={metrics['reward_loss'].numpy():.3f}" + \
                              f", L2={metrics['l2_reg_loss'].numpy():.5f}"
                    logger.info(log_msg)
            else: logger.debug(f"S{step}: Training skipped, batch from buffer was empty.")
        elif step % 10 == 0 : logger.info(f"S{step}: Buffer fill ({len(replay_buffer)}/{min_buffer_for_training} games)...")

        if step % config.CHECKPOINT_INTERVAL == 0 and step > 0:
            logger.info(f"S{step}: Attempting to save checkpoint (Interval: {config.CHECKPOINT_INTERVAL})...")
            os.makedirs(config.MUZERO_MODELS_DIR,exist_ok=True)
            checkpoint_base_path = getattr(config, 'MUZERO_CHECKPOINT_FORMAT', os.path.join(config.MUZERO_MODELS_DIR, "ckpt_S{step}"))
            pref = checkpoint_base_path.format(step=step)

            h_path_save = pref+"_h.weights.h5"; g_path_save = pref+"_g.weights.h5"; f_path_save = pref+"_f.weights.h5"
            logger.info(f"  Target H-Net: {h_path_save}, G-Net: {g_path_save}, F-Net: {f_path_save}")
            logger.info(f"  Target Replay Buffer: {replay_buffer.save_path}, Latest Step File: {latest_step_file_path}")
            try:
                h_net_train.save_weights(h_path_save); logger.info(f"    SUCCESS: Saved H-Net for S{step}")
                g_net_train.save_weights(g_path_save); logger.info(f"    SUCCESS: Saved G-Net for S{step}")
                f_net_train.save_weights(f_path_save); logger.info(f"    SUCCESS: Saved F-Net for S{step}")
                replay_buffer.save();
                with open(latest_step_file_path, 'w') as f_sw: f_sw.write(str(step))
                logger.info(f"    SUCCESS: Updated latest step file '{latest_step_file_path}' to {step}.")
                logger.info(f"----> CHECKPOINT COMPLETELY SAVED for S{step} <----")
            except Exception as e_ckpt: logger.error(f"Error saving complete checkpoint at step {step}: {e_ckpt}", exc_info=True)

        if step % 100 == 0:
            gpu_mem_str = "N/A"
            try:
                if tf.config.list_physical_devices('GPU'):
                    mem_info = tf.config.experimental.get_memory_info('GPU:0')
                    gpu_mem_str = f"{mem_info['current'] / 1024**2:.1f}MB / {mem_info['peak'] / 1024**2:.1f}MB (peak)"
                else: gpu_mem_str = "No GPU found by TF"
            except Exception as e_gpu_log: gpu_mem_str = f"Error getting GPU mem: {str(e_gpu_log)}"
            loop_duration = time.time() - loop_start_time
            logger.info(f"S{step} LoopTime: {loop_duration:.2f}s. GPU Mem: {gpu_mem_str}")

    logger.info("===== MUZERO AGENT TRAINING FINISHED =====")
    logger.setLevel(original_log_level)

# --- Execution Guard ---
if __name__ == "__main__":
    missing_comp = False
    essential_globals_for_main_execution = ['config_mz', 'muzero_logger',
                                            'TRAINED_FEATURE_NAMES_GLOBAL', 'feature_mask_global',
                                            'ORIGINAL_MODEL_FEATURE_COUNT', 'load_ga_feature_mask',
                                            'FinancialMuZeroEnv', 'ReplayBuffer', 'MCTSNode', 'run_mcts',
                                            'calculate_muzero_losses', 'get_muzero_agent_networks' ]
    for var_name_item_main in essential_globals_for_main_execution:
        if var_name_item_main not in globals() or globals()[var_name_item_main] is None:
            print(f"CRITICAL: Prerequisite '{var_name_item_main}' missing for training loop.")
            missing_comp = True

    if not missing_comp:
        if not hasattr(config_mz, 'FREEZE_H_NET_MAIN_RUN'):
            muzero_logger.warning("config_mz.FREEZE_H_NET_MAIN_RUN not set. Defaulting to True for this run.")
            config_mz.FREEZE_H_NET_MAIN_RUN = True
        if not hasattr(config_mz, 'POLICY_HEAD_ARCHITECTURE'):
            muzero_logger.warning("config_mz.POLICY_HEAD_ARCHITECTURE not set. Defaulting to 'custom_v2'.")
            config_mz.POLICY_HEAD_ARCHITECTURE = 'custom_v2'
        if config_mz.POLICY_HEAD_ARCHITECTURE == 'custom_v2' and not hasattr(config_mz, 'POLICY_HEAD_V2_SETTINGS'):
            muzero_logger.warning("config_mz.POLICY_HEAD_V2_SETTINGS not set for 'custom_v2' architecture. Using defaults.")
            config_mz.POLICY_HEAD_V2_SETTINGS = {
                'dense1_units': 12, 'noise1_stddev': 0.2,
                'dense2_units': 1024, 'noise2_stddev': 0.2, 'dropout_rate': 0.2
            }

        script_version_tag = getattr(config_mz, 'SCRIPT_VERSION', 'SpecificArch_Run_V2_SoftmaxDebug_Main') # Updated version tag
        muzero_logger.info(f"All prerequisites met. Starting MuZero training agent ({script_version_tag})...")

        if not hasattr(config_mz, 'ALWAYS_ATTEMPT_RESUME'): config_mz.ALWAYS_ATTEMPT_RESUME = True
        if not hasattr(config_mz, 'LOAD_CHECKPOINT_FROM_STEP'): config_mz.LOAD_CHECKPOINT_FROM_STEP = -1
        if not hasattr(config_mz, 'LATEST_CHECKPOINT_FILENAME'): config_mz.LATEST_CHECKPOINT_FILENAME = "latest_checkpoint_step.txt"
        if not hasattr(config_mz, 'MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING'): config_mz.MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING = config_mz.BATCH_SIZE * 10
        if not hasattr(config_mz, 'INITIAL_EPISODE_LENGTH'): config_mz.INITIAL_EPISODE_LENGTH = 252
        if not hasattr(config_mz, 'DEFAULT_EPISODE_LENGTH_PLAY'): config_mz.DEFAULT_EPISODE_LENGTH_PLAY = 252
        if not hasattr(config_mz, 'GRADIENT_CLIP_NORM'): config_mz.GRADIENT_CLIP_NORM = 1.0

        config_mz.SCRIPT_VERSION = script_version_tag

        gpu_devices = tf.config.list_physical_devices('GPU')
        if not gpu_devices:
            muzero_logger.warning("NO GPU DETECTED BY TENSORFLOW. Training will be very slow.")
        else:
            muzero_logger.info(f"GPU detected: {gpu_devices}")
            try:
                for gpu in gpu_devices:
                    tf.config.experimental.set_memory_growth(gpu, True)
                muzero_logger.info("Attempted to set TensorFlow GPU memory growth.")
            except RuntimeError as e_mem_growth:
                muzero_logger.warning(f"Could not set memory growth for GPU (may be already initialized): {e_mem_growth}")

        try:
            train_muzero_agent(config_mz, muzero_logger)
        except Exception as e_main:
            muzero_logger.critical(f"Unhandled exception in __main__ calling train_muzero_agent: {e_main}", exc_info=True)
    else:
        print("Halting due to missing prerequisites for __main__ execution block.")

2025-06-09 10:17:52 - INFO - [<ipython-input-2-833242d5f5d8>:54] - <ipython-input-2-833242d5f5d8> - --- MONOLITH PRELUDE: Loading Training Features List and GA Mask ---
2025-06-09 10:17:53 - INFO - [<ipython-input-2-833242d5f5d8>:61] - <ipython-input-2-833242d5f5d8> - Loaded 1272 feature names from /content/drive/MyDrive/Afterford/MU/models/best_model/training_features.json.
2025-06-09 10:17:53 - INFO - [<ipython-input-1-59b15789585a>:409] - <ipython-input-1-59b15789585a> - Successfully loaded feature mask from /content/drive/MyDrive/Afterford/MU/models/best_model/best_model_mask.txt. Original shape: (1272,), Sum (selected features): 620
2025-06-09 10:17:53 - INFO - [<ipython-input-2-833242d5f5d8>:68] - <ipython-input-2-833242d5f5d8> - GA Mask loaded. Length: 1272. Effective features: 620
2025-06-09 10:17:53 - INFO - [<ipython-input-2-833242d5f5d8>:72] - <ipython-input-2-833242d5f5d8> - --- MONOLITH PRELUDE Complete ---
2025-06-09 10:17:53 - INFO - [<ipython-input-2-833242d5f5d8>:78] -

In [ ]:
# This entire block should be run as a SINGLE CELL
# @title 12 Original # AFTER Section 1 (defining config_mz, muzero_logger, load_ga_feature_mask) is complete.

import tensorflow as tf
from tensorflow import keras
import numpy as np
import time
import os
import math
import collections
import json
import logging
import sys
import random
import pandas as pd
import pickle
import copy

# --- Basic Logger & Config Check ---
if 'muzero_logger' not in globals() or muzero_logger is None:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(module)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    muzero_logger = logging.getLogger("MuZeroProject_Monolith_BN_Logits") #
    muzero_logger.warning("Re-initialized basic logger for Monolithic Block (BN + Logits).")

essential_globals_for_script = ['config_mz', 'muzero_logger', 'load_ga_feature_mask'] #
for var_name in essential_globals_for_script:
    if var_name not in globals() or globals()[var_name] is None:
        muzero_logger.critical(f"Essential global component '{var_name}' from Section 1 is missing. Halting.") #
        raise NameError(f"Missing Section 1 component '{var_name}'.")

# ==============================================================================
# PRELUDE: Load Feature Lists and Masks (Assumed correct from original script)
# ==============================================================================
muzero_logger.info("--- MONOLITH PRELUDE: Loading Training Features List and GA Mask ---") #
TRAINED_FEATURE_NAMES_GLOBAL = None #
ORIGINAL_MODEL_FEATURE_COUNT = 0 #
if not hasattr(config_mz, 'FEATURE_LIST_PATH') or not os.path.exists(config_mz.FEATURE_LIST_PATH):
    raise FileNotFoundError(f"Training features file path missing/invalid in config: {getattr(config_mz, 'FEATURE_LIST_PATH', 'N/A')}") #
try:
    with open(config_mz.FEATURE_LIST_PATH, 'r') as f: TRAINED_FEATURE_NAMES_GLOBAL = json.load(f) #
    muzero_logger.info(f"Loaded {len(TRAINED_FEATURE_NAMES_GLOBAL)} feature names from {config_mz.FEATURE_LIST_PATH}.") #
    ORIGINAL_MODEL_FEATURE_COUNT = len(TRAINED_FEATURE_NAMES_GLOBAL) #
except Exception as e: muzero_logger.critical(f"Error loading training features: {e}"); raise #
feature_mask_global = load_ga_feature_mask(config_mz.GA_BEST_MASK_PATH, muzero_logger) #
if feature_mask_global is None: raise FileNotFoundError(f"GA Feature Mask not found: {config_mz.GA_BEST_MASK_PATH}") #
if len(feature_mask_global) != ORIGINAL_MODEL_FEATURE_COUNT: raise ValueError("GA mask length mismatch with training features.") #
NUM_EFFECTIVE_MASKED_FEATURES = np.sum(feature_mask_global) #
muzero_logger.info(f"GA Mask loaded. Length: {len(feature_mask_global)}. Effective features: {NUM_EFFECTIVE_MASKED_FEATURES}") #
NUM_PORTFOLIO_STATES = config_mz.PORTFOLIO_STATE_SIZE #
NUM_ACTIONS = config_mz.NUM_ACTIONS #
MUZERO_HIDDEN_STATE_SIZE = config_mz.MUZERO_HIDDEN_STATE_SIZE #
muzero_logger.info("--- MONOLITH PRELUDE Complete ---") #

essential_globals_after_prelude = ['TRAINED_FEATURE_NAMES_GLOBAL', 'feature_mask_global', 'ORIGINAL_MODEL_FEATURE_COUNT'] #
for var_name in essential_globals_after_prelude:
    if var_name not in globals() or globals()[var_name] is None:
        raise NameError(f"Essential Prelude component '{var_name}' missing.")
muzero_logger.info("Prelude global variable checks passed.") #

# ==============================================================================
# STEP 8: FinancialMuZeroEnv (Assumed to be correct as per original script)
# ==============================================================================
muzero_logger.info("\n--- Defining Step 8: FinancialMuZeroEnv ---") #
class FinancialMuZeroEnv: #
    def __init__(self, config_mz_instance, logger_instance,
                 trained_feature_names_list, global_ga_mask_arr,
                 start_index=0, episode_length=None): #
        self.config = config_mz_instance; self.logger = logger_instance #
        self.trained_feature_names = trained_feature_names_list #
        self.global_ga_mask = global_ga_mask_arr #
        self.original_model_feature_count = ORIGINAL_MODEL_FEATURE_COUNT #
        self.start_index = start_index; self.episode_length = episode_length #
        self.POSITION_SPXL, self.POSITION_SPXS, self.POSITION_CASH = 0, 1, 2 #
        self.portfolio_state_names = {self.POSITION_SPXL: "SPXL", self.POSITION_SPXS: "SPXS", self.POSITION_CASH: "CASH"} #
        self.num_portfolio_states = len(self.portfolio_state_names) #
        self.ACTION_TARGET_SPXL, self.ACTION_TARGET_SPXS, self.ACTION_TARGET_CASH = 0, 1, 2 #
        self.num_actions = self.config.NUM_ACTIONS #
        self.transaction_cost_bps = getattr(self.config, 'TRANSACTION_COST_BPS', 10) #
        self.leverage = getattr(self.config, 'LEVERAGE', 3.0) #
        self._load_and_prepare_data() #
        self.current_step_in_episode = 0; self.current_portfolio_position = self.POSITION_CASH #
        self.current_data_idx = self.start_index; self.done = False #
        self.episode_start_data_idx = self.start_index #
        self.logger.info(f"FinancialMuZeroEnv initialized. Usable steps: {self.total_data_steps}. Ep length: {self.episode_length if self.episode_length else 'End of Data'}.") #

    def _rename_etf_columns(self, df): #
        new_cols = {c: c.replace("SPXS_('Open', 'SPXS')", 'SPXS_Open').replace("SPXS_('Close', 'SPXS')", 'SPXS_Close')
                      .replace("SPXL_('Open', 'SPXL')", 'SPXL_Open').replace("SPXL_('Close', 'SPXL')", 'SPXL_Close')
                      .replace("SPXS_('High', 'SPXS')", 'SPXS_High').replace("SPXS_('Low', 'SPXS')", 'SPXS_Low')
                      .replace("SPXS_('Volume', 'SPXS')", 'SPXS_Volume')
                      .replace("SPXL_('High', 'SPXL')", 'SPXL_High').replace("SPXL_('Low', 'SPXL')", 'SPXL_Low')
                      .replace("SPXL_('Volume', 'SPXL')", 'SPXL_Volume')
                   for c in df.columns} #
        df.rename(columns=new_cols, inplace=True); return df #

    def _load_and_prepare_data(self): #
            date_col = self.config.existing_config_module.DATE_COLUMN #
            data_csv = pd.read_csv(self.config.MARKET_DATA_PATH, index_col=date_col, parse_dates=True) #
            etf_raw = pd.read_csv(self.config.ETF_DATA_PATH, index_col=date_col, parse_dates=True) #
            etf_proc = self._rename_etf_columns(etf_raw.copy()) #
            req_etf = ['SPXL_Close', 'SPXS_Close', 'SPXL_Open', 'SPXS_Open'] #
            if not all(c in etf_proc.columns for c in req_etf): raise ValueError(f"ETF data missing required columns. Found: {etf_proc.columns.tolist()}") #

            aligned_features = pd.DataFrame(index=data_csv.index, columns=self.trained_feature_names) #
            cols_to_fill = [c for c in self.trained_feature_names if c in data_csv.columns] #
            if cols_to_fill: aligned_features[cols_to_fill] = data_csv[cols_to_fill] #
            missing = set(self.trained_feature_names) - set(cols_to_fill) #
            if missing:
                self.logger.warning(f"{len(missing)} features expected by model were MISSING in data.csv and filled with 0s: {list(missing)[:5]}...") #
                aligned_features[list(missing)] = 0.0 #

            common_idx = aligned_features.index.intersection(etf_proc.index) #
            # Load all available aligned data first
            market_features_all_aligned = aligned_features.loc[common_idx].sort_index() #
            etf_prices_all_aligned = etf_proc.loc[common_idx].sort_index() #

            if market_features_all_aligned.empty: raise ValueError("Data empty after alignment.") #

            # --- Modification to hold out the last 252 days ---
            num_oos_days = 252
            if len(market_features_all_aligned) > num_oos_days:
                self.logger.info(f"Holding out the last {num_oos_days} days for out-of-sample testing.")
                # Training portion
                self.market_features_df_raw_aligned = market_features_all_aligned.iloc[:-num_oos_days]
                self.etf_prices_df = etf_prices_all_aligned.iloc[:-num_oos_days]

                # Optional: Store OOS data if this env instance needs to know about it for some reason
                # self.oos_market_features_df = market_features_all_aligned.iloc[-num_oos_days:]
                # self.oos_etf_prices_df = etf_prices_all_aligned.iloc[-num_oos_days:]
                self.logger.info(f"Training data will use {len(self.market_features_df_raw_aligned)} days.")
            else:
                self.logger.warning(f"Total available data ({len(market_features_all_aligned)} days) is not enough to hold out {num_oos_days} days. Using all available data for training.")
                self.market_features_df_raw_aligned = market_features_all_aligned
                self.etf_prices_df = etf_prices_all_aligned
            # --- End of modification ---

            if self.market_features_df_raw_aligned.empty: raise ValueError("Training data became empty after attempting to hold out OOS data.") # Add a check

            self.market_features_df = self.market_features_df_raw_aligned.copy() #
            if len(self.global_ga_mask) != self.market_features_df.shape[1]:
                raise ValueError(f"GA Mask length ({len(self.global_ga_mask)}) != Aligned Market Features cols ({self.market_features_df.shape[1]})") #
            self.market_features_df.loc[:, ~self.global_ga_mask] = 0.0 #

            self.total_data_steps = len(self.market_features_df) # This will now reflect the length of the training portion

            self.logger.info(f"Data prep done for training. Market features (masked): {self.market_features_df.shape}. Effective non-zero: {np.sum(self.global_ga_mask)}. Total usable steps for this env instance: {self.total_data_steps}") #

    def _get_observation(self): #
        market_obs = np.zeros(self.original_model_feature_count, dtype=np.float32) #
        if self.current_data_idx < self.total_data_steps:
            market_obs = self.market_features_df.iloc[self.current_data_idx].values.astype(np.float32) #
        portfolio_obs = np.zeros(self.num_portfolio_states, dtype=np.float32); portfolio_obs[self.current_portfolio_position] = 1.0 #
        return [market_obs, portfolio_obs] #

    def reset(self, start_index=None, episode_length=None): #
        self.current_step_in_episode = 0; self.done = False; self.current_portfolio_position = self.POSITION_CASH #
        self.episode_start_data_idx = start_index if start_index is not None else self.start_index #
        self.current_data_idx = self.episode_start_data_idx #
        self.episode_length = episode_length if episode_length is not None else (self.total_data_steps - self.current_data_idx) #
        if self.current_data_idx >= self.total_data_steps:
            self.logger.warning(f"Reset start index {self.current_data_idx} is out of bounds. Setting done.") #
            self.done = True; return self._get_observation() #
        self.logger.debug(f"Env reset. Start: {self.current_data_idx}. Pos: {self.portfolio_state_names[self.current_portfolio_position]}. Len: {self.episode_length}.") #
        return self._get_observation() #

    def step(self, action): #
        if self.done: return self._get_observation(), 0.0, True, {'status': 'Episode ended'} #
        if self.current_data_idx + 1 >= self.total_data_steps:
            self.logger.warning(f"Env step: End of data for ETF prices at index {self.current_data_idx + 1}.") #
            self.done = True; return self._get_observation(), 0.0, True, {'status': 'End of data for reward'} #

        next_day_prices = self.etf_prices_df.iloc[self.current_data_idx + 1] #
        spxl_open, spxl_close = next_day_prices.get('SPXL_Open', np.nan), next_day_prices.get('SPXL_Close', np.nan) #
        spxs_open, spxs_close = next_day_prices.get('SPXS_Open', np.nan), next_day_prices.get('SPXS_Close', np.nan) #

        raw_reward, prev_pos, transacted = 0.0, self.current_portfolio_position, False #
        target_pos = action if action in range(self.num_actions) else self.POSITION_CASH #

        if prev_pos != target_pos:
            transacted = True #
            if prev_pos != self.POSITION_CASH: raw_reward -= (self.transaction_cost_bps / 10000.0) #
            if target_pos != self.POSITION_CASH: raw_reward -= (self.transaction_cost_bps / 10000.0) #

        pnl = 0.0 #
        if target_pos == self.POSITION_SPXL:
            if not pd.isna(spxl_open) and spxl_open != 0 and not pd.isna(spxl_close): pnl = self.leverage * (spxl_close - spxl_open) / spxl_open #
            elif pd.isna(spxl_open) or pd.isna(spxl_close): self.logger.warning(f"SPXL price NaN at data_idx {self.current_data_idx+1}") #
        elif target_pos == self.POSITION_SPXS:
            if not pd.isna(spxs_open) and spxs_open != 0 and not pd.isna(spxs_close): pnl = self.leverage * (spxs_close - spxs_open) / spxs_open #
            elif pd.isna(spxs_open) or pd.isna(spxs_close): self.logger.warning(f"SPXS price NaN at data_idx {self.current_data_idx+1}") #

        if pd.isna(pnl): pnl = 0.0 #

        raw_reward += pnl; self.current_portfolio_position = target_pos #
        self.current_step_in_episode += 1; self.current_data_idx += 1 #

        if (self.episode_length and self.current_step_in_episode >= self.episode_length) or \
           self.current_data_idx >= self.total_data_steps: #
            self.done = True #

        next_obs = self._get_observation() #
        info = {'ts': self.market_features_df.index[self.current_data_idx-1] if self.current_data_idx > 0 and self.current_data_idx <= len(self.market_features_df) else "N/A",
                'act':action, 'tgt_pos': self.portfolio_state_names.get(target_pos), 'raw_reward': raw_reward} #
        return next_obs, raw_reward, self.done, info #
    def get_observation_space_shape(self): return [(self.original_model_feature_count,), (self.num_portfolio_states,)] #
    def get_action_space_size(self): return self.num_actions #
muzero_logger.info("--- Step 8: FinancialMuZeroEnv Class Defined ---") #

# ==============================================================================
# STEP 9: MCTS (Corrected Unpacking)
# ==============================================================================
muzero_logger.info("\n--- Step 9: Defining Monte Carlo Tree Search (MCTS) (Corrected Unpacking) ---") #
class MCTSNode: #
    def __init__(self, parent=None, action_that_led_here=None, prior_p=0.0, hidden_state=None): #
        self.parent, self.action_that_led_here, self.prior_p, self.hidden_state = parent, action_that_led_here, prior_p, hidden_state #
        self.children, self.visit_count, self.value_sum, self.reward_from_parent_action = {}, 0, 0.0, 0.0 #
        self.policy_p_values_from_f_net, self.value_v_from_f_net = None, None #
    def get_mean_value(self): return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0 #
    def is_expanded(self): return self.policy_p_values_from_f_net is not None #
    def select_child_with_ucb(self, config): #
        best_score, best_action, best_child_node = -float('inf'), -1, None #
        if not self.is_expanded(): muzero_logger.error("UCB on non-expanded node!"); return np.random.choice(config.NUM_ACTIONS), None #
        for action_idx in range(config.NUM_ACTIONS): #
            q_val, visits = 0.0, 0 #
            if action_idx in self.children: child = self.children[action_idx]; q_val = child.reward_from_parent_action + config.DISCOUNT_FACTOR * child.get_mean_value(); visits = child.visit_count #
            prior = self.policy_p_values_from_f_net[action_idx] #
            parent_visit_count_for_ucb = self.visit_count if self.visit_count > 0 else 1 #
            ucb_exploration_component = prior * (config.MCTS_PB_C_BASE + math.log((parent_visit_count_for_ucb + config.MCTS_PB_C_BASE + 1) / config.MCTS_PB_C_BASE)) * (math.sqrt(parent_visit_count_for_ucb) / (1 + visits)) #
            ucb = q_val + ucb_exploration_component #
            if ucb > best_score: best_score, best_action, best_child_node = ucb, action_idx, self.children.get(action_idx) #
        if best_action == -1 and config.NUM_ACTIONS > 0: best_action = np.random.choice(config.NUM_ACTIONS); best_child_node = self.children.get(best_action) #
        return best_action, best_child_node #

def run_mcts(initial_observation, h_net_func, g_net_func, f_net_func, config_obj, num_simulations_mcts, num_actions_mcts): #
    obs_np = [np.array(obs, dtype=np.float32)[np.newaxis,...] for obs in initial_observation] #
    root_s = h_net_func.predict_on_batch(obs_np) #
    root = MCTSNode(hidden_state=root_s) #

    # Call f_net for root node: expects [policy_logits, value_tanh, value_logits_pre_tanh]
    policy_logits_at_root, value_tanh_at_root, _ = f_net_func.predict_on_batch(root.hidden_state) #

    probs_r = tf.nn.softmax(policy_logits_at_root[0]).numpy() #
    root.value_v_from_f_net = value_tanh_at_root[0,0] # This is the tanh activated value
    root.policy_p_values_from_f_net = probs_r #

    if config_obj.MCTS_ROOT_DIRICHLET_ALPHA > 0: root.policy_p_values_from_f_net = (1-config_obj.MCTS_ROOT_EXPLORATION_FRACTION)*root.policy_p_values_from_f_net + config_obj.MCTS_ROOT_EXPLORATION_FRACTION*np.random.dirichlet([config_obj.MCTS_ROOT_DIRICHLET_ALPHA]*num_actions_mcts) #
    for _ in range(num_simulations_mcts): #
        node, path, action_sel = root, [root], -1 #
        while node.is_expanded(): #
            action_sel, child_cand = node.select_child_with_ucb(config_obj) #
            if child_cand is not None: node = child_cand; path.append(node) #
            else: break #
        parent_exp, action_exp = node, action_sel #
        val_backup = 0.0 #
        if parent_exp.children.get(action_exp) is None : #
            action_oh = np.zeros(num_actions_mcts,dtype=np.float32); action_oh[action_exp] = 1.0 #

            # Call g_net: expects [next_hidden_state, reward_output_tanh, reward_logits_pre_tanh]
            next_s, reward_g_pred_tanh, _ = g_net_func.predict_on_batch( #
                [parent_exp.hidden_state, action_oh[np.newaxis,...]]
            ) #
            reward_g_pred_scalar = reward_g_pred_tanh[0,0] # Use the tanh-activated reward

            # Call f_net for new node: expects [policy_logits, value_output_tanh, value_logits_pre_tanh]
            policy_logits_new, value_tanh_new, _ = f_net_func.predict_on_batch(next_s) #

            new_node = MCTSNode(parent=parent_exp,action_that_led_here=action_exp,prior_p=parent_exp.policy_p_values_from_f_net[action_exp],hidden_state=next_s) #
            new_node.reward_from_parent_action=reward_g_pred_scalar #
            new_node.policy_p_values_from_f_net=tf.nn.softmax(policy_logits_new[0]).numpy(); #
            new_node.value_v_from_f_net=value_tanh_new[0,0] #

            parent_exp.children[action_exp]=new_node; path.append(new_node) #
            val_backup = new_node.value_v_from_f_net #
        else: val_backup = node.value_v_from_f_net if node.value_v_from_f_net is not None else 0.0 #

        for node_bp in reversed(path): #
            node_bp.visit_count+=1 #
            if node_bp.parent is not None: val_backup = node_bp.reward_from_parent_action + config_obj.DISCOUNT_FACTOR * val_backup #
            node_bp.value_sum += val_backup #

    visits = np.array([root.children[a].visit_count if a in root.children else 0 for a in range(num_actions_mcts)]) #
    action_probs = (visits/np.sum(visits)) if np.sum(visits)>0 else (np.ones(num_actions_mcts)/num_actions_mcts) #
    return action_probs, root.get_mean_value(), root #
muzero_logger.info("--- Step 9: MCTS Implementation Defined (Corrected Unpacking) ---") #

# ==============================================================================
# STEP 10: REPLAY BUFFER (Assumed to be correct as per original script)
# ==============================================================================
# ... (ReplayBuffer class definition as provided in original script) ...
muzero_logger.info("\n--- Step 10: Defining Replay Buffer ---") #
class ReplayBuffer: #
    def __init__(self, config, logger): #
        self.config, self.logger, self.buffer_size = config, logger, config.REPLAY_BUFFER_SIZE #
        self.buffer = collections.deque(maxlen=self.buffer_size); self.num_games_added, self.num_samples_drawn = 0,0 #
        self.save_path = os.path.join(self.config.MUZERO_REPLAY_BUFFER_DIR, "replay_buffer.pkl") #
        self.logger.debug(f"ReplayBuffer init. Size: {self.buffer_size}. Path: {self.save_path}") #
    def add_game(self, game_trajectory): #
        if len(self.buffer) >= self.buffer_size: self.buffer.popleft() #
        self.buffer.append(game_trajectory); self.num_games_added += 1 #
        if self.num_games_added % 10 == 0: self.logger.debug(f"Game {self.num_games_added} added. Buffer: {len(self.buffer)}/{self.buffer_size}") #
    def sample_batch(self, batch_size): #
        if not self.buffer: self.logger.warning("Buffer empty, cannot sample."); return None #
        actual_batch_size = min(batch_size, len(self.buffer)) #
        unroll_K = self.config.NUM_UNROLL_STEPS #
        indices = np.random.choice(len(self.buffer), size=actual_batch_size, replace=len(self.buffer) < batch_size) #
        samples = [] #
        for i in indices: #
            game = self.buffer[i]; game_len_actions = len(game['actions_list']) #
            if game_len_actions < 1: continue #
            max_start_idx_for_obs = len(game['observations_list']) -1 #
            if max_start_idx_for_obs < 0 : continue #
            start_idx = np.random.randint(0, max_start_idx_for_obs + 1) #
            obs_init = game['observations_list'][start_idx] #
            actions, rewards_target, policies_target, values_target = [], [], [], [] #
            policies_target.append(game['mcts_policy_targets_list'][start_idx]) #
            values_target.append(game['value_targets_list'][start_idx]) #
            for k_idx in range(unroll_K): #
                action_step_idx = start_idx + k_idx #
                next_state_target_idx = start_idx + k_idx + 1 #
                if action_step_idx < game_len_actions: #
                    actions.append(game['actions_list'][action_step_idx]) #
                    rewards_target.append(game['rewards_list'][action_step_idx]) #
                    if next_state_target_idx < len(game['observations_list']): #
                        policies_target.append(game['mcts_policy_targets_list'][next_state_target_idx]) #
                        values_target.append(game['value_targets_list'][next_state_target_idx]) #
                    else: policies_target.append(np.ones(self.config.NUM_ACTIONS)/self.config.NUM_ACTIONS); values_target.append(0.0) #
                else: #
                    actions.append(np.random.randint(0,self.config.NUM_ACTIONS)); rewards_target.append(0.0) #
                    policies_target.append(np.ones(self.config.NUM_ACTIONS)/self.config.NUM_ACTIONS); values_target.append(0.0) #
            samples.append({'initial_observation':obs_init, 'action_history':actions,
                            'target_rewards':rewards_target,
                            'target_policies':policies_target,
                            'target_values':values_target}) #
        return samples if samples else None #
    def __len__(self): return len(self.buffer) #
    def save(self):
        try:
            with open(self.save_path,'wb') as f: pickle.dump(list(self.buffer),f)
            self.logger.info(f"Replay buffer saved to {self.save_path}. Size: {len(self.buffer)} games.")
        except Exception as e: self.logger.error(f"Error saving replay buffer: {e}", exc_info=True)
    def load(self):
        if os.path.exists(self.save_path):
            try:
                with open(self.save_path,'rb') as f:
                    loaded_list = pickle.load(f)
                    self.buffer=collections.deque(loaded_list, maxlen=self.buffer_size)
                    self.num_games_added=len(self.buffer)
                self.logger.info(f"Replay buffer loaded from {self.save_path}. Size: {len(self.buffer)} games.")
            except Exception as e: self.logger.error(f"Error loading replay buffer: {e}. Starting empty.", exc_info=True); self.buffer=collections.deque(maxlen=self.buffer_size)
        else: self.logger.info(f"No replay buffer file found at {self.save_path}. Starting with an empty buffer.")
muzero_logger.info("--- Step 10: ReplayBuffer Class Defined ---") #

# ==============================================================================
# STEP 11: LOSS FUNCTIONS (Assumed to be correct as per original script)
# ==============================================================================
# ... (calculate_muzero_losses definition as provided in original script) ...
muzero_logger.info("\n--- Step 11: Defining Loss Functions ---") #
mse_loss_fn = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE) #
cross_entropy_loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE) #
def calculate_muzero_losses(pred_vals, target_vals, pred_rewards, target_rewards, pred_policies, target_policies, nets, cfg): #
    v_loss, r_loss, p_loss = tf.constant(0.0), tf.constant(0.0), tf.constant(0.0) #
    for k in range(cfg.NUM_UNROLL_STEPS + 1): v_loss += mse_loss_fn(target_vals[:,k,:], pred_vals[k]) #
    for k in range(cfg.NUM_UNROLL_STEPS): r_loss += mse_loss_fn(target_rewards[:,k,:], pred_rewards[k]) #
    for k in range(cfg.NUM_UNROLL_STEPS + 1): p_loss += cross_entropy_loss_fn(target_policies[:,k,:], pred_policies[k]) #
    l2_loss = tf.constant(0.0) #
    if cfg.WEIGHT_DECAY > 0: #
        for net_item in nets: #
            for var in net_item.trainable_variables: #
                if 'bias' not in var.name and 'normalization' not in var.name and 'gamma' not in var.name and 'beta' not in var.name : l2_loss += tf.nn.l2_loss(var) #
    l2_loss *= cfg.WEIGHT_DECAY #
    avg_v = v_loss / tf.cast(cfg.NUM_UNROLL_STEPS+1,tf.float32); avg_r = r_loss / tf.cast(tf.maximum(1,cfg.NUM_UNROLL_STEPS),tf.float32); avg_p = p_loss / tf.cast(cfg.NUM_UNROLL_STEPS+1,tf.float32) #
    w_v, w_r, w_p = avg_v*cfg.VALUE_LOSS_WEIGHT, avg_r*cfg.REWARD_LOSS_WEIGHT, avg_p*cfg.POLICY_LOSS_WEIGHT #
    total = w_v + w_r + w_p + l2_loss #
    return total, {"total_loss":total, "value_loss":avg_v, "reward_loss":avg_r, "policy_loss":avg_p, "l2_reg_loss":l2_loss} #
muzero_logger.info("--- Step 11: Loss Functions Defined ---") #

# ==============================================================================
# STEP 12: MAIN TRAINING LOOP
# ==============================================================================
muzero_logger.info("\n--- Step 12: Defining Main Training Loop (BN + Logits) ---") #

TF_PRINT_INTERVAL = tf.constant(10, dtype=tf.int64)

def get_muzero_agent_networks(config, num_original_features, num_portfolio_states, num_actions, logger): #
    logger.info(f"Instantiating MuZero networks (with BN & Logit outputs). Base NN: {config.FINAL_BEST_NN_PATH}, Transfer Layer: {config.TRANSFER_LAYER_NAME}")
    if not os.path.exists(config.FINAL_BEST_NN_PATH): raise FileNotFoundError(f"Base NN {config.FINAL_BEST_NN_PATH} not found.")
    loaded_nn = keras.models.load_model(config.FINAL_BEST_NN_PATH)
    try: base_model = keras.Model(inputs=loaded_nn.input, outputs=loaded_nn.get_layer(config.TRANSFER_LAYER_NAME).output, name="BaseRepresentation")
    except ValueError as e: logger.critical(f"Layer '{config.TRANSFER_LAYER_NAME}' not in loaded NN."); loaded_nn.summary(print_fn=logger.error); raise e
    base_model.trainable = not config.FREEZE_TRANSFER_WEIGHTS_INITIALLY
    logger.info(f"Base representation model FROZEN: {not base_model.trainable}")

    # Representation Network (h_net)
    m_in = keras.layers.Input(shape=(num_original_features,), name='h_market_in')
    p_in = keras.layers.Input(shape=(num_portfolio_states,), name='h_portfolio_in')
    enc_m = base_model(m_in)
    # Potential BN here if enc_m is not already well-conditioned by base_model's last layer
    concat_h_layer = keras.layers.Concatenate()([enc_m, p_in])
    # Potential BN here before the final dense if needed
    s_out_h_layer = keras.layers.Dense(config.MUZERO_HIDDEN_STATE_SIZE, activation='relu', name='h_final_dense')(concat_h_layer)
    h_net = keras.Model(inputs=[m_in, p_in], outputs=s_out_h_layer, name="Representation_h")

    # Dynamics Network (g_net)
    s_g_in = keras.layers.Input(shape=(config.MUZERO_HIDDEN_STATE_SIZE,), name='g_s_in')
    a_g_in = keras.layers.Input(shape=(num_actions,), name='g_a_in')
    concat_g_layer = keras.layers.Concatenate(name='g_concat_s_a')([s_g_in, a_g_in])

    # Path for next state prediction
    x_dyn_s = keras.layers.Dense(config.DYNAMICS_INTERNAL_SIZE, name='g_dynamics_hidden_dense_linear')(concat_g_layer)
    x_dyn_s = keras.layers.BatchNormalization(name='g_dynamics_bn')(x_dyn_s)
    g_h1_processed_s = keras.layers.Activation('relu', name='g_dynamics_relu')(x_dyn_s)
    s_g_out_layer = keras.layers.Dense(config.MUZERO_HIDDEN_STATE_SIZE, activation='relu', name='g_s_out')(g_h1_processed_s)

    # Path for reward prediction
    x_dyn_r = keras.layers.Dense(config.DYNAMICS_INTERNAL_SIZE_REWARD, name='g_reward_hidden_dense_linear')(concat_g_layer)
    x_dyn_r = keras.layers.BatchNormalization(name='g_reward_bn')(x_dyn_r)
    g_h1_processed_r = keras.layers.Activation('relu', name='g_reward_relu')(x_dyn_r)
    r_g_logits_layer = keras.layers.Dense(1, name='g_r_logits')(g_h1_processed_r)
    r_g_out_layer = keras.layers.Activation('tanh', name='g_r_out')(r_g_logits_layer)

    g_net = keras.Model(inputs=[s_g_in, a_g_in], outputs=[s_g_out_layer, r_g_out_layer, r_g_logits_layer], name="Dynamics_g_BN")

    # Prediction Network (f_net)
    s_f_in = keras.layers.Input(shape=(config.MUZERO_HIDDEN_STATE_SIZE,), name='f_s_in')

    # Path for policy prediction
    x_pred_p = keras.layers.Dense(config.PREDICTION_INTERNAL_SIZE_POLICY, name='f_policy_hidden_dense_linear')(s_f_in)
    x_pred_p = keras.layers.BatchNormalization(name='f_policy_bn')(x_pred_p)
    f_p_h1_processed = keras.layers.Activation('relu', name='f_policy_relu')(x_pred_p)
    p_f_out_layer = keras.layers.Dense(num_actions, name='f_p_logits')(f_p_h1_processed)

    # Path for value prediction
    x_pred_v = keras.layers.Dense(config.PREDICTION_INTERNAL_SIZE_VALUE, name='f_value_hidden_dense_linear')(s_f_in)
    x_pred_v = keras.layers.BatchNormalization(name='f_value_bn')(x_pred_v)
    f_v_h1_processed = keras.layers.Activation('relu', name='f_value_relu')(x_pred_v)
    v_f_logits_layer = keras.layers.Dense(1, name='f_v_logits')(f_v_h1_processed)
    v_f_out_layer = keras.layers.Activation('tanh', name='f_v_out')(v_f_logits_layer)

    f_net = keras.Model(inputs=s_f_in, outputs=[p_f_out_layer, v_f_out_layer, v_f_logits_layer], name="Prediction_f_BN")

    logger.info("MuZero networks created (with BN & Logit outputs).")
    return h_net, g_net, f_net, base_model

def train_muzero_agent(config: MuZeroConfig, logger: logging.Logger): #
    original_log_level = logger.level #
    script_version = getattr(config, 'SCRIPT_VERSION', 'V11_BN_Logits') #
    logger.info(f"===== STARTING MUZERO AGENT TRAINING (Version: {script_version}) =====") #
    np.random.seed(config.SEED); tf.random.set_seed(config.SEED); random.seed(config.SEED) #
    logger.info("Initializing components...") #
    environment = FinancialMuZeroEnv(
        config_mz_instance=config, logger_instance=logger,
        trained_feature_names_list=TRAINED_FEATURE_NAMES_GLOBAL,
        global_ga_mask_arr=feature_mask_global,
        start_index=0, episode_length=getattr(config, 'INITIAL_EPISODE_LENGTH', 252)
    ) #
    h_net_train, g_net_train, f_net_train, base_repr_train = get_muzero_agent_networks(
        config, ORIGINAL_MODEL_FEATURE_COUNT, config.PORTFOLIO_STATE_SIZE, config.NUM_ACTIONS, logger
    ) #
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=config.LEARNING_RATE,
        clipnorm=getattr(config, 'GRADIENT_CLIP_NORM', 1.0)
    ) #
    replay_buffer = ReplayBuffer(config, logger) #

    # ... (Checkpoint loading logic - will likely start fresh due to BN architecture change) ...
    start_step = 1
    latest_step_file_path = os.path.join(config.MUZERO_MODELS_DIR, getattr(config, 'LATEST_CHECKPOINT_FILENAME', "latest_checkpoint_step.txt"))
    load_checkpoint_explicitly_requested_step = getattr(config, 'LOAD_CHECKPOINT_FROM_STEP', 0)
    always_try_resume_flag = getattr(config, 'ALWAYS_ATTEMPT_RESUME', True)
    determined_load_step = 0
    if load_checkpoint_explicitly_requested_step > 0:
        determined_load_step = load_checkpoint_explicitly_requested_step
    elif always_try_resume_flag and load_checkpoint_explicitly_requested_step != 0:
        if os.path.exists(latest_step_file_path):
            try:
                with open(latest_step_file_path, 'r') as f_s_l: latest_saved_step = int(f_s_l.read().strip())
                determined_load_step = latest_saved_step
                logger.info(f"Found latest checkpoint step file: {latest_step_file_path}, indicates step: {latest_saved_step}.")
            except Exception as e_l: logger.warning(f"Could not read latest checkpoint step from {latest_step_file_path}: {e_l}.")
        else: logger.info(f"No latest checkpoint file found at {latest_step_file_path}.")
        if load_checkpoint_explicitly_requested_step == -1 and determined_load_step == 0 : logger.info("LOAD_CHECKPOINT_FROM_STEP was -1 but no valid latest found to load.")

    if determined_load_step > 0:
        logger.info(f"Attempting to load checkpoint from determined step {determined_load_step}...")
        try:
            h_path = config.MUZERO_CHECKPOINT_FORMAT.format(step=determined_load_step) + "_h.weights.h5"
            g_path = config.MUZERO_CHECKPOINT_FORMAT.format(step=determined_load_step) + "_g.weights.h5"
            f_path = config.MUZERO_CHECKPOINT_FORMAT.format(step=determined_load_step) + "_f.weights.h5"
            if os.path.exists(h_path) and os.path.exists(g_path) and os.path.exists(f_path):
                h_net_train.load_weights(h_path); g_net_train.load_weights(g_path); f_net_train.load_weights(f_path)
                logger.info(f"Successfully loaded model weights from step {determined_load_step}.")
                replay_buffer.load()
                start_step = determined_load_step + 1
                logger.info(f"Resuming training from step {start_step}.")
                if config.FREEZE_TRANSFER_WEIGHTS_INITIALLY and start_step > config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP and not base_repr_train.trainable:
                    base_repr_train.trainable = True; logger.info(f"Resuming: Unfroze base_repr_train.")
                elif config.FREEZE_TRANSFER_WEIGHTS_INITIALLY and base_repr_train.trainable and start_step <= config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP:
                     base_repr_train.trainable = False; logger.info(f"Resuming: Kept base_repr_train frozen.")
            else:
                logger.warning(f"Checkpoint weight files for step {determined_load_step} not fully found. Paths checked: \n H: {h_path}\n G: {g_path}\n F: {f_path}\nStarting training from scratch."); start_step = 1
        except Exception as e_load:
            logger.error(f"Error loading checkpoint from step {determined_load_step}: {e_load}. Starting fresh.", exc_info=True); start_step = 1
    else:
        logger.info("Starting training from scratch (no valid checkpoint specified or found to load).")


    @tf.function
    def train_step_tf(batch_tf, current_train_step_tf):
        with tf.GradientTape() as tape:
            vars_train = []
            for layer in h_net_train.layers:
                if layer.name == 'BaseRepresentation':
                    if base_repr_train.trainable: vars_train.extend(layer.trainable_variables)
                elif layer.trainable: vars_train.extend(layer.trainable_variables)
            vars_train.extend(g_net_train.trainable_variables)
            vars_train.extend(f_net_train.trainable_variables)

            s_k = h_net_train([batch_tf['initial_market_features'], batch_tf['initial_portfolio_states']], training=True)

            preds_p, preds_v_tanh, preds_r_tanh = [], [], [] # Tanh activated outputs for loss
            preds_v_logits, preds_r_logits = [], [] # Pre-activation logits for debugging

            # Initial prediction for s_0
            policy_s0_logits, value_s0_tanh, value_s0_logits = f_net_train(s_k, training=True)
            preds_p.append(policy_s0_logits)
            preds_v_tanh.append(value_s0_tanh)
            preds_v_logits.append(value_s0_logits)

            # TF.PRINT for initial predictions
            def print_initial_tf_monitoring_fn():
                tf.print("--- S", current_train_step_tf, "TF_DEBUG (Initial s_0, Batch[0]) ---", output_stream=sys.stdout)
                tf.print("  Target Value (scaled):", batch_tf['target_values'][0,0,0], output_stream=sys.stdout)
                tf.print("  Pred Value Logits (s_0):", preds_v_logits[0][0,0], output_stream=sys.stdout)
                tf.print("  Pred Value (tanh, s_0):", preds_v_tanh[0][0,0], output_stream=sys.stdout)
                tf.print("  Target Policy (MCTS, p_0):", batch_tf['target_policies'][0,0,:], summarize=-1, output_stream=sys.stdout)
                tf.print("  Pred Policy Logits (s_0):", preds_p[0][0,:], summarize=-1, output_stream=sys.stdout)
                return tf.constant(True)

            tf.cond(
                tf.equal(tf.math.floormod(current_train_step_tf, TF_PRINT_INTERVAL), tf.cast(0, dtype=TF_PRINT_INTERVAL.dtype)),
                print_initial_tf_monitoring_fn,
                lambda: tf.constant(False)
            )

            # Main unroll loop
            for k_step in range(config.NUM_UNROLL_STEPS):
                actions_g = tf.one_hot(batch_tf['action_history'][:, k_step], depth=config.NUM_ACTIONS)

                s_k_plus_1, rew_k_tanh, rew_k_logits = g_net_train([s_k, actions_g], training=True)
                preds_r_tanh.append(rew_k_tanh)
                preds_r_logits.append(rew_k_logits)

                s_k = s_k_plus_1

                policy_sk_plus_1_logits, value_sk_plus_1_tanh, value_sk_plus_1_logits = f_net_train(s_k, training=True)
                preds_p.append(policy_sk_plus_1_logits)
                preds_v_tanh.append(value_sk_plus_1_tanh)
                preds_v_logits.append(value_sk_plus_1_logits)

                # TF.PRINT for unrolled predictions
                def print_unroll_tf_monitoring_fn():
                    tf.print("--- S", current_train_step_tf, "TF_DEBUG (Unroll k_step=", k_step, ", Batch[0]) ---", output_stream=sys.stdout)
                    tf.print("  Target Reward (scaled, r_k):", batch_tf['target_rewards'][0,k_step,0], output_stream=sys.stdout)
                    tf.print("  Pred Reward Logits (r_k):", preds_r_logits[k_step][0,0], output_stream=sys.stdout)
                    tf.print("  Pred Reward (tanh, r_k):", preds_r_tanh[k_step][0,0], output_stream=sys.stdout)

                    tf.print("  Target Value (scaled, v_{k+1}):", batch_tf['target_values'][0,k_step+1,0], output_stream=sys.stdout)
                    tf.print("  Pred Value Logits (s_{k+1}):", preds_v_logits[k_step+1][0,0], output_stream=sys.stdout)
                    tf.print("  Pred Value (tanh, s_{k+1}):", preds_v_tanh[k_step+1][0,0], output_stream=sys.stdout)
                    tf.print("  Target Policy (MCTS, p_{k+1}):", batch_tf['target_policies'][0,k_step+1,:], summarize=-1, output_stream=sys.stdout)
                    tf.print("  Pred Policy Logits (s_{k+1}):", preds_p[k_step+1][0,:], summarize=-1, output_stream=sys.stdout)
                    pred_policy_probs_sk_plus_1 = tf.nn.softmax(preds_p[k_step+1][0,:])
                    tf.print("  Pred Policy Probs (s_{k+1}):", pred_policy_probs_sk_plus_1, summarize=-1, output_stream=sys.stdout)
                    return tf.constant(True)

                tf.cond(
                    tf.logical_and(
                        tf.equal(tf.math.floormod(current_train_step_tf, TF_PRINT_INTERVAL), tf.cast(0, dtype=TF_PRINT_INTERVAL.dtype)),
                        tf.equal(k_step, 0)
                    ),
                    print_unroll_tf_monitoring_fn,
                    lambda: tf.constant(False)
                )

            loss_total, loss_comps = calculate_muzero_losses(
                preds_v_tanh, batch_tf['target_values'],
                preds_r_tanh, batch_tf['target_rewards'],
                preds_p, batch_tf['target_policies'],
                [h_net_train, g_net_train, f_net_train], config
            )
        if vars_train:
            grads = tape.gradient(loss_total, vars_train)
            optimizer.apply_gradients(zip(grads, vars_train))
        else:
            tf.print("No trainable vars in train_step_tf.", output_stream=sys.stdout)
        return loss_total, loss_comps

    # Main game playing and training loop
    # ... (Rest of the train_muzero_agent function remains the same as the previous "Rewritten Monolith")
    # Ensure that within this loop, calls to f_net_train or g_net_train (if any outside MCTS for bootstrapping, like at episode end)
    # are also updated to handle 3 outputs. The current logic for end-of-episode bootstrap seems to be:
    # last_policy_logits, last_value_output_tanh, _ = f_net_train.predict_on_batch(last_s) # This is already correct.

    for step in range(start_step, config.MAX_TRAINING_STEPS + 1): #
        loop_start_time = time.time() #
        if config.FREEZE_TRANSFER_WEIGHTS_INITIALLY and step >= config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP and not base_repr_train.trainable: #
            base_repr_train.trainable = True; logger.info(f"Step {step}: Unfroze base_repr_train.") #

        current_episode_max_len = environment.episode_length or getattr(config, 'DEFAULT_EPISODE_LENGTH_PLAY', 252) #
        max_possible_start_index = environment.total_data_steps - current_episode_max_len - 1 #
        if max_possible_start_index < 0: max_possible_start_index = 0 #
        current_start_idx_play = np.random.randint(0, max_possible_start_index + 1) if max_possible_start_index >=0 else 0 #

        obs_loop = environment.reset(start_index=current_start_idx_play, episode_length=current_episode_max_len) #
        game_data = {'observations_list':[obs_loop],'actions_list':[],'rewards_list':[],
                     'mcts_policy_targets_list':[],'value_targets_list':[],
                     'raw_mcts_values_list': [], 'raw_rewards_list': [],
                     'done_list':[],'total_episode_reward_raw':0.0} #
        done_loop, game_steps_count = False, 0 #

        while not done_loop and game_steps_count < current_episode_max_len : #
            mcts_policy_target, mcts_val_raw, _ = run_mcts(
                obs_loop, h_net_train,g_net_train,f_net_train,config,config.NUM_SIMULATIONS,config.NUM_ACTIONS) #

            mcts_val_scaled_target = np.tanh(mcts_val_raw / config.VALUE_TARGET_SCALING_DIVISOR) #

            current_temp = config_mz.INITIAL_TEMPERATURE if step < config_mz.ANNEAL_TEMPERATURE_AFTER_N_MOVES else config_mz.FINAL_TEMPERATURE

            action_probs_for_sampling = mcts_policy_target #
            if np.sum(action_probs_for_sampling) > 1e-6 and current_temp > 1e-3: #
                action_probs_temp = np.power(action_probs_for_sampling + 1e-9, 1.0 / current_temp) #
                action_probs_temp_sum = np.sum(action_probs_temp) #
                if action_probs_temp_sum > 1e-6 : action_probs_temp /= action_probs_temp_sum #
                else: action_probs_temp = np.ones_like(action_probs_temp) / len(action_probs_temp) #
                if np.isnan(action_probs_temp).any(): action = np.argmax(action_probs_for_sampling) #
                else: action = np.random.choice(config.NUM_ACTIONS, p=action_probs_temp) #
            else: action = np.argmax(action_probs_for_sampling) #
            logger.debug(f"  S{step} G{len(replay_buffer)+1}-P{game_steps_count+1}: Pol={np.round(mcts_policy_target,2)}, RawVal={mcts_val_raw:.3f}, ScaledValTgt={mcts_val_scaled_target:.3f}, Act={action} (T={current_temp:.2f})") #

            next_obs, raw_reward_from_env, done_loop, info = environment.step(action) #

            scaled_reward_target_for_buffer = np.tanh(raw_reward_from_env / config.REWARD_TARGET_SCALING_DIVISOR) #
            logger.debug(f"    Env Step: RawRew={raw_reward_from_env:.4f}, ScaledRewTgt={scaled_reward_target_for_buffer:.4f} Done={done_loop}, TgtPos={info.get('tgt_pos')}") #

            game_data['actions_list'].append(action);
            game_data['rewards_list'].append(scaled_reward_target_for_buffer)
            game_data['raw_rewards_list'].append(raw_reward_from_env);
            game_data['mcts_policy_targets_list'].append(mcts_policy_target);
            game_data['value_targets_list'].append(mcts_val_scaled_target);
            game_data['raw_mcts_values_list'].append(mcts_val_raw);
            game_data['done_list'].append(done_loop);
            game_data['total_episode_reward_raw'] += raw_reward_from_env
            obs_loop = next_obs
            if not done_loop: game_data['observations_list'].append(obs_loop)
            game_steps_count += 1

        if not done_loop and len(game_data['observations_list']) > len(game_data['mcts_policy_targets_list']):
            last_obs = game_data['observations_list'][-1]
            last_s = h_net_train.predict_on_batch([np.array(o,dtype=np.float32)[np.newaxis,...] for o in last_obs])
            last_policy_logits, last_value_output_tanh, _ = f_net_train.predict_on_batch(last_s)
            game_data['mcts_policy_targets_list'].append(tf.nn.softmax(last_policy_logits[0]).numpy());
            game_data['value_targets_list'].append(last_value_output_tanh[0,0])
            game_data['raw_mcts_values_list'].append(last_value_output_tanh[0,0]) # Storing tanh output as raw mcts value for bootstrap

        replay_buffer.add_game(game_data)
        logger.debug(f"S{step} GameStore: Game {len(replay_buffer)} added. EpRawRew={game_data['total_episode_reward_raw']:.2f}, Steps={game_steps_count}")

        if step % 50 == 0:
            logger.info(f"--- S{step} Post-Game Monitoring (Targets from Last Episode) ---")
            if game_data.get('raw_mcts_values_list') and game_data.get('value_targets_list'):
                raw_vals_to_log = game_data['raw_mcts_values_list']
                scaled_val_targets_to_log = game_data['value_targets_list']
                num_value_samples_to_display = min(5, len(raw_vals_to_log))
                if num_value_samples_to_display > 0:
                    logger.info(f"  Last {num_value_samples_to_display} Raw MCTS Values from episode: {['{:.3f}'.format(x) for x in raw_vals_to_log[-num_value_samples_to_display:]]}")
                    logger.info(f"  Last {num_value_samples_to_display} Scaled Value Targets (for loss): {['{:.3f}'.format(x) for x in scaled_val_targets_to_log[-num_value_samples_to_display:]]}")
                else: logger.info("  MCTS Value lists for logging were empty or too short for this episode's sample.")
            else: logger.info("  'raw_mcts_values_list' or 'value_targets_list' not found/empty in game_data for logging.")
            if game_data.get('raw_rewards_list') and game_data.get('rewards_list'):
                raw_rewards_to_log = game_data['raw_rewards_list']
                scaled_reward_targets_to_log = game_data['rewards_list']
                num_reward_samples_to_display = min(5, len(raw_rewards_to_log))
                if num_reward_samples_to_display > 0:
                    logger.info(f"  Last {num_reward_samples_to_display} Raw Env Rewards from episode: {['{:.3f}'.format(x) for x in raw_rewards_to_log[-num_reward_samples_to_display:]]}")
                    logger.info(f"  Last {num_reward_samples_to_display} Scaled Reward Targets (for loss): {['{:.3f}'.format(x) for x in scaled_reward_targets_to_log[-num_reward_samples_to_display:]]}")
                else: logger.info("  Reward lists for logging were empty or too short for this episode's sample.")
            else: logger.info("  'raw_rewards_list' or 'rewards_list' not found/empty in game_data for logging.")
            logger.info(f"--- End Post-Game Monitoring S{step} ---")

        min_buffer_for_training = getattr(config, 'MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING', config.BATCH_SIZE * 2)
        if len(replay_buffer) >= min_buffer_for_training:
            raw_b = replay_buffer.sample_batch(config.BATCH_SIZE)
            if raw_b:
                tf_b = {'initial_market_features': tf.constant(np.array([s['initial_observation'][0] for s in raw_b],dtype=np.float32)),
                        'initial_portfolio_states': tf.constant(np.array([s['initial_observation'][1] for s in raw_b],dtype=np.float32)),
                        'action_history': tf.constant(np.array([s['action_history'] for s in raw_b],dtype=np.int32)),
                        'target_rewards': tf.constant(np.array([s['target_rewards'] for s in raw_b],dtype=np.float32)[...,np.newaxis]),
                        'target_policies': tf.constant(np.array([s['target_policies'] for s in raw_b],dtype=np.float32)),
                        'target_values': tf.constant(np.array([s['target_values'] for s in raw_b],dtype=np.float32)[...,np.newaxis])}
                loss_val, metrics = train_step_tf(tf_b, tf.cast(step, dtype=tf.int64))
                if step % 10 == 0:
                    log_msg = f"S{step} Train: Loss={loss_val.numpy():.4f}, " + \
                              ", ".join([f"{k.replace('_loss','L').replace('total_','T_')[:7]}={v.numpy():.3f}" for k,v in metrics.items() if k not in ['total_loss', 'l2_reg_loss']]) + \
                              f", L2={metrics['l2_reg_loss'].numpy():.5f}"
                    logger.info(log_msg)
            else: logger.debug(f"S{step}: Training skipped, batch from buffer was empty.")
        elif step % 10 == 0 : logger.info(f"S{step}: Buffer fill ({len(replay_buffer)}/{min_buffer_for_training} games)...")

        if step % config.CHECKPOINT_INTERVAL == 0 and step > 0:
            logger.info(f"S{step}: Attempting to save checkpoint (Interval: {config.CHECKPOINT_INTERVAL})...")
            os.makedirs(config.MUZERO_MODELS_DIR,exist_ok=True)
            pref = config.MUZERO_CHECKPOINT_FORMAT.format(step=step)
            h_path_save = pref+"_h.weights.h5"; g_path_save = pref+"_g.weights.h5"; f_path_save = pref+"_f.weights.h5"
            logger.info(f"  Target H-Net: {h_path_save}, G-Net: {g_path_save}, F-Net: {f_path_save}")
            logger.info(f"  Target Replay Buffer: {replay_buffer.save_path}, Latest Step File: {latest_step_file_path}")
            try:
                h_net_train.save_weights(h_path_save); logger.info(f"    SUCCESS: Saved H-Net for S{step}")
                g_net_train.save_weights(g_path_save); logger.info(f"    SUCCESS: Saved G-Net for S{step}")
                f_net_train.save_weights(f_path_save); logger.info(f"    SUCCESS: Saved F-Net for S{step}")
                replay_buffer.save();
                with open(latest_step_file_path, 'w') as f_sw: f_sw.write(str(step))
                logger.info(f"    SUCCESS: Updated latest step file '{latest_step_file_path}' to {step}.")
                logger.info(f"----> CHECKPOINT COMPLETELY SAVED for S{step} <----")
            except Exception as e_ckpt: logger.error(f"Error saving complete checkpoint at step {step}: {e_ckpt}", exc_info=True)

        if step % 100 == 0:
            gpu_mem_str = "N/A"
            try:
                if tf.config.list_physical_devices('GPU'):
                    mem_info = tf.config.experimental.get_memory_info('GPU:0')
                    gpu_mem_str = f"{mem_info['current'] / 1024**2:.1f}MB / {mem_info['peak'] / 1024**2:.1f}MB (peak)"
                else: gpu_mem_str = "No GPU found by TF"
            except Exception as e_gpu_log: gpu_mem_str = f"Error getting GPU mem: {str(e_gpu_log)}"
            loop_duration = time.time() - loop_start_time
            logger.info(f"S{step} LoopTime: {loop_duration:.2f}s. GPU Mem: {gpu_mem_str}")

    logger.info("===== MUZERO AGENT TRAINING FINISHED =====") #
    logger.setLevel(original_log_level) #

# --- Execution Guard ---
if __name__ == "__main__": #
    missing_comp = False #
    essential_globals_for_main_execution = ['config_mz', 'muzero_logger',
                                            'TRAINED_FEATURE_NAMES_GLOBAL', 'feature_mask_global',
                                            'ORIGINAL_MODEL_FEATURE_COUNT', 'load_ga_feature_mask',
                                            'FinancialMuZeroEnv', 'ReplayBuffer', 'MCTSNode', 'run_mcts',
                                            'calculate_muzero_losses', 'get_muzero_agent_networks' ] #
    for var_name_item_main in essential_globals_for_main_execution: #
        if var_name_item_main not in globals() or globals()[var_name_item_main] is None: #
            print(f"CRITICAL: Prerequisite '{var_name_item_main}' missing for training loop.") #
            missing_comp = True #
    if not missing_comp: #
        script_version_tag = getattr(config_mz, 'SCRIPT_VERSION', 'V11_BN_Logits')
        muzero_logger.info(f"All prerequisites met. Starting MuZero training agent ({script_version_tag})...") #
        try: #
            if not hasattr(config_mz, 'ALWAYS_ATTEMPT_RESUME'): config_mz.ALWAYS_ATTEMPT_RESUME = True
            if not hasattr(config_mz, 'LOAD_CHECKPOINT_FROM_STEP'): config_mz.LOAD_CHECKPOINT_FROM_STEP = -1
            if not hasattr(config_mz, 'LATEST_CHECKPOINT_FILENAME'): config_mz.LATEST_CHECKPOINT_FILENAME = "latest_checkpoint_step.txt"
            if not hasattr(config_mz, 'MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING'): config_mz.MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING = config_mz.BATCH_SIZE * 2
            if not hasattr(config_mz, 'INITIAL_EPISODE_LENGTH'): config_mz.INITIAL_EPISODE_LENGTH = 252
            if not hasattr(config_mz, 'DEFAULT_EPISODE_LENGTH_PLAY'): config_mz.DEFAULT_EPISODE_LENGTH_PLAY = 252
            if not hasattr(config_mz, 'GRADIENT_CLIP_NORM'): config_mz.GRADIENT_CLIP_NORM = 1.0

            config_mz.SCRIPT_VERSION = script_version_tag

            gpu_devices = tf.config.list_physical_devices('GPU')
            if not gpu_devices:
                muzero_logger.warning("NO GPU DETECTED BY TENSORFLOW. Training will be very slow.")
            else:
                muzero_logger.info(f"GPU detected: {gpu_devices}")

            train_muzero_agent(config_mz, muzero_logger)
        except Exception as e_main:
            muzero_logger.critical(f"Unhandled exception in __main__ calling train_muzero_agent: {e_main}", exc_info=True)
    else:
        print("Halting due to missing prerequisites for __main__ execution block.")

In [ ]:
!pip install optuna

In [ ]:
# This entire block should be run as a SINGLE CELL
# @title 12 OPTUNA # AFTER Section 1 (defining config_mz, muzero_logger, load_ga_feature_mask) is complete.

import tensorflow as tf
from tensorflow import keras
import numpy as np
import time
import os
import math
import collections
import json
import logging
import sys
import random
import pandas as pd
import pickle
import copy
import optuna # Added for Optuna

# --- Basic Logger & Config Check ---
if 'muzero_logger' not in globals() or muzero_logger is None:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(module)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    muzero_logger = logging.getLogger("MuZeroProject_Optuna") #
    muzero_logger.warning("Re-initialized basic logger for Monolithic Block (Optuna).")

essential_globals_for_script = ['config_mz', 'muzero_logger', 'load_ga_feature_mask'] #
for var_name in essential_globals_for_script:
    if var_name not in globals() or globals()[var_name] is None:
        muzero_logger.critical(f"Essential global component '{var_name}' from Section 1 is missing. Halting.") #
        raise NameError(f"Missing Section 1 component '{var_name}'.")

# ==============================================================================
# Optuna Configuration (User can adjust these)
# ==============================================================================
OPTUNA_N_TRIALS = getattr(config_mz, 'OPTUNA_N_TRIALS', 30)
OPTUNA_TRIAL_TOTAL_GAME_STEPS = getattr(config_mz, 'OPTUNA_TRIAL_TOTAL_GAME_STEPS', 2000)
OPTUNA_TRIAL_TRAINING_UPDATES = getattr(config_mz, 'OPTUNA_TRIAL_TRAINING_UPDATES', 500)
OPTUNA_TRIAL_TRAIN_UPDATE_FREQ = getattr(config_mz, 'OPTUNA_TRIAL_TRAIN_UPDATE_FREQ', 1)
OPTUNA_TRIAL_GAME_STEPS_BEFORE_TRAIN = getattr(config_mz, 'OPTUNA_TRIAL_GAME_STEPS_BEFORE_TRAIN', 500)
OPTUNA_TRIAL_POLICY_LOSS_AVG_STEPS = getattr(config_mz, 'OPTUNA_TRIAL_POLICY_LOSS_AVG_STEPS', 100)
OPTUNA_TRIAL_EPISODE_LENGTH = getattr(config_mz, 'OPTUNA_TRIAL_EPISODE_LENGTH', 63)
OPTUNA_TRIAL_REPLAY_BUFFER_SIZE = getattr(config_mz, 'OPTUNA_TRIAL_REPLAY_BUFFER_SIZE', 2000)
OPTUNA_TRIAL_MIN_BUFFER_FOR_TRAINING_FACTOR = getattr(config_mz, 'OPTUNA_TRIAL_MIN_BUFFER_FOR_TRAINING_FACTOR', 5)

muzero_logger.info(f"Optuna Config: N_TRIALS={OPTUNA_N_TRIALS}, TRIAL_GAME_STEPS={OPTUNA_TRIAL_TOTAL_GAME_STEPS}, TRIAL_TRAINING_UPDATES={OPTUNA_TRIAL_TRAINING_UPDATES}")

# ==============================================================================
# PRELUDE: Load Feature Lists and Masks (Assumed correct from original script)
# ==============================================================================
muzero_logger.info("--- MONOLITH PRELUDE: Loading Training Features List and GA Mask ---") #
TRAINED_FEATURE_NAMES_GLOBAL = None #
ORIGINAL_MODEL_FEATURE_COUNT = 0 #
if not hasattr(config_mz, 'FEATURE_LIST_PATH') or not os.path.exists(config_mz.FEATURE_LIST_PATH):
    raise FileNotFoundError(f"Training features file path missing/invalid in config: {getattr(config_mz, 'FEATURE_LIST_PATH', 'N/A')}") #
try:
    with open(config_mz.FEATURE_LIST_PATH, 'r') as f: TRAINED_FEATURE_NAMES_GLOBAL = json.load(f) #
    muzero_logger.info(f"Loaded {len(TRAINED_FEATURE_NAMES_GLOBAL)} feature names from {config_mz.FEATURE_LIST_PATH}.") #
    ORIGINAL_MODEL_FEATURE_COUNT = len(TRAINED_FEATURE_NAMES_GLOBAL) #
except Exception as e: muzero_logger.critical(f"Error loading training features: {e}"); raise #
feature_mask_global = load_ga_feature_mask(config_mz.GA_BEST_MASK_PATH, muzero_logger) #
if feature_mask_global is None: raise FileNotFoundError(f"GA Feature Mask not found: {config_mz.GA_BEST_MASK_PATH}") #
if len(feature_mask_global) != ORIGINAL_MODEL_FEATURE_COUNT: raise ValueError("GA mask length mismatch with training features.") #
NUM_EFFECTIVE_MASKED_FEATURES = np.sum(feature_mask_global) #
muzero_logger.info(f"GA Mask loaded. Length: {len(feature_mask_global)}. Effective features: {NUM_EFFECTIVE_MASKED_FEATURES}") #
NUM_PORTFOLIO_STATES = config_mz.PORTFOLIO_STATE_SIZE #
NUM_ACTIONS = config_mz.NUM_ACTIONS #
MUZERO_HIDDEN_STATE_SIZE = config_mz.MUZERO_HIDDEN_STATE_SIZE #
muzero_logger.info("--- MONOLITH PRELUDE Complete ---") #

essential_globals_after_prelude = ['TRAINED_FEATURE_NAMES_GLOBAL', 'feature_mask_global', 'ORIGINAL_MODEL_FEATURE_COUNT'] #
for var_name in essential_globals_after_prelude:
    if var_name not in globals() or globals()[var_name] is None:
        raise NameError(f"Essential Prelude component '{var_name}' missing.")
muzero_logger.info("Prelude global variable checks passed.") #

# ==============================================================================
# STEP 8: FinancialMuZeroEnv (Assumed to be correct as per original script)
# ==============================================================================
muzero_logger.info("\n--- Defining Step 8: FinancialMuZeroEnv ---") #
class FinancialMuZeroEnv: #
    def __init__(self, config_mz_instance, logger_instance,
                 trained_feature_names_list, global_ga_mask_arr,
                 start_index=0, episode_length=None): #
        self.config = config_mz_instance; self.logger = logger_instance #
        self.trained_feature_names = trained_feature_names_list #
        self.global_ga_mask = global_ga_mask_arr #
        self.original_model_feature_count = ORIGINAL_MODEL_FEATURE_COUNT #
        self.start_index = start_index; self.episode_length = episode_length # Instance default episode length
        self.POSITION_SPXL, self.POSITION_SPXS, self.POSITION_CASH = 0, 1, 2 #
        self.portfolio_state_names = {self.POSITION_SPXL: "SPXL", self.POSITION_SPXS: "SPXS", self.POSITION_CASH: "CASH"} #
        self.num_portfolio_states = len(self.portfolio_state_names) #
        self.ACTION_TARGET_SPXL, self.ACTION_TARGET_SPXS, self.ACTION_TARGET_CASH = 0, 1, 2 #
        self.num_actions = self.config.NUM_ACTIONS #
        self.transaction_cost_bps = getattr(self.config, 'TRANSACTION_COST_BPS', 10) #
        self.leverage = getattr(self.config, 'LEVERAGE', 3.0) #
        self._load_and_prepare_data() #
        self.current_step_in_episode = 0; self.current_portfolio_position = self.POSITION_CASH #
        self.current_data_idx = self.start_index; self.done = False #
        self.episode_start_data_idx = self.start_index #
        # self.current_episode_run_length is NOT set here, it's set in reset()
        self.logger.info(f"FinancialMuZeroEnv initialized. Usable steps: {self.total_data_steps}. Instance default ep length: {self.episode_length if self.episode_length else 'End of Data'}.") #

    def _rename_etf_columns(self, df): #
        new_cols = {c: c.replace("SPXS_('Open', 'SPXS')", 'SPXS_Open').replace("SPXS_('Close', 'SPXS')", 'SPXS_Close')
                         .replace("SPXL_('Open', 'SPXL')", 'SPXL_Open').replace("SPXL_('Close', 'SPXL')", 'SPXL_Close')
                         .replace("SPXS_('High', 'SPXS')", 'SPXS_High').replace("SPXS_('Low', 'SPXS')", 'SPXS_Low')
                         .replace("SPXS_('Volume', 'SPXS')", 'SPXS_Volume')
                         .replace("SPXL_('High', 'SPXL')", 'SPXL_High').replace("SPXL_('Low', 'SPXL')", 'SPXL_Low')
                         .replace("SPXL_('Volume', 'SPXL')", 'SPXL_Volume')
                      for c in df.columns} #
        df.rename(columns=new_cols, inplace=True); return df #

    def _load_and_prepare_data(self): #
            date_col = self.config.existing_config_module.DATE_COLUMN #
            data_csv = pd.read_csv(self.config.MARKET_DATA_PATH, index_col=date_col, parse_dates=True) #
            etf_raw = pd.read_csv(self.config.ETF_DATA_PATH, index_col=date_col, parse_dates=True) #
            etf_proc = self._rename_etf_columns(etf_raw.copy()) #
            req_etf = ['SPXL_Close', 'SPXS_Close', 'SPXL_Open', 'SPXS_Open'] #
            if not all(c in etf_proc.columns for c in req_etf): raise ValueError(f"ETF data missing required columns. Found: {etf_proc.columns.tolist()}") #

            aligned_features = pd.DataFrame(index=data_csv.index, columns=self.trained_feature_names) #
            cols_to_fill = [c for c in self.trained_feature_names if c in data_csv.columns] #
            if cols_to_fill: aligned_features[cols_to_fill] = data_csv[cols_to_fill] #
            missing = set(self.trained_feature_names) - set(cols_to_fill) #
            if missing:
                self.logger.warning(f"{len(missing)} features expected by model were MISSING in data.csv and filled with 0s: {list(missing)[:5]}...") #
                aligned_features[list(missing)] = 0.0 #

            common_idx = aligned_features.index.intersection(etf_proc.index) #
            market_features_all_aligned = aligned_features.loc[common_idx].sort_index() #
            etf_prices_all_aligned = etf_proc.loc[common_idx].sort_index() #

            if market_features_all_aligned.empty: raise ValueError("Data empty after alignment.") #

            num_oos_days = 252
            if len(market_features_all_aligned) > num_oos_days:
                self.logger.info(f"Holding out the last {num_oos_days} days for out-of-sample testing.")
                self.market_features_df_raw_aligned = market_features_all_aligned.iloc[:-num_oos_days]
                self.etf_prices_df = etf_prices_all_aligned.iloc[:-num_oos_days]
                self.logger.info(f"Training data will use {len(self.market_features_df_raw_aligned)} days.")
            else:
                self.logger.warning(f"Total available data ({len(market_features_all_aligned)} days) is not enough to hold out {num_oos_days} days. Using all available data for training.")
                self.market_features_df_raw_aligned = market_features_all_aligned
                self.etf_prices_df = etf_prices_all_aligned
            if self.market_features_df_raw_aligned.empty: raise ValueError("Training data became empty after attempting to hold out OOS data.") #

            self.market_features_df = self.market_features_df_raw_aligned.copy() #
            if len(self.global_ga_mask) != self.market_features_df.shape[1]:
                raise ValueError(f"GA Mask length ({len(self.global_ga_mask)}) != Aligned Market Features cols ({self.market_features_df.shape[1]})") #
            self.market_features_df.loc[:, ~self.global_ga_mask] = 0.0 #
            self.total_data_steps = len(self.market_features_df) #
            self.logger.info(f"Data prep done for training. Market features (masked): {self.market_features_df.shape}. Effective non-zero: {np.sum(self.global_ga_mask)}. Total usable steps for this env instance: {self.total_data_steps}") #

    def _get_observation(self): #
        market_obs = np.zeros(self.original_model_feature_count, dtype=np.float32) #
        if self.current_data_idx < self.total_data_steps:
            market_obs = self.market_features_df.iloc[self.current_data_idx].values.astype(np.float32) #
        portfolio_obs = np.zeros(self.num_portfolio_states, dtype=np.float32); portfolio_obs[self.current_portfolio_position] = 1.0 #
        return [market_obs, portfolio_obs] #

    def reset(self, start_index=None, episode_length=None): #
        self.current_step_in_episode = 0; self.done = False; self.current_portfolio_position = self.POSITION_CASH #
        self.episode_start_data_idx = start_index if start_index is not None else self.start_index #
        self.current_data_idx = self.episode_start_data_idx #

        # Determine the actual length this episode will run for
        default_ep_len_on_reset = self.episode_length if self.episode_length is not None else (self.total_data_steps - self.current_data_idx)
        # Use episode_length if provided, else use instance default, else remaining steps
        self.current_episode_run_length = episode_length if episode_length is not None else default_ep_len_on_reset
        # Ensure it doesn't exceed available data from current_data_idx
        self.current_episode_run_length = max(0, min(self.current_episode_run_length, self.total_data_steps - self.current_data_idx))


        if self.current_data_idx >= self.total_data_steps:
            self.logger.warning(f"Reset start index {self.current_data_idx} is out of bounds. Setting done.") #
            self.done = True; # current_episode_run_length might be 0 here, which is fine.
            return self._get_observation() #
        self.logger.debug(f"Env reset. Start: {self.current_data_idx}. Pos: {self.portfolio_state_names[self.current_portfolio_position]}. MaxLenForThisRun: {self.current_episode_run_length}.") #
        return self._get_observation() #

    def step(self, action): #
        if self.done: return self._get_observation(), 0.0, True, {'status': 'Episode ended'} #
        if self.current_data_idx + 1 >= self.total_data_steps: # Need data for next day's prices for reward
            self.logger.warning(f"Env step: End of data for ETF prices at index {self.current_data_idx + 1} (current_data_idx: {self.current_data_idx}).") #
            self.done = True; return self._get_observation(), 0.0, True, {'status': 'End of data for reward'} #

        next_day_prices = self.etf_prices_df.iloc[self.current_data_idx + 1] #
        spxl_open, spxl_close = next_day_prices.get('SPXL_Open', np.nan), next_day_prices.get('SPXL_Close', np.nan) #
        spxs_open, spxs_close = next_day_prices.get('SPXS_Open', np.nan), next_day_prices.get('SPXS_Close', np.nan) #

        raw_reward, prev_pos, transacted = 0.0, self.current_portfolio_position, False #
        target_pos = action if action in range(self.num_actions) else self.POSITION_CASH #

        if prev_pos != target_pos:
            transacted = True #
            if prev_pos != self.POSITION_CASH: raw_reward -= (self.transaction_cost_bps / 10000.0) #
            if target_pos != self.POSITION_CASH: raw_reward -= (self.transaction_cost_bps / 10000.0) #

        pnl = 0.0 #
        if target_pos == self.POSITION_SPXL:
            if not pd.isna(spxl_open) and spxl_open != 0 and not pd.isna(spxl_close): pnl = self.leverage * (spxl_close - spxl_open) / spxl_open #
            elif pd.isna(spxl_open) or pd.isna(spxl_close): self.logger.warning(f"SPXL price NaN at data_idx {self.current_data_idx+1}") #
        elif target_pos == self.POSITION_SPXS:
            if not pd.isna(spxs_open) and spxs_open != 0 and not pd.isna(spxs_close): pnl = self.leverage * (spxs_close - spxs_open) / spxs_open #
            elif pd.isna(spxs_open) or pd.isna(spxs_close): self.logger.warning(f"SPXS price NaN at data_idx {self.current_data_idx+1}") #

        if pd.isna(pnl): pnl = 0.0 #

        raw_reward += pnl; self.current_portfolio_position = target_pos #
        self.current_step_in_episode += 1; self.current_data_idx += 1 #

        # Check if episode is done based on its run length or end of all data
        if (self.current_episode_run_length > 0 and self.current_step_in_episode >= self.current_episode_run_length) or \
           self.current_data_idx >= self.total_data_steps: # Done if run length is met OR absolutely no more data
            self.done = True #

        next_obs = self._get_observation() #
        info = {'ts': self.market_features_df.index[self.current_data_idx-1] if self.current_data_idx > 0 and self.current_data_idx <= len(self.market_features_df) else "N/A",
                'act':action, 'tgt_pos': self.portfolio_state_names.get(target_pos), 'raw_reward': raw_reward} #
        return next_obs, raw_reward, self.done, info #
    def get_observation_space_shape(self): return [(self.original_model_feature_count,), (self.num_portfolio_states,)] #
    def get_action_space_size(self): return self.num_actions #
muzero_logger.info("--- Step 8: FinancialMuZeroEnv Class Defined ---") #

# ==============================================================================
# STEP 9: MCTS (Corrected Unpacking)
# ==============================================================================
muzero_logger.info("\n--- Step 9: Defining Monte Carlo Tree Search (MCTS) (Corrected Unpacking) ---") #
class MCTSNode: #
    def __init__(self, parent=None, action_that_led_here=None, prior_p=0.0, hidden_state=None): #
        self.parent, self.action_that_led_here, self.prior_p, self.hidden_state = parent, action_that_led_here, prior_p, hidden_state #
        self.children, self.visit_count, self.value_sum, self.reward_from_parent_action = {}, 0, 0.0, 0.0 #
        self.policy_p_values_from_f_net, self.value_v_from_f_net = None, None #
    def get_mean_value(self): return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0 #
    def is_expanded(self): return self.policy_p_values_from_f_net is not None #
    def select_child_with_ucb(self, config): #
        best_score, best_action, best_child_node = -float('inf'), -1, None #
        if not self.is_expanded(): muzero_logger.error("UCB on non-expanded node!"); return np.random.choice(config.NUM_ACTIONS), None #
        for action_idx in range(config.NUM_ACTIONS): #
            q_val, visits = 0.0, 0 #
            if action_idx in self.children: child = self.children[action_idx]; q_val = child.reward_from_parent_action + config.DISCOUNT_FACTOR * child.get_mean_value(); visits = child.visit_count #
            prior = self.policy_p_values_from_f_net[action_idx] #
            parent_visit_count_for_ucb = self.visit_count if self.visit_count > 0 else 1 #
            ucb_exploration_component = prior * (config.MCTS_PB_C_BASE + math.log((parent_visit_count_for_ucb + config.MCTS_PB_C_BASE + 1) / config.MCTS_PB_C_BASE)) * (math.sqrt(parent_visit_count_for_ucb) / (1 + visits)) #
            ucb = q_val + ucb_exploration_component #
            if ucb > best_score: best_score, best_action, best_child_node = ucb, action_idx, self.children.get(action_idx) #
        if best_action == -1 and config.NUM_ACTIONS > 0: best_action = np.random.choice(config.NUM_ACTIONS); best_child_node = self.children.get(best_action) #
        return best_action, best_child_node #

def run_mcts(initial_observation, h_net_func, g_net_func, f_net_func, config_obj, num_simulations_mcts, num_actions_mcts): #
    obs_np = [np.array(obs, dtype=np.float32)[np.newaxis,...] for obs in initial_observation] #
    root_s = h_net_func.predict_on_batch(obs_np) #
    root = MCTSNode(hidden_state=root_s) #

    policy_logits_at_root, value_tanh_at_root, _ = f_net_func.predict_on_batch(root.hidden_state) #
    probs_r = tf.nn.softmax(policy_logits_at_root[0]).numpy() #
    root.value_v_from_f_net = value_tanh_at_root[0,0] #
    root.policy_p_values_from_f_net = probs_r #

    if config_obj.MCTS_ROOT_DIRICHLET_ALPHA > 0: root.policy_p_values_from_f_net = (1-config_obj.MCTS_ROOT_EXPLORATION_FRACTION)*root.policy_p_values_from_f_net + config_obj.MCTS_ROOT_EXPLORATION_FRACTION*np.random.dirichlet([config_obj.MCTS_ROOT_DIRICHLET_ALPHA]*num_actions_mcts) #
    for _ in range(num_simulations_mcts): #
        node, path, action_sel = root, [root], -1 #
        while node.is_expanded(): #
            action_sel, child_cand = node.select_child_with_ucb(config_obj) #
            if child_cand is not None: node = child_cand; path.append(node) #
            else: break #
        parent_exp, action_exp = node, action_sel #
        val_backup = 0.0 #
        if parent_exp.children.get(action_exp) is None : #
            action_oh = np.zeros(num_actions_mcts,dtype=np.float32); action_oh[action_exp] = 1.0 #
            next_s, reward_g_pred_tanh, _ = g_net_func.predict_on_batch( #
                [parent_exp.hidden_state, action_oh[np.newaxis,...]]
            ) #
            reward_g_pred_scalar = reward_g_pred_tanh[0,0] #
            policy_logits_new, value_tanh_new, _ = f_net_func.predict_on_batch(next_s) #
            new_node = MCTSNode(parent=parent_exp,action_that_led_here=action_exp,prior_p=parent_exp.policy_p_values_from_f_net[action_exp],hidden_state=next_s) #
            new_node.reward_from_parent_action=reward_g_pred_scalar #
            new_node.policy_p_values_from_f_net=tf.nn.softmax(policy_logits_new[0]).numpy(); #
            new_node.value_v_from_f_net=value_tanh_new[0,0] #
            parent_exp.children[action_exp]=new_node; path.append(new_node) #
            val_backup = new_node.value_v_from_f_net #
        else: val_backup = node.value_v_from_f_net if node.value_v_from_f_net is not None else 0.0 #

        for node_bp in reversed(path): #
            node_bp.visit_count+=1 #
            if node_bp.parent is not None: val_backup = node_bp.reward_from_parent_action + config_obj.DISCOUNT_FACTOR * val_backup #
            node_bp.value_sum += val_backup #

    visits = np.array([root.children[a].visit_count if a in root.children else 0 for a in range(num_actions_mcts)]) #
    action_probs = (visits/np.sum(visits)) if np.sum(visits)>0 else (np.ones(num_actions_mcts)/num_actions_mcts) #
    return action_probs, root.get_mean_value(), root #
muzero_logger.info("--- Step 9: MCTS Implementation Defined (Corrected Unpacking) ---") #

# ==============================================================================
# STEP 10: REPLAY BUFFER (Assumed to be correct as per original script)
# ==============================================================================
muzero_logger.info("\n--- Step 10: Defining Replay Buffer ---") #
class ReplayBuffer: #
    def __init__(self, config, logger): #
        self.config, self.logger, self.buffer_size = config, logger, config.REPLAY_BUFFER_SIZE #
        self.buffer = collections.deque(maxlen=self.buffer_size); self.num_games_added, self.num_samples_drawn = 0,0 #
        self.save_path = os.path.join(self.config.MUZERO_REPLAY_BUFFER_DIR, "replay_buffer.pkl") #
        self.logger.debug(f"ReplayBuffer init. Size: {self.buffer_size}. Path: {self.save_path}") #
    def add_game(self, game_trajectory): #
        if len(self.buffer) >= self.buffer_size and self.buffer_size > 0 : self.buffer.popleft() #
        self.buffer.append(game_trajectory); self.num_games_added += 1 #
        if self.num_games_added % 10 == 0: self.logger.debug(f"Game {self.num_games_added} added. Buffer: {len(self.buffer)}/{self.buffer_size}") #
    def sample_batch(self, batch_size): #
        if not self.buffer: self.logger.warning("Buffer empty, cannot sample."); return None #
        actual_batch_size = min(batch_size, len(self.buffer)) #
        unroll_K = self.config.NUM_UNROLL_STEPS #
        indices = np.random.choice(len(self.buffer), size=actual_batch_size, replace=len(self.buffer) < batch_size) #
        samples = [] #
        for i in indices: #
            game = self.buffer[i]; game_len_actions = len(game['actions_list']) #
            if game_len_actions < 1: continue #
            max_start_idx_for_obs = len(game['observations_list']) -1 #
            if max_start_idx_for_obs < 0 : continue #
            start_idx = np.random.randint(0, max_start_idx_for_obs + 1) #
            obs_init = game['observations_list'][start_idx] #
            actions, rewards_target, policies_target, values_target = [], [], [], [] #
            policies_target.append(game['mcts_policy_targets_list'][start_idx]) #
            values_target.append(game['value_targets_list'][start_idx]) #
            for k_idx in range(unroll_K): #
                action_step_idx = start_idx + k_idx #
                next_state_target_idx = start_idx + k_idx + 1 #
                if action_step_idx < game_len_actions: #
                    actions.append(game['actions_list'][action_step_idx]) #
                    rewards_target.append(game['rewards_list'][action_step_idx]) #
                    if next_state_target_idx < len(game['observations_list']): #
                        policies_target.append(game['mcts_policy_targets_list'][next_state_target_idx]) #
                        values_target.append(game['value_targets_list'][next_state_target_idx]) #
                    else: policies_target.append(np.ones(self.config.NUM_ACTIONS)/self.config.NUM_ACTIONS); values_target.append(0.0) #
                else: #
                    actions.append(np.random.randint(0,self.config.NUM_ACTIONS)); rewards_target.append(0.0) #
                    policies_target.append(np.ones(self.config.NUM_ACTIONS)/self.config.NUM_ACTIONS); values_target.append(0.0) #
            samples.append({'initial_observation':obs_init, 'action_history':actions,
                            'target_rewards':rewards_target,
                            'target_policies':policies_target,
                            'target_values':values_target}) #
        return samples if samples else None #
    def __len__(self): return len(self.buffer) #
    def save(self):
        try:
            os.makedirs(os.path.dirname(self.save_path), exist_ok=True) #
            with open(self.save_path,'wb') as f: pickle.dump(list(self.buffer),f)
            self.logger.info(f"Replay buffer saved to {self.save_path}. Size: {len(self.buffer)} games.")
        except Exception as e: self.logger.error(f"Error saving replay buffer: {e}", exc_info=True)
    def load(self):
        if os.path.exists(self.save_path):
            try:
                with open(self.save_path,'rb') as f:
                    loaded_list = pickle.load(f)
                    self.buffer=collections.deque(loaded_list, maxlen=self.buffer_size)
                    self.num_games_added=len(self.buffer)
                self.logger.info(f"Replay buffer loaded from {self.save_path}. Size: {len(self.buffer)} games.")
            except Exception as e: self.logger.error(f"Error loading replay buffer: {e}. Starting empty.", exc_info=True); self.buffer=collections.deque(maxlen=self.buffer_size)
        else: self.logger.info(f"No replay buffer file found at {self.save_path}. Starting with an empty buffer.")
muzero_logger.info("--- Step 10: ReplayBuffer Class Defined ---") #

# ==============================================================================
# STEP 11: LOSS FUNCTIONS (Assumed to be correct as per original script)
# ==============================================================================
muzero_logger.info("\n--- Step 11: Defining Loss Functions ---") #
mse_loss_fn = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE) #
cross_entropy_loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE) #
def calculate_muzero_losses(pred_vals, target_vals, pred_rewards, target_rewards, pred_policies, target_policies, nets_for_l2, cfg): #
    v_loss, r_loss, p_loss = tf.constant(0.0), tf.constant(0.0), tf.constant(0.0) #
    for k in range(cfg.NUM_UNROLL_STEPS + 1): v_loss += mse_loss_fn(target_vals[:,k,:], pred_vals[k]) #
    for k in range(cfg.NUM_UNROLL_STEPS): r_loss += mse_loss_fn(target_rewards[:,k,:], pred_rewards[k]) #
    for k in range(cfg.NUM_UNROLL_STEPS + 1): p_loss += cross_entropy_loss_fn(target_policies[:,k,:], pred_policies[k]) #
    l2_loss = tf.constant(0.0) #
    if cfg.WEIGHT_DECAY > 0: #
        for net_item in nets_for_l2: #
            if hasattr(net_item, 'trainable_variables'): #
                for var in net_item.trainable_variables: #
                    if 'bias' not in var.name and 'normalization' not in var.name and 'gamma' not in var.name and 'beta' not in var.name : l2_loss += tf.nn.l2_loss(var) #
    l2_loss *= cfg.WEIGHT_DECAY #
    avg_v = v_loss / tf.cast(cfg.NUM_UNROLL_STEPS+1,tf.float32); avg_r = r_loss / tf.cast(tf.maximum(1,cfg.NUM_UNROLL_STEPS),tf.float32); avg_p = p_loss / tf.cast(cfg.NUM_UNROLL_STEPS+1,tf.float32) #
    w_v, w_r, w_p = avg_v*cfg.VALUE_LOSS_WEIGHT, avg_r*cfg.REWARD_LOSS_WEIGHT, avg_p*cfg.POLICY_LOSS_WEIGHT #
    total = w_v + w_r + w_p + l2_loss #
    return total, {"total_loss":total, "value_loss":avg_v, "reward_loss":avg_r, "policy_loss":avg_p, "l2_reg_loss":l2_loss} #
muzero_logger.info("--- Step 11: Loss Functions Defined ---") #

# ==============================================================================
# STEP 12: NETWORK DEFINITION & TRAINING LOOP
# ==============================================================================
muzero_logger.info("\n--- Step 12: Defining Networks and Training Loop ---") #

TF_PRINT_INTERVAL = tf.constant(10, dtype=tf.int64) # Not used in Optuna context's train step

def get_muzero_agent_networks(config, num_original_features, num_portfolio_states, num_actions, logger,
                              freeze_representation_network=False,
                              policy_head_hyperparams=None):
    logger.info(f"Instantiating MuZero networks. Base NN: {config.FINAL_BEST_NN_PATH}, Transfer Layer: {config.TRANSFER_LAYER_NAME}")
    if not os.path.exists(config.FINAL_BEST_NN_PATH): raise FileNotFoundError(f"Base NN {config.FINAL_BEST_NN_PATH} not found.")

    loaded_nn = keras.models.load_model(config.FINAL_BEST_NN_PATH, compile=False)
    try:
        base_model_output = loaded_nn.get_layer(config.TRANSFER_LAYER_NAME).output
        base_model = keras.Model(inputs=loaded_nn.input, outputs=base_model_output, name="BaseRepresentation")
    except ValueError as e:
        logger.critical(f"Layer '{config.TRANSFER_LAYER_NAME}' not in loaded NN."); loaded_nn.summary(print_fn=logger.error); raise e

    if freeze_representation_network:
        base_model.trainable = False
        logger.info(f"Optuna/Frozen context: base_model (GA NN) explicitly set to trainable=False.")
    else:
        base_model.trainable = not config.FREEZE_TRANSFER_WEIGHTS_INITIALLY
        logger.info(f"Base representation model (GA NN) trainable: {base_model.trainable} (initial freeze: {config.FREEZE_TRANSFER_WEIGHTS_INITIALLY})")

    m_in = keras.layers.Input(shape=(num_original_features,), name='h_market_in')
    p_in = keras.layers.Input(shape=(num_portfolio_states,), name='h_portfolio_in')
    enc_m = base_model(m_in)
    concat_h_layer = keras.layers.Concatenate()([enc_m, p_in])
    s_out_h_layer = keras.layers.Dense(config.MUZERO_HIDDEN_STATE_SIZE, activation='relu', name='h_final_dense')(concat_h_layer)
    h_net = keras.Model(inputs=[m_in, p_in], outputs=s_out_h_layer, name="Representation_h")

    if freeze_representation_network:
        h_net.trainable = False
        logger.info("Optuna/Frozen context: Full h_net (including its own Dense layers) set to trainable=False.")
    else:
        logger.info(f"Normal context: h_net layers (excluding base_model) are trainable. Base_model trainable: {base_model.trainable}")

    s_g_in = keras.layers.Input(shape=(config.MUZERO_HIDDEN_STATE_SIZE,), name='g_s_in')
    a_g_in = keras.layers.Input(shape=(num_actions,), name='g_a_in')
    concat_g_layer = keras.layers.Concatenate(name='g_concat_s_a')([s_g_in, a_g_in])
    x_dyn_s = keras.layers.Dense(config.DYNAMICS_INTERNAL_SIZE, name='g_dynamics_hidden_dense_linear')(concat_g_layer)
    x_dyn_s = keras.layers.BatchNormalization(name='g_dynamics_bn')(x_dyn_s)
    g_h1_processed_s = keras.layers.Activation('relu', name='g_dynamics_relu')(x_dyn_s)
    s_g_out_layer = keras.layers.Dense(config.MUZERO_HIDDEN_STATE_SIZE, activation='relu', name='g_s_out')(g_h1_processed_s)
    x_dyn_r = keras.layers.Dense(config.DYNAMICS_INTERNAL_SIZE_REWARD, name='g_reward_hidden_dense_linear')(concat_g_layer)
    x_dyn_r = keras.layers.BatchNormalization(name='g_reward_bn')(x_dyn_r)
    g_h1_processed_r = keras.layers.Activation('relu', name='g_reward_relu')(x_dyn_r)
    r_g_logits_layer = keras.layers.Dense(1, name='g_r_logits')(g_h1_processed_r)
    r_g_out_layer = keras.layers.Activation('tanh', name='g_r_out')(r_g_logits_layer)
    g_net = keras.Model(inputs=[s_g_in, a_g_in], outputs=[s_g_out_layer, r_g_out_layer, r_g_logits_layer], name="Dynamics_g_BN")

    s_f_in = keras.layers.Input(shape=(config.MUZERO_HIDDEN_STATE_SIZE,), name='f_s_in')

    if policy_head_hyperparams:
        logger.info(f"Using NEW Optuna-tuned policy head with HPs: {policy_head_hyperparams}")
        x_p = keras.layers.Dense(12, name='f_policy_pseudo_enc_dense')(s_f_in)
        x_p = keras.layers.BatchNormalization(name='f_policy_pseudo_enc_bn')(x_p)
        x_p = keras.layers.Activation('relu', name='f_policy_pseudo_enc_relu')(x_p)
        x_p = keras.layers.GaussianNoise(policy_head_hyperparams['stddev1'], name='f_policy_gauss_noise1')(x_p)
        x_p = keras.layers.Dense(256, name='f_policy_exp_dense')(x_p)
        x_p = keras.layers.BatchNormalization(name='f_policy_exp_bn')(x_p)
        x_p = keras.layers.Activation('relu', name='f_policy_exp_relu')(x_p)
        x_p = keras.layers.GaussianNoise(policy_head_hyperparams['stddev2'], name='f_policy_gauss_noise2')(x_p)
        x_p = keras.layers.Dropout(policy_head_hyperparams['dropout_rate'], name='f_policy_dropout')(x_p)
        p_f_out_layer = keras.layers.Dense(num_actions, activation='linear', name='f_p_logits')(x_p)
    else:
        logger.info("Using ORIGINAL policy head design.")
        x_pred_p = keras.layers.Dense(config.PREDICTION_INTERNAL_SIZE_POLICY, name='f_policy_hidden_dense_linear')(s_f_in)
        x_pred_p = keras.layers.BatchNormalization(name='f_policy_bn')(x_pred_p)
        f_p_h1_processed = keras.layers.Activation('relu', name='f_policy_relu')(x_pred_p)
        p_f_out_layer = keras.layers.Dense(num_actions, name='f_p_logits')(f_p_h1_processed)

    x_pred_v = keras.layers.Dense(config.PREDICTION_INTERNAL_SIZE_VALUE, name='f_value_hidden_dense_linear')(s_f_in)
    x_pred_v = keras.layers.BatchNormalization(name='f_value_bn')(x_pred_v)
    f_v_h1_processed = keras.layers.Activation('relu', name='f_value_relu')(x_pred_v)
    v_f_logits_layer = keras.layers.Dense(1, name='f_v_logits')(f_v_h1_processed)
    v_f_out_layer = keras.layers.Activation('tanh', name='f_v_out')(v_f_logits_layer)

    f_net = keras.Model(inputs=s_f_in, outputs=[p_f_out_layer, v_f_out_layer, v_f_logits_layer], name="Prediction_f_BN_OptunaPolicy")

    logger.info("MuZero networks created.")
    if freeze_representation_network:
        logger.info(f"TRAINABLE STATUS for Optuna/Frozen: h_net: {h_net.trainable}, base_model_in_h: {h_net.get_layer('BaseRepresentation').trainable}, g_net: {g_net.trainable}, f_net: {f_net.trainable}")
    return h_net, g_net, f_net, base_model


def train_muzero_agent(config: type, logger: logging.Logger,
                       optuna_trial_obj=None,
                       policy_hps_override=None):
    script_version = getattr(config, 'SCRIPT_VERSION', 'Optuna_PolicyHead_Fixed')
    logger.info(f"===== STARTING MUZERO AGENT (Version: {script_version}) =====")
    if optuna_trial_obj:
        logger.info(f"Running in Optuna context. Trial Number: {optuna_trial_obj.number}")

    np.random.seed(config.SEED); tf.random.set_seed(config.SEED); random.seed(config.SEED)

    freeze_h_net_for_this_run = True if optuna_trial_obj else getattr(config, "FORCE_FREEZE_H_NET_FINAL_RUN", False)
    current_policy_hps = policy_hps_override

    logger.info("Initializing components...")
    environment = FinancialMuZeroEnv(
        config_mz_instance=config, logger_instance=logger,
        trained_feature_names_list=TRAINED_FEATURE_NAMES_GLOBAL,
        global_ga_mask_arr=feature_mask_global,
        start_index=0,
        episode_length=OPTUNA_TRIAL_EPISODE_LENGTH if optuna_trial_obj else getattr(config, 'INITIAL_EPISODE_LENGTH', 252)
    )
    h_net_train, g_net_train, f_net_train, base_repr_train = get_muzero_agent_networks(
        config, ORIGINAL_MODEL_FEATURE_COUNT, config.PORTFOLIO_STATE_SIZE, config.NUM_ACTIONS, logger,
        freeze_representation_network=freeze_h_net_for_this_run,
        policy_head_hyperparams=current_policy_hps
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=config.LEARNING_RATE,
        clipnorm=getattr(config, 'GRADIENT_CLIP_NORM', 1.0)
    )
    replay_buffer = ReplayBuffer(config, logger)

    start_step = 1
    if not optuna_trial_obj and getattr(config, 'ATTEMPT_LOAD_CHECKPOINT_FINAL_RUN', False):
        logger.info("Checkpoint loading logic for final run would be here (currently skipped).")
    elif optuna_trial_obj:
        logger.info("Optuna trial: Starting from scratch, no checkpoint loading.")
    else:
        logger.info("Starting from scratch (no checkpoint loading specified for this run).")

    @tf.function
    def _perform_train_step_tf(batch_tf_local, current_train_step_tf_local):
        with tf.GradientTape() as tape:
            vars_to_train_local = []
            if h_net_train.trainable:
                for layer in h_net_train.layers:
                    if layer.name != 'BaseRepresentation':
                        vars_to_train_local.extend(layer.trainable_variables)
                if base_repr_train.trainable:
                    vars_to_train_local.extend(base_repr_train.trainable_variables)

            vars_to_train_local.extend(g_net_train.trainable_variables)
            vars_to_train_local.extend(f_net_train.trainable_variables)

            if not vars_to_train_local:
                dummy_loss_val = tf.constant(0.0, dtype=tf.float32)
                dummy_metrics = {"total_loss":dummy_loss_val, "value_loss":dummy_loss_val,
                                 "reward_loss":dummy_loss_val, "policy_loss":dummy_loss_val, "l2_reg_loss":dummy_loss_val}
                return dummy_loss_val, dummy_metrics

            s_k = h_net_train([batch_tf_local['initial_market_features'], batch_tf_local['initial_portfolio_states']], training=h_net_train.trainable)

            preds_p, preds_v_tanh, preds_r_tanh = [], [], []
            # preds_v_logits, preds_r_logits = [], [] # Kept for potential future debug, not used now

            policy_s0_logits, value_s0_tanh, _ = f_net_train(s_k, training=True)
            preds_p.append(policy_s0_logits)
            preds_v_tanh.append(value_s0_tanh)
            # preds_v_logits.append(value_s0_logits)

            for k_step in range(config.NUM_UNROLL_STEPS):
                actions_g = tf.one_hot(batch_tf_local['action_history'][:, k_step], depth=config.NUM_ACTIONS)
                s_k_plus_1, rew_k_tanh, _ = g_net_train([s_k, actions_g], training=True)
                preds_r_tanh.append(rew_k_tanh)
                # preds_r_logits.append(rew_k_logits)
                s_k = s_k_plus_1
                policy_sk_plus_1_logits, value_sk_plus_1_tanh, _ = f_net_train(s_k, training=True)
                preds_p.append(policy_sk_plus_1_logits)
                preds_v_tanh.append(value_sk_plus_1_tanh)
                # preds_v_logits.append(value_sk_plus_1_logits)

            nets_for_l2_calc = []
            # Only add base_repr_train to L2 if it's actually part of h_net and h_net is trainable
            if h_net_train.trainable:
                 if base_repr_train.trainable: # Check specific trainability of base_repr_train
                    nets_for_l2_calc.append(base_repr_train)
                 # Add other trainable layers of h_net if any (e.g., the final Dense)
                 # This needs careful handling if h_net.trainable=True but base_repr_train.trainable=False
                 # For simplicity, if h_net is trainable, consider its top-level structure.
                 # The current logic correctly includes base_repr_train's vars if base_repr_train.trainable.
                 # And includes other h_net layer vars if h_net.trainable.
                 # The L2 loss function iterates through net_item.trainable_variables.
                 nets_for_l2_calc.append(h_net_train) # Simpler: if h_net is trainable, its L2 is considered.

            nets_for_l2_calc.append(g_net_train)
            nets_for_l2_calc.append(f_net_train)

            loss_total, loss_comps = calculate_muzero_losses(
                pred_vals=preds_v_tanh, target_vals=batch_tf_local['target_values'],
                pred_rewards=preds_r_tanh, target_rewards=batch_tf_local['target_rewards'],
                pred_policies=preds_p, target_policies=batch_tf_local['target_policies'],
                nets_for_l2=[net for net in nets_for_l2_calc if net.trainable], # Ensure only trainable nets contribute to L2
                cfg=config
            )

            grads = tape.gradient(loss_total, vars_to_train_local)
            optimizer.apply_gradients(zip(grads, vars_to_train_local))
            return loss_total, loss_comps

    max_steps_for_this_run = OPTUNA_TRIAL_TOTAL_GAME_STEPS if optuna_trial_obj else config.MAX_TRAINING_STEPS
    training_updates_this_run = 0
    max_training_updates = OPTUNA_TRIAL_TRAINING_UPDATES if optuna_trial_obj else float('inf')

    policy_losses_collected = collections.deque(maxlen=OPTUNA_TRIAL_POLICY_LOSS_AVG_STEPS)

    for current_run_step in range(start_step, max_steps_for_this_run + 1):
        loop_start_time = time.time()

        if not freeze_h_net_for_this_run and config.FREEZE_TRANSFER_WEIGHTS_INITIALLY and \
           current_run_step >= config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP and not base_repr_train.trainable:
            base_repr_train.trainable = True
            logger.info(f"Step {current_run_step}: Unfroze base_repr_train (GA NN part of h_net).")

        # Determine the intended episode length for the upcoming reset
        intended_episode_length_for_reset = OPTUNA_TRIAL_EPISODE_LENGTH if optuna_trial_obj else getattr(config, 'DEFAULT_EPISODE_LENGTH_PLAY', 252)

        # Calculate max_possible_start_index using this intended length
        safe_intended_ep_len = max(1, intended_episode_length_for_reset) # Ensure at least 1
        max_possible_start_index = environment.total_data_steps - safe_intended_ep_len

        if max_possible_start_index < 0:
            max_possible_start_index = 0
            if environment.total_data_steps < safe_intended_ep_len: # Only log if it's truly shorter
                 logger.warning(f"Dataset shorter ({environment.total_data_steps} steps) than intended episode length ({safe_intended_ep_len}). Episode will be truncated by env.")

        current_start_idx_play = np.random.randint(0, max_possible_start_index + 1) if max_possible_start_index >= 0 else 0

        obs_loop = environment.reset(start_index=current_start_idx_play,
                                     episode_length=intended_episode_length_for_reset)

        # After reset, environment.current_episode_run_length is reliably set.
        actual_episode_run_length_from_env = environment.current_episode_run_length

        game_data = {'observations_list':[obs_loop],'actions_list':[],'rewards_list':[],
                     'mcts_policy_targets_list':[],'value_targets_list':[],
                     'raw_mcts_values_list': [], 'raw_rewards_list': [],
                     'done_list':[],'total_episode_reward_raw':0.0}
        done_loop, game_steps_count_in_episode = False, 0

        if optuna_trial_obj:
            current_temp = getattr(config, 'OPTUNA_FIXED_TEMPERATURE', 1.0)
        else:
            if training_updates_this_run < config.ANNEAL_TEMPERATURE_AFTER_N_MOVES :
                 current_temp = config.INITIAL_TEMPERATURE
            else:
                 current_temp = config.FINAL_TEMPERATURE

        while not done_loop and game_steps_count_in_episode < actual_episode_run_length_from_env :
            mcts_policy_target, mcts_val_raw, _ = run_mcts(
                obs_loop, h_net_train,g_net_train,f_net_train,config,config.NUM_SIMULATIONS,config.NUM_ACTIONS)
            mcts_val_scaled_target = np.tanh(mcts_val_raw / config.VALUE_TARGET_SCALING_DIVISOR)
            action_probs_for_sampling = mcts_policy_target
            if np.sum(action_probs_for_sampling) > 1e-6 and current_temp > 1e-3:
                action_probs_temp = np.power(action_probs_for_sampling + 1e-9, 1.0 / current_temp)
                action_probs_temp_sum = np.sum(action_probs_temp)
                if action_probs_temp_sum > 1e-6 : action_probs_temp /= action_probs_temp_sum
                else: action_probs_temp = np.ones_like(action_probs_temp) / len(action_probs_temp)
                if np.isnan(action_probs_temp).any(): action = np.argmax(action_probs_for_sampling)
                else: action = np.random.choice(config.NUM_ACTIONS, p=action_probs_temp)
            else: action = np.argmax(action_probs_for_sampling)

            if not optuna_trial_obj or (game_steps_count_in_episode % 50 == 0) :
                 logger.debug(f" S{current_run_step} T{training_updates_this_run} G{len(replay_buffer)+1}-P{game_steps_count_in_episode+1}: Pol={np.round(mcts_policy_target,2)}, RawVal={mcts_val_raw:.3f}, Act={action} (T={current_temp:.2f})")

            next_obs, raw_reward_from_env, done_loop, info = environment.step(action)
            scaled_reward_target_for_buffer = np.tanh(raw_reward_from_env / config.REWARD_TARGET_SCALING_DIVISOR)
            if not optuna_trial_obj or (game_steps_count_in_episode % 50 == 0) :
                logger.debug(f" Env Step: RawRew={raw_reward_from_env:.4f}, Done={done_loop}, TgtPos={info.get('tgt_pos')}")

            game_data['actions_list'].append(action); game_data['rewards_list'].append(scaled_reward_target_for_buffer)
            game_data['raw_rewards_list'].append(raw_reward_from_env); game_data['mcts_policy_targets_list'].append(mcts_policy_target);
            game_data['value_targets_list'].append(mcts_val_scaled_target); game_data['raw_mcts_values_list'].append(mcts_val_raw);
            game_data['done_list'].append(done_loop); game_data['total_episode_reward_raw'] += raw_reward_from_env
            obs_loop = next_obs
            if not done_loop: game_data['observations_list'].append(obs_loop)
            game_steps_count_in_episode += 1

        if not done_loop and len(game_data['observations_list']) > len(game_data['mcts_policy_targets_list']):
            last_obs = game_data['observations_list'][-1]
            last_s = h_net_train.predict_on_batch([np.array(o,dtype=np.float32)[np.newaxis,...] for o in last_obs])
            last_policy_logits, last_value_output_tanh, _ = f_net_train.predict_on_batch(last_s)
            game_data['mcts_policy_targets_list'].append(tf.nn.softmax(last_policy_logits[0]).numpy());
            game_data['value_targets_list'].append(last_value_output_tanh[0,0])
            game_data['raw_mcts_values_list'].append(last_value_output_tanh[0,0])

        replay_buffer.add_game(game_data)
        if not optuna_trial_obj or (len(replay_buffer) % 10 ==0 ):
             logger.debug(f"S{current_run_step} T{training_updates_this_run} GameStore: Game {len(replay_buffer)} added. EpRawRew={game_data['total_episode_reward_raw']:.2f}, Steps={game_steps_count_in_episode}")

        min_buffer_for_training_actual = config.BATCH_SIZE * OPTUNA_TRIAL_MIN_BUFFER_FOR_TRAINING_FACTOR if optuna_trial_obj else getattr(config, 'MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING', config.BATCH_SIZE * 2)

        if current_run_step >= (OPTUNA_TRIAL_GAME_STEPS_BEFORE_TRAIN if optuna_trial_obj else 0) and \
           len(replay_buffer) >= min_buffer_for_training_actual and \
           training_updates_this_run < max_training_updates:

            if current_run_step % (OPTUNA_TRIAL_TRAIN_UPDATE_FREQ if optuna_trial_obj else 1) == 0:
                raw_b = replay_buffer.sample_batch(config.BATCH_SIZE)
                if raw_b:
                    tf_b = {'initial_market_features': tf.constant(np.array([s['initial_observation'][0] for s in raw_b],dtype=np.float32)),
                            'initial_portfolio_states': tf.constant(np.array([s['initial_observation'][1] for s in raw_b],dtype=np.float32)),
                            'action_history': tf.constant(np.array([s['action_history'] for s in raw_b],dtype=np.int32)),
                            'target_rewards': tf.constant(np.array([s['target_rewards'] for s in raw_b],dtype=np.float32)[...,np.newaxis]),
                            'target_policies': tf.constant(np.array([s['target_policies'] for s in raw_b],dtype=np.float32)),
                            'target_values': tf.constant(np.array([s['target_values'] for s in raw_b],dtype=np.float32)[...,np.newaxis])}

                    loss_val, metrics = _perform_train_step_tf(tf_b, tf.cast(training_updates_this_run, dtype=tf.int64))
                    training_updates_this_run += 1

                    policy_losses_collected.append(metrics['policy_loss'].numpy())

                    if training_updates_this_run % 10 == 0 :
                        log_msg = f"S{current_run_step} Update {training_updates_this_run}: Loss={loss_val.numpy():.4f}, " + \
                                  f"P Loss={metrics['policy_loss'].numpy():.3f}, V Loss={metrics['value_loss'].numpy():.3f}, R Loss={metrics['reward_loss'].numpy():.3f}" + \
                                  f", L2={metrics['l2_reg_loss'].numpy():.5f}" # Corrected key to P Loss, V Loss, R Loss
                        logger.info(log_msg)

                    if optuna_trial_obj and training_updates_this_run > OPTUNA_TRIAL_POLICY_LOSS_AVG_STEPS // 2:
                        current_avg_policy_loss = np.mean(list(policy_losses_collected))
                        optuna_trial_obj.report(current_avg_policy_loss, training_updates_this_run)
                        if optuna_trial_obj.should_prune():
                            logger.info(f"Optuna Trial {optuna_trial_obj.number} pruned at update {training_updates_this_run} with avg P Loss {current_avg_policy_loss:.4f}.")
                            return 10.0
                else:
                     logger.debug(f"S{current_run_step} Update {training_updates_this_run}: Training skipped, batch from buffer was empty.")
        elif current_run_step % 10 == 0 and training_updates_this_run < max_training_updates:
             logger.info(f"S{current_run_step} Update {training_updates_this_run}: Buffer fill ({len(replay_buffer)}/{min_buffer_for_training_actual} games)...")

        if not optuna_trial_obj and current_run_step % config.CHECKPOINT_INTERVAL == 0 and current_run_step > 0:
            logger.info(f"S{current_run_step}: Checkpoint saving would occur here (currently skipped for brevity in Optuna focus).")
            os.makedirs(config.MUZERO_MODELS_DIR, exist_ok=True)
            pref = config.MUZERO_CHECKPOINT_FORMAT.format(step=current_run_step)
            h_path_save = pref+"_h.weights.h5"; g_path_save = pref+"_g.weights.h5"; f_path_save = pref+"_f.weights.h5"
            latest_step_file_path = os.path.join(config.MUZERO_MODELS_DIR, getattr(config, 'LATEST_CHECKPOINT_FILENAME', "latest_checkpoint_step.txt"))
            try:
                h_net_train.save_weights(h_path_save)
                g_net_train.save_weights(g_path_save)
                f_net_train.save_weights(f_path_save)
                replay_buffer.save()
                with open(latest_step_file_path, 'w') as f_sw: f_sw.write(str(current_run_step))
                logger.info(f"----> CHECKPOINT COMPLETELY SAVED for S{current_run_step} <----")
            except Exception as e_ckpt: logger.error(f"Error saving complete checkpoint at step {current_run_step}: {e_ckpt}", exc_info=True)

        if current_run_step % 100 == 0 and not optuna_trial_obj:
            loop_duration = time.time() - loop_start_time
            logger.info(f"S{current_run_step} LoopTime: {loop_duration:.2f}s.")

        if optuna_trial_obj and training_updates_this_run >= max_training_updates:
            logger.info(f"Optuna Trial {optuna_trial_obj.number} reached max training updates ({max_training_updates}). Ending trial early.")
            break

    if optuna_trial_obj:
        if not policy_losses_collected or len(policy_losses_collected) < OPTUNA_TRIAL_POLICY_LOSS_AVG_STEPS // 2:
            logger.warning(f"Optuna Trial {optuna_trial_obj.number}: Not enough policy loss samples ({len(policy_losses_collected)}) to evaluate. Returning high value.")
            return 10.0
        final_avg_policy_loss = np.mean(list(policy_losses_collected))
        logger.info(f"Optuna Trial {optuna_trial_obj.number} finished. Avg Policy Loss: {final_avg_policy_loss:.5f}")
        return final_avg_policy_loss

    logger.info(f"===== MUZERO AGENT RUN FINISHED (Version: {script_version}) =====")
    return None


# Optuna Objective Function
def objective(trial: optuna.trial.Trial):
    global config_mz, muzero_logger

    p_stddev1 = trial.suggest_float("policy_stddev1", 0.0, 0.4)
    p_stddev2 = trial.suggest_float("policy_stddev2", 0.0, 0.4)
    p_dropout_rate = trial.suggest_float("policy_dropout_rate", 0.0, 0.5)

    policy_hps = {'stddev1': p_stddev1, 'stddev2': p_stddev2, 'dropout_rate': p_dropout_rate}
    muzero_logger.info(f"Optuna Trial {trial.number}: Starting with HPs: stddev1={p_stddev1:.4f}, stddev2={p_stddev2:.4f}, dropout={p_dropout_rate:.4f}")

    problematic_module_ref = None
    original_config_had_module = False
    module_attr_name = 'existing_config_module'

    if hasattr(config_mz, module_attr_name):
        problematic_module_ref = getattr(config_mz, module_attr_name)
        original_config_had_module = True
        delattr(config_mz, module_attr_name)
        muzero_logger.debug(f"Temporarily removed '{module_attr_name}' from global config_mz for deepcopy.")

    try:
        trial_config = copy.deepcopy(config_mz)
        muzero_logger.debug("Successfully deepcopied config_mz (after temporary module removal).")
    except Exception as e_deepcopy:
        muzero_logger.error(f"Deepcopy failed even after attempting to remove '{module_attr_name}': {e_deepcopy}", exc_info=True)
        trial_config = copy.copy(config_mz)
        muzero_logger.warning("Used shallow copy for trial_config as a fallback due to deepcopy error.")
    finally:
        if original_config_had_module:
            setattr(config_mz, module_attr_name, problematic_module_ref)
            muzero_logger.debug(f"Restored '{module_attr_name}' to global config_mz.")

    if original_config_had_module:
        setattr(trial_config, module_attr_name, problematic_module_ref)
        muzero_logger.debug(f"Assigned '{module_attr_name}' (by reference) to trial_config.")

    trial_config.REPLAY_BUFFER_SIZE = OPTUNA_TRIAL_REPLAY_BUFFER_SIZE
    trial_config.FREEZE_TRANSFER_WEIGHTS_INITIALLY = True
    trial_config.UNFREEZE_TRANSFER_WEIGHTS_AFTER_STEP = OPTUNA_TRIAL_TOTAL_GAME_STEPS * 10

    avg_policy_loss_result = float('inf') # Default to a high value
    try:
        avg_policy_loss_result = train_muzero_agent(
            config=trial_config,
            logger=muzero_logger,
            optuna_trial_obj=trial,
            policy_hps_override=policy_hps
        )
    except optuna.exceptions.TrialPruned: # Explicitly re-raise if train_muzero_agent returns the pruning signal
        muzero_logger.info(f"Optuna Trial {trial.number} was pruned inside train_muzero_agent.")
        raise
    except Exception as e:
        muzero_logger.error(f"Exception in Optuna Trial {trial.number} during train_muzero_agent: {e}", exc_info=True)
        # avg_policy_loss_result remains float('inf')

    tf.keras.backend.clear_session() # Crucial for releasing GPU memory

    return avg_policy_loss_result


# --- Execution Guard ---
if __name__ == "__main__":
    missing_comp = False
    essential_globals_for_main_execution = ['config_mz', 'muzero_logger',
                                            'TRAINED_FEATURE_NAMES_GLOBAL', 'feature_mask_global',
                                            'ORIGINAL_MODEL_FEATURE_COUNT', 'load_ga_feature_mask',
                                            'FinancialMuZeroEnv', 'ReplayBuffer', 'MCTSNode', 'run_mcts',
                                            'calculate_muzero_losses', 'get_muzero_agent_networks' ]
    for var_name_item_main in essential_globals_for_main_execution:
        if var_name_item_main not in globals() or globals()[var_name_item_main] is None:
            print(f"CRITICAL: Prerequisite '{var_name_item_main}' missing for training loop.")
            missing_comp = True

    if not missing_comp:
        script_version_tag = getattr(config_mz, 'SCRIPT_VERSION', 'Optuna_Run_Fixed')
        muzero_logger.info(f"All prerequisites met. Starting MuZero Optuna study ({script_version_tag})...")

        if not hasattr(config_mz, 'ALWAYS_ATTEMPT_RESUME'): config_mz.ALWAYS_ATTEMPT_RESUME = True
        if not hasattr(config_mz, 'LOAD_CHECKPOINT_FROM_STEP'): config_mz.LOAD_CHECKPOINT_FROM_STEP = -1
        if not hasattr(config_mz, 'LATEST_CHECKPOINT_FILENAME'): config_mz.LATEST_CHECKPOINT_FILENAME = "latest_checkpoint_step.txt"
        if not hasattr(config_mz, 'MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING'): config_mz.MIN_REPLAY_BUFFER_SIZE_FOR_TRAINING = config_mz.BATCH_SIZE * 10
        if not hasattr(config_mz, 'INITIAL_EPISODE_LENGTH'): config_mz.INITIAL_EPISODE_LENGTH = 252
        if not hasattr(config_mz, 'DEFAULT_EPISODE_LENGTH_PLAY'): config_mz.DEFAULT_EPISODE_LENGTH_PLAY = 252
        if not hasattr(config_mz, 'GRADIENT_CLIP_NORM'): config_mz.GRADIENT_CLIP_NORM = 1.0
        if not hasattr(config_mz, 'VALUE_TARGET_SCALING_DIVISOR'): config_mz.VALUE_TARGET_SCALING_DIVISOR = 1.0
        if not hasattr(config_mz, 'REWARD_TARGET_SCALING_DIVISOR'): config_mz.REWARD_TARGET_SCALING_DIVISOR = 1.0
        if not hasattr(config_mz, 'ANNEAL_TEMPERATURE_AFTER_N_MOVES'): config_mz.ANNEAL_TEMPERATURE_AFTER_N_MOVES = config_mz.MAX_TRAINING_STEPS // 2
        if not hasattr(config_mz, 'INITIAL_TEMPERATURE'): config_mz.INITIAL_TEMPERATURE = 1.0
        if not hasattr(config_mz, 'FINAL_TEMPERATURE'): config_mz.FINAL_TEMPERATURE = 0.5


        config_mz.SCRIPT_VERSION = script_version_tag

        gpu_devices = tf.config.list_physical_devices('GPU')
        if not gpu_devices:
            muzero_logger.warning("NO GPU DETECTED BY TENSORFLOW. Training will be very slow.")
        else:
            muzero_logger.info(f"GPU detected: {gpu_devices}")
            try:
                for gpu in gpu_devices:
                    tf.config.experimental.set_memory_growth(gpu, True)
                muzero_logger.info("TensorFlow GPU memory growth enabled.")
            except RuntimeError as e_mem_growth:
                muzero_logger.warning(f"Could not set memory growth for GPU: {e_mem_growth}")

        try:
            study_name = f"muzero-policy-head-study-{script_version_tag}-{time.strftime('%Y%m%d-%H%M%S')}"
            pruner = optuna.pruners.MedianPruner(
                n_startup_trials=max(5, OPTUNA_N_TRIALS // 10),
                n_warmup_steps=OPTUNA_TRIAL_TRAINING_UPDATES // 3,
                interval_steps=10
            )

            study = optuna.create_study(
                study_name=study_name,
                direction="minimize",
                pruner=pruner
            )

            muzero_logger.info(f"Starting Optuna study: {study_name} with {OPTUNA_N_TRIALS} trials.")
            study.optimize(objective, n_trials=OPTUNA_N_TRIALS, timeout=getattr(config_mz, 'OPTUNA_TIMEOUT_SECONDS', None))

            muzero_logger.info("Optuna study finished.")
            muzero_logger.info(f"Best trial number: {study.best_trial.number}")
            muzero_logger.info(f"Best value (average policy loss): {study.best_value}")
            muzero_logger.info(f"Best hyperparameters: {study.best_params}")

            try:
                study_df = study.trials_dataframe()
                results_filename = f"{study_name}_results.csv"
                study_df.to_csv(results_filename)
                muzero_logger.info(f"Optuna study results saved to {results_filename}")
            except Exception as e_save_df:
                muzero_logger.error(f"Could not save Optuna study results dataframe: {e_save_df}")


        except Exception as e_main_optuna:
            muzero_logger.critical(f"Unhandled exception in __main__ during Optuna study: {e_main_optuna}", exc_info=True)
    else:
        print("Halting due to missing prerequisites for __main__ execution block.")